# DMRGT Phase 5 — Nasdaq-100: Mamba fixes, graph retuning, cost-aware portfolio, regime-triggered graph refresh

One notebook, one committed run (Kaggle: private notebook `dmrgt-ndx`, GPU T4 x2, Internet off, input `dmrgt-ndx-cache`).
All model and experiment code is in the cells below. The `%%writefile` cells write it to `/kaggle/working/dmrgt/*.py` at
runtime so that one worker process per GPU can import it; those files are not committed separately.

**Test discipline.** The 2019-2022 test rows and the 2023-2024 holdout are read first and only in the one-shot scoring cell,
after four lock files exist (`lock_graph.json`, `lock_postproc.json`, `lock_winner.json`, `lock_policy.json`).
**Survivorship bias.** The universe is today's Nasdaq-100 members with full history since 2015 (see `build_manifest.json`).

### Provenance (spec §1.3) — what was ported from the old code, and from where
Old code = tag `archive-before-ndx` (Phase 3 files: commit `175768d`, tag `phase3-complete`; Phase 4 / S&P 100 working notebook
sources: commit `a58dd21`). The detailed line-by-line diff log is `PORTING_LOG.md` at `a58dd21`.

| Component | Old file / function | Old commit | Status |
|---|---|---|---|
| `Fold`, `make_folds` | `splits.py` | 175768d | identical |
| `FoldBuilder` (windows, label alignment, train-only `StandardScaler`, graph-coverage drop) | `train.py::assemble_fold` | 175768d (restructured in a58dd21 `d4_data.py`) | logic identical; intentional changes: lookback is a parameter (bounded drop < 5%), test split gated by four locks, validation split into `val_es` / `val_sel` halves |
| `_pct_change_no_fill`, `_universe_returns`, `_sector_edges`, correlation-to-edges rule | `graph.py` | 175768d | identical rule; `edges_from_corr` re-expressed with array output; intentional change: `sector_level` parameter (sector / sub_sector) |
| `PositionalEncoding`, `TransformerEncoder`, `RelationMessagePassing` (hand-rolled `index_add_` R-GCN), `stack_per_sample_edges`, prediction head | `model.py` | 175768d | identical |
| `DMRGT` wrapper | `model.py::DMRGT` | 175768d | intentional change: selectable encoder, split forward `encode` / `propagate` |
| `MambaEncoder` family | new in Phase 4 (a58dd21 `d4_models.py`) | a58dd21 | intentional change: corrected Δ (dt) initialisation, `_no_reinit` tags, RMSNorm blocks per spec §4.3 |
| Evaluation harness (`backtest_daily_deciles`, metrics, `evaluate_fold`) and baselines | `eval.py`, `baselines.py` | 175768d | identical function bodies; intentional changes: random baseline draws fresh noise per (seed, day); test access asserts the four locks; `backtest_from_weights` added |
| Significance tests (Wilcoxon, stationary bootstrap) | `stats.py` | 175768d | identical bodies; added `holm_correction`, `compare_arms` |
| `set_seed`, parameter fingerprint, random-init assertion; Phase 3 training schedule (arm P) | `train.py::train_one_seed` | 175768d | ported; intentional change: early stopping on `val_es` only; `set_seed` also seeds Python `random` |
| Adaptive training schedule (arms A, B, C) | new in Phase 4 / 4.1 | a58dd21 | intentional change vs Phase 3: AdamW two groups, warmup, ReduceLROnPlateau, clipping, rolling-3 `val_es` early stopping |
| `RollingCorr`, trigger policies, calibration, inference engine (Phase 4B) | Phase 4 notebook design | a58dd21 | rewritten for this notebook; covered by unit tests 4-5 |
| Cost-aware portfolio (`ensemble_scores`, `ema_smooth`, `buffered_weights`) | new in Phase 5 | a58dd21 (S&P 100 WIP), spec §4.6 | new; covered by unit tests 9-11 |


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""                 # O1: the notebook kernel never sees a GPU; each worker sets its own
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"       # workers set it too

import hashlib, json, time
from pathlib import Path

WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
(WORK / "dmrgt").mkdir(parents=True, exist_ok=True)   # %%writefile cells below do not create folders
(WORK / "dmrgt" / "__init__.py").write_text("")       # makes dmrgt a package
RUN_T0 = time.time()

# ─── Pre-registered CONFIG (Phase 5 Nasdaq-100 spec §4.2) — fixed BEFORE any data is touched ───
# No value may change after the first committed run.
_ALL_SEEDS = [0, 1, 2, 3, 4]
CONFIG = {
    "phase": "5-ndx",
    "cache_dataset": "dmrgt-ndx-cache",
    "seeds": _ALL_SEEDS,
    "features": ["f1_logret", "f2_price_ma", "f3_rsi", "f4_macd", "f5_atr",
                 "f6_vol_zscore", "f7_market_ret", "f8_vix"],
    "backtest": {"commission_bps": 10, "slippage_bps": 5, "decile_fraction": 0.20},

    "folds_selection": [
        {"name": "fold_1_2019_calm",  "train_start": "2015-01-01", "train_end": "2018-12-31", "test_start": "2019-01-01", "test_end": "2019-12-31"},
        {"name": "fold_2_2020_covid", "train_start": "2015-01-01", "train_end": "2019-12-31", "test_start": "2020-01-01", "test_end": "2020-12-31"},
        {"name": "fold_3_2021",       "train_start": "2015-01-01", "train_end": "2020-12-31", "test_start": "2021-01-01", "test_end": "2021-12-31"},
        {"name": "fold_4_2022_rates", "train_start": "2015-01-01", "train_end": "2021-12-31", "test_start": "2022-01-01", "test_end": "2022-12-31"},
    ],
    "folds_holdout": [
        {"name": "fold_5_2023", "train_start": "2015-01-01", "train_end": "2022-12-31", "test_start": "2023-01-01", "test_end": "2023-12-31"},
        {"name": "fold_6_2024", "train_start": "2015-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01", "test_end": "2024-12-31"},
    ],
    "validation": {"rule": "same train/val split rule as the old pipeline (ported): the latest 15% of each fold's training dates",
                   "val_fraction": 0.15,
                   "halves": "chronological: val_es = first half, val_sel = second half"},

    "arms": {
        "P": {"encoder": "transformer", "lookback": 30, "schedule": "phase3"},
        "A": {"encoder": "transformer", "lookback": 30, "schedule": "adaptive"},
        "B": {"encoder": "mamba",       "lookback": 30, "schedule": "adaptive"},
        "C": {"encoder": "mamba",       "lookback": 60, "schedule": "adaptive"},
    },

    "training_adaptive": {
        "max_epochs": 50, "warmup_epochs": 3, "warmup_start_lr": 1e-4, "lr": 1e-3,
        "optimizer": "AdamW two groups (weight_decay 1e-4 / 0.0 for biases, norms, A_log, D, dt bias)",
        "weight_decay": 1e-4,
        "early_stop": "3-epoch rolling mean of val_es mean rank IC; patience 10 after warmup; best epoch must be >= warmup_epochs",
        "early_stop_patience": 10,
        "plateau": {"mode": "max", "factor": 0.5, "patience": 4, "min_lr": 1e-5,
                    "metric": "same rolling val_es metric, after warmup"},
        "grad_clip_norm": 1.0, "loss": "Huber, delta 1.0", "huber_delta": 1.0,
        "retry_failed_job_once": True,
    },
    "training_phase3": {"note": "ported verbatim from the old code (`../dmrgt-old-ref`, tag `archive-before-ndx`); its early-stopping rule runs on val_es only",
                        "optimizer": "Adam, fixed lr", "lr": 1e-3, "huber_delta": 1.0, "early_stop_patience": 10,
                        "source": "Phase 3 configs.yaml training block"},

    "mamba": {"d_model": 64, "n_layers": 2, "d_state": 16, "d_conv": 4, "expand": 2, "dropout": 0.1,
              "dt_min": 1e-3, "dt_max": 1e-1, "dt_init_floor": 1e-4},

    "graph_grid": {
        "trained_with_arm": "A",
        "variants": {
            "baseline":    {"k_pos": 8,  "k_neg": 4, "corr_window": 60, "sector_level": "sector"},
            "kpos12":      {"k_pos": 12, "k_neg": 4, "corr_window": 60, "sector_level": "sector"},
            "kpos16":      {"k_pos": 16, "k_neg": 4, "corr_window": 60, "sector_level": "sector"},
            "kneg6":       {"k_pos": 8,  "k_neg": 6, "corr_window": 60, "sector_level": "sector"},
            "kneg8":       {"k_pos": 8,  "k_neg": 8, "corr_window": 60, "sector_level": "sector"},
            "win40":       {"k_pos": 8,  "k_neg": 4, "corr_window": 40, "sector_level": "sector"},
            "win90":       {"k_pos": 8,  "k_neg": 4, "corr_window": 90, "sector_level": "sector"},
            "sector_fine": {"k_pos": 8,  "k_neg": 4, "corr_window": 60, "sector_level": "sub_sector"},
        },
        "sector_fine_fallback": "skip and record if universe.csv has no sub_sector column",
        "selection_metric": "val_sel mean per-run raw Sharpe", "tiebreak": "val_sel mean rank IC if within 0.02",
        "tie_threshold": 0.02,
    },

    "postproc": {
        "ensemble": "z-score per (date, seed), mean over seeds",
        "ema_half_life_days": [0, 2, 5],
        "q_in": 0.20, "q_out": [0.20, 0.30, 0.40],
        "selection": "highest mean val_sel strategy Sharpe across arms P, A, B, C; within 0.05 -> lower mean turnover",
        "tie_threshold": 0.05,
    },
    "winner_selection": {"candidates": ["B", "C"], "metric": "val_sel strategy Sharpe (chosen postproc)",
                         "tiebreak": "val_sel raw mean rank IC if within 0.05", "tie_threshold": 0.05},

    "phase4b": {"corr_window": "from lock_graph.json", "target_rebuild_rates": [0.10, 0.20, 0.35, 0.50],
                "signals": ["corr_drift", "mamba_delta", "combined"], "max_staleness_days": 20,
                "full_recompute_every": 250, "val_noninferiority_margin_sharpe": 0.10,
                "test_noninferiority_margin_sharpe": 0.25,
                "scale_benchmark": {"n_stocks": [40, 100, 500, 1000, 2000], "windows": [60, 250]}},

    "holdout": {"arms": ["winner", "A", "P"], "seeds": _ALL_SEEDS},
    "significance": {"bootstrap_iterations": 5000, "bootstrap_block_length": 10, "alpha": 0.05, "secondary_correction": "holm"},
    "leakage_gate": {"max_abs_ic_in_standard_errors": 3.0},

    # ── run 2 (memory- and speed-optimised fresh run): implementation settings, experiment design unchanged ──
    "run": 2,
    "memory_ladder": [{"batch_size": 32, "grad_checkpoint": False},
                      {"batch_size": 32, "grad_checkpoint": True},
                      {"batch_size": 16, "grad_checkpoint": True},
                      {"batch_size": 8,  "grad_checkpoint": True}],
    "memory_limits_gb": {"max_peak": 12.5, "min_headroom": 1.5},
    "selective_scan": {"impl": "chunked", "chunk": 8, "fallback_max_log": 60.0},
    "fold_cache": True,
    "kernel_gpu_free": True,
    "budget": {"session_limit_hours": 12.0, "plan_limit_hours": 10.5, "expected_epochs_for_projection": 35,
               "safety_factor": 1.15,
               "drop_order": ["graph grid non-baseline variants use seeds [0,1,2]",
                              "drop arm C (winner = B)",
                              "holdout seeds [0,1,2]",
                              "fail fast"]},
}

# Not part of the pre-registered CONFIG (planning constants of the notebook, recorded in the snapshot for the record).
PLANNING = {"phase4b_estimate_minutes": 45, "scoring_and_bench_estimate_minutes": 20, "scale_benchmark_days": 690}

CHANGES_FROM_RUN1 = {
    "run": 2,
    "why": "run 1 aborted at S2 (GPU out of memory: the notebook kernel retained 10.2 GB on GPU 0); no test or holdout access",
    "O1": "all GPU work in worker subprocesses; the kernel never initialises CUDA",
    "O2": "nvidia-smi free-GPU check before every launch; expandable_segments in workers; out-of-memory retry with a pause",
    "O3": "vectorised fold builder, fold and edge caches on disk, byte-identity test against the ported loop builder",
    "O4": "chunked selective scan (chunk 8) with an overflow fallback to the step-by-step scan",
    "O5": "memory ladder with gradient checkpointing before a smaller batch",
    "O6": "inference_mode everywhere; one validation pass per epoch shared by val_es and val_sel",
    "unchanged": "universe, folds, holdout, arms, schedules, graph grid, post-processing grid, selection rules, 4B, significance, budget",
}
SNAPSHOT = {"CONFIG": CONFIG, "changes_from_run1": CHANGES_FROM_RUN1}
_snap_text = json.dumps(SNAPSHOT, indent=2, sort_keys=True)
(WORK / "config_snapshot.json").write_text(_snap_text)
CONFIG_SHA256 = hashlib.sha256(_snap_text.encode()).hexdigest()
print("config_snapshot.json sha256 (pre-registered):", CONFIG_SHA256)


def update_snapshot(**kw):
    """Add runtime facts (universe, graph winner, budget plan) to the snapshot FILE. The
    pre-registered hash above is of CONFIG only and never changes."""
    snap = json.loads((WORK / "config_snapshot.json").read_text())
    snap.update(kw)
    (WORK / "config_snapshot.json").write_text(json.dumps(snap, indent=2, sort_keys=True))
    return snap


In [ ]:
# ─── Cell 2: environment + GPU checks via nvidia-smi (the kernel never initialises CUDA) ───
import platform, subprocess, sys
import numpy, pandas, scipy, torch

def _smi_rows(query):
    out = subprocess.run(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True, check=True).stdout.strip().splitlines()
    return [[x.strip() for x in r.split(",")] for r in out]

def assert_gpus_free(max_used_gb=1.0):
    """O2: every GPU must be (nearly) empty before a worker is launched — checked without touching CUDA."""
    for i, used, total in _smi_rows("index,memory.used,memory.total"):
        used_gb = float(used) / 1024
        print(f"GPU {i}: {used_gb:.2f}/{float(total) / 1024:.2f} GB used")
        assert used_gb <= max_used_gb, f"GPU {i} has {used_gb:.2f} GB in use before launch"

def kernel_guard(label=""):
    """O1: the notebook kernel must never initialise CUDA — all GPU work belongs in a worker."""
    assert not torch.cuda.is_initialized(), f"Kernel must not initialise CUDA — move this work to a worker ({label})."

try:
    _rows = _smi_rows("index,name,compute_cap,memory.total")
except subprocess.CalledProcessError:                    # driver too old for compute_cap
    _rows = [[i, name, "unknown", mem] for i, name, mem in _smi_rows("index,name,memory.total")]
n_gpus = len(_rows)
assert n_gpus >= 1, "No CUDA GPU visible — set the Kaggle accelerator to GPU T4 x2."
gpu_info = []
for i, name, cc, mem in _rows:
    cc_f = float(cc) if cc.replace(".", "", 1).isdigit() else (6.0 if "P100" in name else 7.5)
    gpu_info.append({"index": int(i), "name": name, "capability": cc_f, "memory_gb": float(mem) / 1024})
    assert cc_f >= 7.0, f"GPU {i} ({name}) has compute capability {cc_f} < 7.0 (P100 is sm_60) — select GPU T4 x2."
print(f"{n_gpus} GPU(s):", [(g["name"], round(g["memory_gb"], 1)) for g in gpu_info])
if n_gpus == 1:
    print("Only 1 GPU detected: the notebook runs sequentially; the budget planner accounts for it.")
assert_gpus_free()
kernel_guard("environment cell")

run_manifest = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "numpy": numpy.__version__, "pandas": pandas.__version__,
    "scipy": scipy.__version__, "gpus": gpu_info, "n_gpus": n_gpus,
    "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
    "kernel_cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"), "run": 2,
    "config_sha256": CONFIG_SHA256, "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(RUN_T0)),
}
(WORK / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))


In [ ]:
# ─── Cell 3: cache resolution + hash verification + universe (hard stop on failure) ───
from pathlib import Path
import hashlib, json
import pandas as pd

CANDIDATES = [
    Path("/kaggle/input/dmrgt-ndx-cache"),
    Path("/kaggle/input/datasets/harishsivakumar1912/dmrgt-ndx-cache"),
]
def resolve_cache():
    for c in CANDIDATES:
        if (c / "features.parquet").exists():
            return c
    hits = list(Path("/kaggle/input").rglob("features.parquet"))
    if len(hits) == 1:
        return hits[0].parent
    raise FileNotFoundError(f"Cache not found or ambiguous: {hits}")

CACHE = resolve_cache()
for _req in ["prices.parquet", "features.parquet", "targets.parquet", "graphs_baseline.parquet",
             "universe.csv", "hashes.json"]:
    assert (CACHE / _req).exists(), f"required cache file missing: {_req}"
expected = json.loads((CACHE / "hashes.json").read_text())
for name, h in expected.items():
    if name == "generated_at":                 # metadata key
        continue
    actual = hashlib.sha256((CACHE / name).read_bytes()).hexdigest()
    assert actual == h, f"Hash mismatch for {name}"
os.environ["DMRGT_CACHE"] = str(CACHE)    # inherited by the worker processes
print("cache OK:", CACHE, "—", len(expected) - 1, "file hashes verified")

# Universe + sectors from universe.csv (the cache build's pre-registered rules decided membership)
_U = pd.read_csv(CACHE / "universe.csv")
_inc = _U[_U["included"].astype(str) == "True"]
UNIVERSE_TICKERS = _inc["ticker"].tolist()
UNIVERSE_SECTORS = dict(zip(_inc["ticker"], _inc["sector"]))
HAS_SUB_SECTOR = "sub_sector" in _U.columns and _inc["sub_sector"].notna().all()
UNIVERSE_SUB_SECTORS = dict(zip(_inc["ticker"], _inc["sub_sector"])) if HAS_SUB_SECTOR else dict(UNIVERSE_SECTORS)
UNIVERSE_SHA256 = hashlib.sha256((CACHE / "universe.csv").read_bytes()).hexdigest()

# Graph-grid variants actually run: sector_fine is skipped (and recorded) if universe.csv has no sub_sector column
GRID_VARIANTS = {k: v for k, v in CONFIG["graph_grid"]["variants"].items()
                 if HAS_SUB_SECTOR or v.get("sector_level", "sector") != "sub_sector"}
SKIPPED_VARIANTS = [k for k in CONFIG["graph_grid"]["variants"] if k not in GRID_VARIANTS]
update_snapshot(UNIVERSE={"tickers": UNIVERSE_TICKERS, "sectors": UNIVERSE_SECTORS, "sub_sectors": UNIVERSE_SUB_SECTORS,
                          "n_stocks": len(UNIVERSE_TICKERS), "universe_csv_sha256": UNIVERSE_SHA256},
                grid_variants_run=list(GRID_VARIANTS), grid_variants_skipped=SKIPPED_VARIANTS)
_BM = json.loads((CACHE / "build_manifest.json").read_text()) if (CACHE / "build_manifest.json").exists() else {}
print(f"universe: N = {len(UNIVERSE_TICKERS)} stocks, {len(set(UNIVERSE_SECTORS.values()))} sectors, "
      f"{len(set(UNIVERSE_SUB_SECTORS.values()))} sub-sectors; excluded: {_U[_U['included'].astype(str) != 'True']['ticker'].tolist()}")
print("sector classification:", _BM.get("sector_classification", "n/a"))
print("SURVIVORSHIP BIAS: the universe is today's Nasdaq-100 constituents with full history, traced back to 2015 "
      "(see build_manifest.json).")
kernel_guard('Cell 3 — cache resolution + hash verification + universe')        # O1: the kernel never initialises CUDA


In [ ]:
%%writefile /kaggle/working/dmrgt/data.py
"""DMRGT Phase 5 — data module (Phase 4 spec §6.1 + Phase 5 spec §3/§6).

Ported from Phase 3 (reference/phase3_src/): splits.py (make_folds, Fold),
train.assemble_fold (windows, label alignment, train-only scaler, graph-
coverage drop). Phase 4 changes kept: lookback is a parameter; the test split
is gated. Phase 5 changes: the gate needs FOUR lock files; the universe and
sector map come from the cache's universe.csv (via the config snapshot); six
folds (1-4 selection, 5-6 holdout); each fold's validation is split into
val_es / val_sel halves (Phase 4.1 F2); graph variants are keyed.
"""

from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

WORK = Path(os.environ.get("DMRGT_WORK", "/kaggle/working"))
LOCKS = [WORK / "lock_graph.json", WORK / "lock_postproc.json",
         WORK / "lock_winner.json", WORK / "lock_policy.json"]

CANDIDATES = [
    Path("/kaggle/input/dmrgt-ndx-cache"),
    Path("/kaggle/input/datasets/harishsivakumar1912/dmrgt-ndx-cache"),
]


def resolve_cache() -> Path:
    env = os.environ.get("DMRGT_CACHE")
    if env:
        return Path(env)
    for c in CANDIDATES:
        if (c / "features.parquet").exists():
            return c
    hits = list(Path("/kaggle/input").rglob("features.parquet"))
    if len(hits) == 1:
        return hits[0].parent
    raise FileNotFoundError(f"Cache not found or ambiguous: {hits}")


# ─── config (from the Cell 0 snapshot) ─────────────────────────────────────

def load_snapshot() -> dict:
    return json.loads((WORK / "config_snapshot.json").read_text())


def _graph_params(snap: dict, variant: str | None) -> dict:
    C = snap["CONFIG"]
    if variant is None:
        variant = snap.get("GRAPH_WINNER", {}).get("name", "baseline")
    return dict(C["graph_grid"]["variants"][variant])


def load_cfg(variant: str | None = None) -> dict:
    """Phase-3-shaped cfg dict assembled from the snapshot, so the ported
    eval / stats / fold code runs unchanged. `variant` selects the graph
    parameters (default: the S1 winner once locked, else baseline)."""
    snap = load_snapshot()
    C, Uv = snap["CONFIG"], snap["UNIVERSE"]
    sig = C["significance"]
    gp = _graph_params(snap, variant)
    level = gp.get("sector_level", "sector")             # "sector" | "sub_sector" (the finer-sector grid variant)
    return {
        "universe": {"tickers": list(Uv["tickers"])},
        "sectors": dict(Uv["sub_sectors"] if level == "sub_sector" else Uv["sectors"]),
        "features": {"lookback_days": 30, "columns": list(C["features"])},
        "graph": {"corr_window": gp["corr_window"], "k_pos": gp["k_pos"], "k_neg": gp["k_neg"],
                  "symmetrise": False, "include_sector": True, "min_history_days": gp["corr_window"] + 1},
        "training": {"huber_delta": C["training_adaptive"]["huber_delta"]},
        "backtest": dict(C["backtest"]),
        "significance": {"wilcoxon_alpha": sig["alpha"],
                         "bootstrap_iterations": sig["bootstrap_iterations"],
                         "bootstrap_block_length": sig["bootstrap_block_length"]},
        "folds": list(C["folds_selection"]) + list(C["folds_holdout"]),
    }


def split_val(val_dates):
    """Phase 4.1 F2.1: chronological halves of a fold's validation period.
    val_es → early stopping / LR scheduler only; val_sel → selection only."""
    k = len(val_dates) // 2
    return val_dates[:k], val_dates[k:]


# ─── folds (ported verbatim from splits.py) ────────────────────────────────

@dataclass
class Fold:
    """One walk-forward fold with train_dates, val_dates, test_dates."""
    name: str
    fold_idx: int
    train_dates: pd.DatetimeIndex
    val_dates: pd.DatetimeIndex
    test_dates: pd.DatetimeIndex

    def __repr__(self) -> str:
        def span(idx: pd.DatetimeIndex) -> str:
            if len(idx) == 0:
                return "empty"
            return f"{idx[0].date()}→{idx[-1].date()} (n={len(idx)})"
        return (f"Fold({self.name!r}, train={span(self.train_dates)}, "
                f"val={span(self.val_dates)}, test={span(self.test_dates)})")


def make_folds(
    cfg: dict,
    available_dates: Iterable[pd.Timestamp] | None = None,
    val_fraction: float = 0.15,
) -> list[Fold]:
    """Build walk-forward folds from configs.yaml.

    `available_dates` restricts each configured window to dates that
    actually exist in the cache (the features table's calendar). The
    latest val_fraction of each training window becomes validation.
    """
    if not 0.0 < val_fraction < 1.0:
        raise ValueError(f"val_fraction must be in (0, 1), got {val_fraction}")

    avail = (pd.DatetimeIndex(sorted(set(pd.Timestamp(d) for d in available_dates)))
             if available_dates is not None else None)
    folds_cfg = cfg.get("folds", [])
    if not folds_cfg:
        raise ValueError("configs.yaml has no 'folds' section")

    out: list[Fold] = []
    for i, fc in enumerate(folds_cfg):
        name = str(fc.get("name", f"fold_{i}"))
        train_start = pd.Timestamp(fc["train_start"])
        train_end = pd.Timestamp(fc["train_end"])
        test_start = pd.Timestamp(fc["test_start"])
        test_end = pd.Timestamp(fc["test_end"])

        # ── hard chronological invariant (Rule 2) ─────────────────────
        if train_end >= test_start:
            raise AssertionError(
                f"Fold {name!r}: train_end ({train_end.date()}) must be "
                f"strictly before test_start ({test_start.date()}) — "
                f"otherwise the model sees the test window during training."
            )

        # ── build the full training and test index ─────────────────────
        if avail is not None:
            # Configured window, intersected with dates the cache has.
            # The boundary itself is owned by the config; the data
            # restricts what we can actually use.
            train_mask = (avail >= train_start) & (avail <= train_end)
            test_mask = (avail >= test_start) & (avail <= test_end)
            train_full = avail[train_mask]
            test_idx = avail[test_mask]
        else:
            # No availability constraint (tests / running splits without
            # the cache): synthesise business-day dates. The next consumer
            # that loads the features table intersects again.
            train_full = pd.bdate_range(train_start, train_end)
            test_idx = pd.bdate_range(test_start, test_end)

        if len(train_full) == 0:
            raise ValueError(f"Fold {name!r}: no train dates after intersecting with available dates")
        if len(test_idx) == 0:
            raise ValueError(f"Fold {name!r}: no test dates after intersecting with available dates")

        # ── split off the validation portion ───────────────────────────
        # The latest val_fraction of the training window becomes the
        # validation set. Chronological, not random.
        n_train_full = len(train_full)
        n_val = int(round(n_train_full * val_fraction))
        if n_val <= 0 or n_val >= n_train_full:
            raise ValueError(
                f"Fold {name!r}: val_fraction={val_fraction} produced "
                f"n_val={n_val} from n_train_full={n_train_full}; "
                f"need at least 1 train and 1 val date"
            )
        train_idx = train_full[:-n_val]
        val_idx = train_full[-n_val:]

        # ── zero-overlap asserts — fail loudly, never silently ─────────
        assert len(set(train_idx).intersection(val_idx)) == 0
        assert len(set(train_idx).intersection(test_idx)) == 0
        assert len(set(val_idx).intersection(test_idx)) == 0
        # Chronological: last train date precedes first val date, which
        # precedes the first test date.
        assert train_idx.max() < val_idx.min()
        assert val_idx.max() < test_idx.min()
        assert train_idx.max() < test_idx.min()

        out.append(Fold(
            name=name,
            fold_idx=i,
            train_dates=train_idx,
            val_dates=val_idx,
            test_dates=test_idx,
        ))

    return out


# ─── test access gate (Change 2) ───────────────────────────────────────────

def test_unlocked() -> bool:
    return all(p.exists() for p in LOCKS)


def assert_test_unlocked() -> None:
    if not test_unlocked():
        raise PermissionError(
            "Test split is locked until lock_graph.json, lock_postproc.json, "
            "lock_winner.json and lock_policy.json all exist.")


# ─── tables, loaded once per process ───────────────────────────────────────

_TABLES: dict = {}


def tables() -> dict:
    """features / targets pivots + folds for this process (cached)."""
    if _TABLES:
        return _TABLES
    cache = resolve_cache()
    cfg = load_cfg()
    features = pd.read_parquet(cache / "features.parquet")
    targets = pd.read_parquet(cache / "targets.parquet")
    feature_cols = list(cfg["features"]["columns"])
    missing = [c for c in feature_cols if c not in features.columns]
    assert not missing, f"config lists feature columns absent from features.parquet: {missing}"
    assert len(feature_cols) == 8, f"Expected 8 features, got {len(feature_cols)}"
    feat_wide = (features.pivot(index="date", columns="ticker", values=feature_cols).sort_index())
    # Fix the feature level to config order (pivot sorts lexicographically).
    feat_wide = feat_wide.reindex(columns=pd.MultiIndex.from_product(
        [feature_cols, list(feat_wide.columns.get_level_values("ticker").unique())],
        names=["feature", "ticker"]))
    tgt_wide = (targets.pivot(index="date", columns="ticker", values="excess_return").sort_index())
    folds = make_folds(cfg, available_dates=features["date"].unique())
    _TABLES.update(cfg=cfg, cache=cache, features=features, targets=targets,
                   feat_wide=feat_wide, tgt_wide=tgt_wide, folds=folds,
                   feature_cols=feature_cols)
    return _TABLES


def fold_by_name(name: str) -> Fold:
    for f in tables()["folds"]:
        if f.name == name:
            return f
    raise KeyError(name)


@dataclass
class SplitData:
    name: str
    split: str
    X: np.ndarray            # [n_days, n_stocks, lookback, n_features], scaled
    y: np.ndarray            # [n_days, n_stocks]
    dates: np.ndarray
    tickers: list


FOLD_CACHE = WORK / "fold_cache"


def _graph_key(graph_dates) -> str:
    """Fold data depends on the graph variant only through which dates have a graph (the corr window)."""
    if graph_dates is None:
        return "nograph"
    d = sorted(graph_dates)
    return f"G{pd.Timestamp(d[0]).date()}_{len(d)}"


class FoldBuilder:
    """Per-fold, per-lookback tensors. Train and val come from the on-disk fold cache (built once, vectorised,
    memory-mapped by every process); test is built lazily and only after the four lock files exist."""

    def __init__(self, fold: Fold, lookback: int, graph_dates=None, use_cache: bool = True, builder: str = "vec"):
        T = tables()
        self.fold, self.lookback, self.graph_dates = fold, lookback, graph_dates
        self.cfg = T["cfg"]
        self.universe = list(self.cfg["universe"]["tickers"])
        self.feature_cols = T["feature_cols"]
        self.feat_wide, self.tgt_wide = T["feat_wide"], T["tgt_wide"]
        self.dropped_no_graph = {"train": [], "val": [], "test": []}
        self._build = self._build_vec if builder == "vec" else self._build_loop
        self.cache_dir = FOLD_CACHE / f"{fold.name}_L{lookback}_{_graph_key(graph_dates)}_N{len(self.universe)}"
        self.cache_hit = False
        if use_cache and (self.cache_dir / "meta.json").exists():
            arrays = self._load_cache()
            self.cache_hit = True
        else:
            arrays = self._build_arrays()
            if use_cache:
                self._save_cache(arrays)
        self._finish(arrays)

    # ── build (from features/targets) ──────────────────────────────────
    def _build_arrays(self) -> dict:
        fold, lookback, graph_dates = self.fold, self.lookback, self.graph_dates
        X_tr, y_tr, d_tr = self._build(fold.train_dates, "train")
        X_va, y_va, d_va = self._build(fold.val_dates, "val")
        assert len(X_tr) > 0, f"{fold.name}: no training windows at lookback {lookback}"

        # ── bounded lookback drop (Change 1) ──────────────────────────
        # Dates Phase 3's 30-day windows kept that this lookback loses.
        locs = [self.feat_wide.index.get_loc(d) for d in fold.train_dates if d in self.feat_wide.index]
        base = sum(1 for l in locs if l >= 30 - 1)
        now = sum(1 for l in locs if l >= lookback - 1)
        self.lookback_dropped = base - now
        self.lookback_drop_frac = self.lookback_dropped / max(1, base)
        assert self.lookback_drop_frac < 0.05, (
            f"{fold.name}: lookback {lookback} drops {self.lookback_dropped}/{base} "
            f"train days ({self.lookback_drop_frac:.2%}) — over the 5% bound")
        print(f"[data] {fold.name} lookback={lookback}: dropped {self.lookback_dropped}/{base} "
              f"train days vs the 30-day window ({self.lookback_drop_frac:.2%})", flush=True)

        # ── bounded graph-coverage check (ported from Phase 3 §3.5) ───
        total_dropped = sum(len(v) for v in self.dropped_no_graph.values())
        total_dates = len(fold.train_dates) + len(fold.val_dates)
        self.graph_dropped = total_dropped
        if graph_dates is not None and total_dropped > 0:
            drop_frac = total_dropped / total_dates
            assert drop_frac < 0.05, (
                f"{total_dropped} dates ({drop_frac:.2%}) missing graph coverage — "
                f"check which graph file this run was pointed at.")

        # ── scaler fitted on training dates ONLY ──────────────────────
        n_feat = X_tr.shape[3]
        scaler = StandardScaler().fit(X_tr.reshape(-1, n_feat))
        mean, scale = scaler.mean_, scaler.scale_
        assert mean.shape == (n_feat,)
        return {"X_train": self._tf(X_tr, mean, scale), "y_train": y_tr, "d_train": pd.DatetimeIndex(d_tr).to_numpy(),
                "X_val": self._tf(X_va, mean, scale), "y_val": y_va, "d_val": pd.DatetimeIndex(d_va).to_numpy(),
                "scaler_mean": mean.astype(float), "scaler_scale": scale.astype(float)}

    @staticmethod
    def _tf(X, mean, scale):
        if len(X) == 0:
            return X
        shape = X.shape
        return ((X.reshape(-1, shape[3]) - mean) / scale).reshape(shape).astype(np.float32)

    def _transform(self, X):
        return self._tf(X, self.scaler_mean, self.scaler_scale)

    # ── disk cache ──────────────────────────────────────────────────────
    def _save_cache(self, a: dict) -> None:
        import shutil, tempfile
        FOLD_CACHE.mkdir(parents=True, exist_ok=True)
        tmp = Path(tempfile.mkdtemp(dir=FOLD_CACHE, prefix=".tmp_"))
        for k, v in a.items():
            np.save(tmp / f"{k}.npy", v)
        (tmp / "meta.json").write_text(json.dumps({
            "lookback_dropped": self.lookback_dropped, "lookback_drop_frac": self.lookback_drop_frac,
            "graph_dropped": self.graph_dropped, "fold": self.fold.name, "lookback": self.lookback}))
        try:
            os.rename(tmp, self.cache_dir)                      # atomic; loses the race harmlessly if another process won
        except OSError:
            shutil.rmtree(tmp, ignore_errors=True)

    def _load_cache(self) -> dict:
        meta = json.loads((self.cache_dir / "meta.json").read_text())
        self.lookback_dropped, self.lookback_drop_frac = meta["lookback_dropped"], meta["lookback_drop_frac"]
        self.graph_dropped = meta["graph_dropped"]
        assert self.lookback_drop_frac < 0.05
        return {k: np.load(self.cache_dir / f"{k}.npy", mmap_mode="r")
                for k in ("X_train", "y_train", "d_train", "X_val", "y_val", "d_val", "scaler_mean", "scaler_scale")}

    def _finish(self, a: dict) -> None:
        fold = self.fold
        self.scaler_mean = np.array(a["scaler_mean"], dtype=float)
        self.scaler_scale = np.array(a["scaler_scale"], dtype=float)
        d_tr, d_va = pd.DatetimeIndex(np.asarray(a["d_train"])), pd.DatetimeIndex(np.asarray(a["d_val"]))
        dt_obj = lambda idx: np.array(list(idx), dtype=object)      # arrays of Timestamps, as before
        self.scaler_fit_dates = (pd.Timestamp(d_tr[0]), pd.Timestamp(d_tr[-1]))
        self._splits = {
            "train": SplitData(fold.name, "train", a["X_train"], np.asarray(a["y_train"]), dt_obj(d_tr), self.universe),
            "val": SplitData(fold.name, "val", a["X_val"], np.asarray(a["y_val"]), dt_obj(d_va), self.universe),
        }
        assert pd.Timestamp(d_tr.max()) < pd.Timestamp(d_va.min()), "train/val chronology violated"
        # Phase 4.1 F2.1: val_es (first half of the fold's validation dates) / val_sel (second half)
        es_dates, sel_dates = split_val(fold.val_dates)
        in_es = np.asarray(d_va.isin(es_dates))
        in_sel = np.asarray(d_va.isin(sel_dates))
        assert in_es.sum() > 0 and in_sel.sum() > 0 and not (in_es & in_sel).any()
        assert pd.Timestamp(d_va[in_es].max()) < pd.Timestamp(d_va[in_sel].min())
        v = self._splits["val"]
        self._splits["val_es"] = SplitData(fold.name, "val_es", v.X[in_es], v.y[in_es], v.dates[in_es], self.universe)
        self._splits["val_sel"] = SplitData(fold.name, "val_sel", v.X[in_sel], v.y[in_sel], v.dates[in_sel], self.universe)

    # ── window assembly ─────────────────────────────────────────────────
    def _select(self, date_set, which: str):
        """The ported date filter: in features and targets, enough history, graph coverage. -> kept dates, feature locs."""
        lookback, feat_wide, tgt_wide = self.lookback, self.feat_wide, self.tgt_wide
        kept, locs = [], []
        for d in date_set:
            if d not in feat_wide.index:
                continue
            if d not in tgt_wide.index:
                continue
            loc = feat_wide.index.get_loc(d)
            if loc - lookback + 1 < 0:
                continue  # not enough history — drop, never zero-pad
            if self.graph_dates is not None and d not in self.graph_dates:
                self.dropped_no_graph[which].append(d)
                continue
            kept.append(d)
            locs.append(loc)
        return kept, np.asarray(locs, dtype=np.int64)

    def _feature_array(self) -> np.ndarray:
        """(T, N, F) float64 in universe order and config feature order."""
        if getattr(self, "_farr", None) is None:
            self._farr = np.stack([self.feat_wide[f][self.universe].to_numpy(dtype=np.float64)
                                   for f in self.feature_cols], axis=-1)
        return self._farr

    def _build_vec(self, date_set, which: str):
        """Vectorised window assembly (sliding_window_view); byte-identical to _build_loop (unit test)."""
        from numpy.lib.stride_tricks import sliding_window_view
        kept, locs = self._select(date_set, which)
        universe, lookback = self.universe, self.lookback
        if not kept:
            return (np.empty((0, len(universe), lookback, len(self.feature_cols))), np.empty((0, len(universe))),
                    np.array(kept))
        A = self._feature_array()
        win = sliding_window_view(A, lookback, axis=0)               # (T-lb+1, N, F, lb); window ending at row i+lb-1
        X = np.ascontiguousarray(win[locs - lookback + 1].transpose(0, 1, 3, 2))
        assert X.shape == (len(kept), len(universe), lookback, len(self.feature_cols))
        tl = self.tgt_wide.index.get_indexer(pd.DatetimeIndex(kept))
        y = self.tgt_wide[universe].to_numpy(dtype=np.float64)[tl]
        return X, y, np.array(kept)

    def _build_loop(self, date_set, which: str):
        """The ported per-day / per-stock loop (Phase 3 assemble_fold). Reference for the byte-identity test."""
        lookback, feat_wide, tgt_wide = self.lookback, self.feat_wide, self.tgt_wide
        universe = self.universe
        kept, locs = self._select(date_set, which)
        Xs, ys = [], []
        for d, loc in zip(kept, locs):
            window_dates = feat_wide.index[loc - lookback + 1: loc + 1]
            assert len(window_dates) == lookback
            assert window_dates[-1] == d, "window last date != prediction date"
            X = np.empty((len(universe), lookback, len(self.feature_cols)), dtype=np.float64)
            for j, t in enumerate(universe):
                X[j] = feat_wide.loc[window_dates, (slice(None), t)].values
            Xs.append(X)
            ys.append(tgt_wide.loc[d, universe].values)
        X = (np.stack(Xs, axis=0) if Xs
             else np.empty((0, len(universe), lookback, len(self.feature_cols))))
        y = np.stack(ys, axis=0) if ys else np.empty((0, len(universe)))
        return X, y, np.array(kept)

    def get(self, split: str) -> SplitData:
        if split == "test":
            assert_test_unlocked()
            if "test" not in self._splits:
                X, y, d = self._build_vec(self.fold.test_dates, "test")
                assert len(X) > 0
                assert pd.Timestamp(self._splits["val"].dates.max()) < pd.Timestamp(d.min()), \
                    "val/test chronology violated"
                self._splits["test"] = SplitData(self.fold.name, "test", self._transform(X), y, d,
                                                 self.universe)
        return self._splits[split]


def prebuild_fold_cache(folds: list, lookbacks: list, graph_dates_list: list) -> int:
    """Build (and persist) the fold tensors for every (fold, lookback, graph coverage). Returns the number built."""
    n = 0
    for gd in graph_dates_list:
        for fold in folds:
            for lb in lookbacks:
                b = FoldBuilder(fold, lb, gd)
                n += 0 if b.cache_hit else 1
                del b
    return n


from collections import OrderedDict

_BUILDERS: "OrderedDict" = OrderedDict()
_BUILDER_CAP = 2     # a fold at lookback 60 is ~0.5-0.7 GB; keep the working set small


def get_builder(fold_name: str, lookback: int = 30, graph_dates=None, graph_key: str = "baseline") -> FoldBuilder:
    key = (fold_name, lookback, graph_key)
    if key in _BUILDERS:
        _BUILDERS.move_to_end(key)
    else:
        _BUILDERS[key] = FoldBuilder(fold_by_name(fold_name), lookback, graph_dates)
        while len(_BUILDERS) > _BUILDER_CAP:
            _BUILDERS.popitem(last=False)
    return _BUILDERS[key]


def get_split(fold, split, lookback: int = 30, graph_dates=None, graph_key: str = "baseline") -> SplitData:
    """The only way tensors leave this module. split='test' raises
    PermissionError until all four phase-5 lock files exist."""
    if split == "test":
        assert_test_unlocked()
    name = fold if isinstance(fold, str) else fold.name
    return get_builder(name, lookback, graph_dates, graph_key).get(split)


In [ ]:
%%writefile /kaggle/working/dmrgt/graph.py
"""DMRGT Phase 5 — graph module (Phase 4 spec §6.2 + Phase 5 §5).

Ported verbatim from Phase 3 graph.py: _pct_change_no_fill, _universe_returns,
_sector_edges, and the correlation→edges rule (nan_to_num, diagonal ∓inf,
argsort, top-k_pos / bottom-k_neg, sector edges, directed). The daily-edge
loader reads graphs_baseline.parquet.

New: RollingCorr (incremental float64 correlation), run_policy (trigger-
policy runner), trigger rules + threshold calibration, and the Phase 4B
inference engine (checkpoint → per-day encodings/Δ → policy graphs →
predictions). Inference only; nothing here trains.
"""

from __future__ import annotations

import time
from collections import Counter, deque

import numpy as np
import pandas as pd
import torch

from dmrgt import data as d_data
from dmrgt import evaluate as d_eval
from dmrgt import models as d_models


# ─── ported from Phase 3 graph.py ──────────────────────────────────────────

def _pct_change_no_fill(df: pd.DataFrame) -> pd.DataFrame:
    """pct_change with NO filling, across pandas 2.x and 3.x.

    pandas 2.x: pct_change(fill_method=None) — explicit no-fill.
    pandas 3.x: the fill_method keyword was removed BECAUSE no-fill
    became the only behaviour, so plain pct_change() is identical.
    The cache was built under pandas 2.x no-fill semantics; the §6b
    baseline-identity test proves the rebuild reproduces it byte-for-
    byte under either version.
    """
    try:
        return df.pct_change(fill_method=None)
    except TypeError:  # pandas ≥ 3: fill_method removed
        return df.pct_change()


def _universe_returns(prices: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    """Wide close prices for the 40 universe tickers → daily returns.

    rets.loc[d] is the return from d-1 to d; the first row (NaN) is
    dropped. Same op order as Phase 2 so the skip logic below lands on
    the same dates.
    """
    universe = list(cfg["universe"]["tickers"])
    px_wide = (prices.pivot(index="date", columns="ticker", values="close")
               .sort_index())
    missing = [t for t in universe if t not in px_wide.columns]
    if missing:
        raise ValueError(f"prices.parquet is missing universe tickers: {missing}")
    px_universe = px_wide[universe]
    rets = _pct_change_no_fill(px_universe)
    return rets.iloc[1:]


def _sector_edges(cfg: dict) -> tuple[list[int], list[int]]:
    """Static intra-sector edges (both directions), as index pairs.

    Every same-sector pair gets a directed edge each way (cfg['sectors'] is the sector or sub-sector map).
    """
    universe = list(cfg["universe"]["tickers"])
    sectors_cfg = cfg.get("sectors", {})
    if not sectors_cfg:
        raise ValueError("universe sector map is empty — required for graph.py")
    include_sector = bool(cfg["graph"].get("include_sector", True))
    srcs: list[int] = []
    dsts: list[int] = []
    if not include_sector:
        return srcs, dsts
    n = len(universe)
    for i in range(n):
        for j in range(n):
            if i >= j:
                continue
            if sectors_cfg.get(universe[i]) == sectors_cfg.get(universe[j]):
                srcs.extend([i, j])  # directed both ways
                dsts.extend([j, i])
    sector_counts = Counter(sectors_cfg[t] for t in universe)
    expected = sum(c * (c - 1) for c in sector_counts.values())
    assert len(srcs) == expected, (
        f"Sector edge count mismatch: built {len(srcs)}, expected "
        f"{expected} from the sector map"
    )
    return srcs, dsts



def edges_from_corr(corr: np.ndarray, cfg: dict, sector_src, sector_dst):
    """Correlation matrix → (edge_index [2, E], edge_type [E]) torch tensors.

    The rule of Phase 3's _edges_for_returns_window, unchanged: nan_to_num →
    diagonal −inf → argsort(−corr) for type 0 (top k_pos) → diagonal +inf →
    argsort(corr) for type 1 (bottom k_neg) → static sector edges (type 2).
    Rows are emitted in the cached file's (edge_type, src, dst) order.
    """
    n_stocks = corr.shape[0]
    k_pos = int(cfg["graph"]["k_pos"])
    k_neg = int(cfg["graph"]["k_neg"])
    corr = np.nan_to_num(corr, nan=0.0, posinf=1.0, neginf=-1.0)   # returns a copy
    src_l, dst_l, typ_l = [], [], []
    np.fill_diagonal(corr, -np.inf)
    pos_order = np.argsort(-corr, axis=1)
    for i in range(n_stocks):
        for rank in range(k_pos):
            src_l.append(i); dst_l.append(int(pos_order[i, rank])); typ_l.append(0)
    np.fill_diagonal(corr, np.inf)
    neg_order = np.argsort(corr, axis=1)
    for i in range(n_stocks):
        for rank in range(k_neg):
            src_l.append(i); dst_l.append(int(neg_order[i, rank])); typ_l.append(1)
    for s, d in zip(sector_src, sector_dst):
        src_l.append(int(s)); dst_l.append(int(d)); typ_l.append(2)
    src_a, dst_a, typ_a = np.array(src_l), np.array(dst_l), np.array(typ_l)
    order = np.lexsort((dst_a, src_a, typ_a))            # (edge_type, src, dst)
    ei = torch.tensor(np.stack([src_a[order], dst_a[order]]), dtype=torch.long)
    et = torch.tensor(typ_a[order], dtype=torch.long)
    return ei, et


def make_graph_builder(cfg: dict):
    sector_src, sector_dst = _sector_edges(cfg)

    def build_graph(C):
        return edges_from_corr(C, cfg, sector_src, sector_dst)
    return build_graph


_EDGES: dict = {}


def build_variant_edges(variant: str) -> dict:
    """The graph of one grid variant, built in-notebook from prices.parquet with the
    ported rule: trailing corr_window Pearson correlation (pandas, as in Phase 3
    graph.py) → edges_from_corr. {Timestamp: (edge_index, edge_type)}."""
    cfgv = d_data.load_cfg(variant)
    W = int(cfgv["graph"]["corr_window"])
    rets = universe_returns(cfgv)
    build = make_graph_builder(cfgv)
    out = {}
    for idx in range(W - 1, len(rets)):
        window_rets = rets.iloc[idx - W + 1: idx + 1]
        assert len(window_rets) == W
        assert window_rets.isna().sum().sum() == 0, f"NaN returns in correlation window on {rets.index[idx].date()}"
        out[pd.Timestamp(rets.index[idx])] = build(window_rets.corr().values)
    return out


def _edges_cache_path(variant: str):
    import hashlib, json
    cfgv = d_data.load_cfg(variant)
    key = json.dumps([variant, cfgv["graph"], list(cfgv["universe"]["tickers"]), cfgv["sectors"]], sort_keys=True)
    return d_data.FOLD_CACHE / f"edges_{variant}_{hashlib.sha1(key.encode()).hexdigest()[:10]}.pt"


def load_daily_edges(variant: str = "baseline") -> dict:
    """{Timestamp: (edge_index, edge_type)} for a graph variant (cached per process, and on disk once per variant so
    no worker process ever recomputes it). Built in-notebook for EVERY variant, including 'baseline' (which unit
    test 5 cross-checks against the cache's graphs_baseline.parquet)."""
    if variant in _EDGES:
        return _EDGES[variant]
    import os, tempfile
    path = _edges_cache_path(variant)
    if path.exists():
        pk = torch.load(path, map_location="cpu", weights_only=False)
        off = pk["offsets"]
        out = {pd.Timestamp(d): (pk["ei"][:, off[i]:off[i + 1]].contiguous(), pk["et"][off[i]:off[i + 1]].contiguous())
               for i, d in enumerate(pk["dates"])}
    else:
        out = build_variant_edges(variant)
        dates = list(out.keys())
        lens = [out[d][1].shape[0] for d in dates]
        pk = {"dates": [str(d) for d in dates], "offsets": [0] + list(np.cumsum(lens)),
              "ei": torch.cat([out[d][0] for d in dates], dim=1), "et": torch.cat([out[d][1] for d in dates])}
        d_data.FOLD_CACHE.mkdir(parents=True, exist_ok=True)
        fd, tmp = tempfile.mkstemp(dir=d_data.FOLD_CACHE, prefix=".tmp_edges_", suffix=".pt")
        os.close(fd)
        torch.save(pk, tmp)
        os.replace(tmp, path)
    _EDGES[variant] = out
    return out


def load_cached_baseline_edges() -> dict:
    """graphs_baseline.parquet from the cache (used only for the cross-check)."""
    g = pd.read_parquet(d_data.resolve_cache() / "graphs_baseline.parquet")
    out = {}
    for d, day in g.groupby("date", sort=True):
        out[pd.Timestamp(d)] = (torch.tensor(day[["src", "dst"]].values.T, dtype=torch.long),
                                torch.tensor(day["edge_type"].values, dtype=torch.long))
    return out


def universe_returns(cfg: dict) -> pd.DataFrame:
    prices = pd.read_parquet(d_data.resolve_cache() / "prices.parquet")
    return _universe_returns(prices, cfg)


# ─── incremental rolling correlation (new) ─────────────────────────────────

class RollingCorr:
    def __init__(self, window, n_stocks, full_recompute_every=250):
        self.W = window
        self.buf = deque()
        self.S1 = np.zeros(n_stocks)
        self.S2 = np.zeros((n_stocks, n_stocks))
        self.k = full_recompute_every
        self.n_updates = 0

    def push(self, r):                       # r: (N,) float64 returns for one day
        assert np.isfinite(r).all()
        self.buf.append(r)
        self.S1 += r
        self.S2 += np.outer(r, r)
        if len(self.buf) > self.W:
            old = self.buf.popleft()
            self.S1 -= old
            self.S2 -= np.outer(old, old)
        self.n_updates += 1
        if self.n_updates % self.k == 0:     # remove floating-point drift
            X = np.asarray(self.buf)
            self.S1, self.S2 = X.sum(0), X.T @ X

    def corr(self):
        W = len(self.buf)
        cov = (self.S2 - np.outer(self.S1, self.S1) / W) / (W - 1)
        d = np.sqrt(np.diag(cov))
        return cov / np.outer(d, d)


def warm_rolling_corr(rets: pd.DataFrame, end_date, window: int, full_recompute_every=250) -> RollingCorr:
    """RollingCorr holding the `window` returns ending at `end_date` inclusive."""
    loc = rets.index.get_loc(pd.Timestamp(end_date))
    assert loc >= window - 1, f"only {loc + 1} returns up to {end_date}"
    rc = RollingCorr(window, rets.shape[1], full_recompute_every)
    for r in rets.iloc[loc - window + 1: loc + 1].to_numpy(dtype=np.float64):
        rc.push(r)
    return rc


class PrecomputedCorr:
    """Replays correlation matrices computed once (calibration sims). Same
    push()/corr() interface as RollingCorr, so run_policy is one code path."""

    def __init__(self, C_by_step):
        self.C = C_by_step          # list: [C before first push, C after push 1, ...]
        self.i = 0

    def push(self, r):
        self.i += 1

    def corr(self):
        return self.C[self.i]


def corr_by_date(rets: pd.DataFrame, window: int) -> dict:
    """{date: corr matrix over the trailing window ending that date} for every
    date with a full window, via one incremental pass."""
    rc = RollingCorr(window, rets.shape[1])
    out = {}
    for i, (d, r) in enumerate(zip(rets.index, rets.to_numpy(dtype=np.float64))):
        rc.push(r)
        if i >= window - 1:
            out[pd.Timestamp(d)] = rc.corr()
    return out


# ─── trigger policy runner (new; inference only) ───────────────────────────

def run_policy(days, returns_for_day, build_graph, signal_for_day, fire_rule, max_stale, rc):
    """
    For each day: update correlations incrementally, compute signals using only
    information available that day, and rebuild the graph only if the rule fires
    (or the graph is older than max_stale days). Returns the graph used on each day
    and the list of rebuild days.
    """
    C_ref = rc.corr()
    graph = build_graph(C_ref)
    last, graphs, rebuilds = 0, {}, []
    for i, d in enumerate(days):
        rc.push(returns_for_day(d))          # alignment ported exactly from Phase 3 graph.py
        C = rc.corr()
        s_corr = np.linalg.norm(C - C_ref) / np.linalg.norm(C_ref)
        s_delta = signal_for_day(d)          # Mamba Δ for day d, computed before message passing
        if fire_rule(s_corr, s_delta) or (i - last) >= max_stale:
            graph, C_ref, last = build_graph(C), C, i
            rebuilds.append(d)
        graphs[d] = graph
    return graphs, rebuilds


# daily = rule always fires; frozen_at_start = rule never fires and max_stale = ∞.
def rule_daily(s_corr, s_delta):
    return True


def rule_never(s_corr, s_delta):
    return False


def make_rule(signal: str, tau_corr: float | None, tau_delta: float | None):
    if signal == "corr_drift":
        return lambda sc, sd: sc > tau_corr
    if signal == "mamba_delta":
        return lambda sc, sd: sd > tau_delta
    if signal == "combined":
        return lambda sc, sd: (sc > tau_corr) or (sd > tau_delta)
    raise ValueError(signal)


# ─── threshold calibration on TRAINING days (leakage-free) ─────────────────

def simulate_rate(sim_days, C_steps, delta_by_day, rule, max_stale) -> float:
    """Realised rebuild rate of `rule` over sim_days, using the SAME
    run_policy as inference (graph building stubbed out)."""
    pc = PrecomputedCorr(C_steps)
    zero = np.zeros(1)
    _, rebuilds = run_policy(
        sim_days, lambda d: zero, lambda C: None,
        lambda d: float(delta_by_day[d]) if delta_by_day is not None else 0.0,
        rule, max_stale, pc)
    return len(rebuilds) / max(1, len(sim_days))


def _bisect(rate_fn, lo, hi, target, iters=20):
    """Threshold whose rate is closest to target (rate is non-increasing in
    the threshold)."""
    best_t, best_err = hi, abs(rate_fn(hi) - target)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        r = rate_fn(mid)
        err = abs(r - target)
        if err < best_err:
            best_t, best_err = mid, err
        if r > target:
            lo = mid
        else:
            hi = mid
    return best_t


def calibrate_thresholds(signal, target_rate, sim_days, C_steps, delta_by_day, max_stale,
                         iters=20, corr_cache=None):
    """{'tau_corr', 'tau_delta', 'sim_rate'} for one (fold, model, signal, target).
    `corr_cache` memoises the corr-drift threshold per rate: it depends only on
    the fold's correlation history, not on the model."""
    corr_cache = {} if corr_cache is None else corr_cache
    d_vals = np.array([delta_by_day[d] for d in sim_days]) if delta_by_day is not None else None

    def corr_tau(rate):
        key = round(float(rate), 9)
        if key not in corr_cache:
            corr_cache[key] = _corr_tau(rate)
        return corr_cache[key]

    def _corr_tau(rate):
        f = lambda t: simulate_rate(sim_days, C_steps, None, make_rule("corr_drift", t, None), max_stale)
        return _bisect(f, 0.0, 5.0, rate, iters)

    def delta_tau(rate):
        f = lambda t: simulate_rate(sim_days, C_steps, delta_by_day, make_rule("mamba_delta", None, t), max_stale)
        return _bisect(f, float(d_vals.min()) - 1e-9, float(d_vals.max()) + 1e-9, rate, iters)

    if signal == "corr_drift":
        tc, td = corr_tau(target_rate), None
        rule = make_rule(signal, tc, None)
    elif signal == "mamba_delta":
        tc, td = None, delta_tau(target_rate)
        rule = make_rule(signal, None, td)
    elif signal == "combined":
        # One shared per-signal rate q; each signal's threshold is its own
        # single-signal calibration at q; bisect q so the UNION hits target.
        cache = {}

        def pair(q):
            if q not in cache:
                cache[q] = (corr_tau(q), delta_tau(q))
            return cache[q]

        def union_rate(q):
            tc_, td_ = pair(q)
            return simulate_rate(sim_days, C_steps, delta_by_day, make_rule("combined", tc_, td_), max_stale)

        # union rate rises with q, _bisect expects a rate that falls with its argument → search s = 1 − q
        s_ = _bisect(lambda x: union_rate(round(1.0 - x, 9)), 0.0, 1.0, target_rate, iters)
        q = round(1.0 - s_, 9)
        tc, td = pair(q)
        rule = make_rule(signal, tc, td)
    else:
        raise ValueError(signal)
    return {"tau_corr": None if tc is None else float(tc),
            "tau_delta": None if td is None else float(td),
            "sim_rate": float(simulate_rate(sim_days, C_steps, delta_by_day, rule, max_stale))}


# ─── Phase 4B inference engine ─────────────────────────────────────────────

def get_split_g(fold_name: str, split: str, lookback: int, graph: str = "baseline"):
    """Split tensors with Phase 3's graph-coverage rule applied against the given graph
    variant's dates — the same call training uses, so datasets match."""
    gd = set(load_daily_edges(graph).keys())
    return d_data.get_split(fold_name, split, lookback=lookback, graph_dates=gd, graph_key=graph)


_RET_ROWS: dict = {}


def return_rows(rets: pd.DataFrame) -> dict:
    if not _RET_ROWS:
        for d, r in zip(rets.index, rets.to_numpy(dtype=np.float64)):
            _RET_ROWS[pd.Timestamp(d)] = r
    return _RET_ROWS


class ModelInference:
    """A trained checkpoint put to inference use: encodings and Δ per day are
    computed once; any graph policy then only needs the cheap propagate step."""

    def __init__(self, ckpt_path, device, snapshot: dict):
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        self.ck = ck
        self.arm, self.fold, self.seed = ck["arm"], ck["fold"], int(ck["seed"])
        self.lookback, self.encoder_type = int(ck["lookback"]), ck["encoder"]
        self.graph = ck.get("graph", "baseline")
        self.device = device
        self.model = d_models.make_model(self.encoder_type, snapshot["CONFIG"]["mamba"])
        self.model.load_state_dict(ck["model_state_dict"])
        self.model.to(device).eval()
        self._enc: dict = {}                 # split -> (h [n,N,d], delta [n] | None)

    def encode(self, split: str, batch: int = 64):
        if split in self._enc:
            return self._enc[split]
        sd = get_split_g(self.fold, split, self.lookback, self.graph)
        X = torch.from_numpy(sd.X).float()
        hs, ds = [], []
        with torch.inference_mode():
            for i in range(0, len(X), batch):
                xb = X[i: i + batch].to(self.device)
                hs.append(self.model.encode(xb))
                if self.encoder_type == "mamba":
                    # signal read after encode, before propagate (spec §12)
                    ds.append(self.model.encoder.delta_per_day(len(xb)).cpu())
        h = torch.cat(hs, 0)
        delta = torch.cat(ds, 0).numpy().astype(np.float64) if ds else None
        self._enc[split] = (h, delta)
        return self._enc[split]

    def split_data(self, split: str):
        return get_split_g(self.fold, split, self.lookback, self.graph)

    def delta_by_day(self, splits) -> dict | None:
        out = {}
        for s in splits:
            _, delta = self.encode(s)
            if delta is None:
                return None
            for d, v in zip(self.split_data(s).dates, delta):
                out[pd.Timestamp(d)] = float(v)
        return out

    def predict_with_graphs(self, split: str, graphs: dict, batch: int = 64) -> np.ndarray:
        """graphs: {date: (ei, et)} covering the split's sample dates."""
        h, _ = self.encode(split)
        dates = [pd.Timestamp(d) for d in self.split_data(split).dates]
        preds = []
        with torch.inference_mode():
            for i in range(0, len(dates), batch):
                dd = dates[i: i + batch]
                ei = [graphs[d][0].to(self.device) for d in dd]
                et = [graphs[d][1].to(self.device) for d in dd]
                preds.append(self.model.propagate_with_edge_lists(h[i: i + len(dd)], ei, et).cpu().numpy())
        return np.concatenate(preds, 0)

    def frame(self, model_name: str, split: str, preds: np.ndarray) -> pd.DataFrame:
        sd = self.split_data(split)
        df = d_eval.make_pred_frame(model_name, self.fold, self.seed, split, sd.dates,
                                     sd.tickers, preds, sd.y)
        if split == "val":
            df["val_half"] = d_eval.val_half_labels(self.fold, df["date"])
        return df

    def delta_channels(self, split: str) -> np.ndarray:
        """[n_days, d_inner] per-day, per-channel Δ of the last layer's final step (diagnostics)."""
        sd = self.split_data(split)
        X = torch.from_numpy(sd.X).float()
        out = []
        with torch.inference_mode():
            for i in range(0, len(X), 64):
                xb = X[i: i + 64].to(self.device)
                self.model.encode(xb)
                out.append(self.model.encoder.delta_channels_per_day(len(xb)).cpu())
        return torch.cat(out, 0).numpy()


def policy_days(rets: pd.DataFrame, fold, end_date) -> list:
    anchor = pd.Timestamp(fold.train_dates[-1])
    idx = rets.index
    return [pd.Timestamp(d) for d in idx[(idx > anchor) & (idx <= pd.Timestamp(end_date))]]


def run_named_policy(policy: dict, thresholds: dict | None, fold, rets: pd.DataFrame,
                     end_date, delta_by_day: dict | None, cfg: dict, window: int,
                     max_stale: float):
    """One policy over (anchor = last training day, end_date]. Returns
    (graphs, rebuild_days, stats). `policy`: {'kind': 'daily'|'frozen'|'trigger',
    'signal': ...}; `thresholds`: {'tau_corr','tau_delta'} for trigger kinds."""
    build_graph = make_graph_builder(cfg)
    build_time = [0.0]

    def timed_build(C):
        t0 = time.perf_counter()
        g = build_graph(C)
        build_time[0] += time.perf_counter() - t0
        return g

    kind = policy["kind"]
    if kind == "daily":
        rule, stale = rule_daily, max_stale
    elif kind == "frozen":
        rule, stale = rule_never, float("inf")
    else:
        rule = make_rule(policy["signal"], thresholds["tau_corr"], thresholds["tau_delta"])
        stale = max_stale
    days = policy_days(rets, fold, end_date)
    rows = return_rows(rets)
    rc = warm_rolling_corr(rets, fold.train_dates[-1], window)
    sig = (lambda d: delta_by_day.get(d, 0.0)) if delta_by_day is not None else (lambda d: 0.0)
    t0 = time.perf_counter()
    graphs, rebuilds = run_policy(days, lambda d: rows[d], timed_build, sig, rule, stale, rc)
    total = time.perf_counter() - t0
    return graphs, rebuilds, {"n_days": len(days), "n_rebuilds": len(rebuilds),
                              "rebuild_rate": len(rebuilds) / max(1, len(days)),
                              "graph_build_seconds": build_time[0], "policy_seconds": total}



# ─── shared context for the policy jobs (built once per process) ───────────

_CTX: dict = {}


class PolicyContext:
    """Returns, correlation matrices by date and a threshold memo for one graph variant (the winner's window)."""

    def __init__(self, graph: str, max_stale: float):
        self.graph, self.max_stale = graph, max_stale
        self.cfg = d_data.load_cfg(graph)
        self.W = int(self.cfg["graph"]["corr_window"])
        self.rets = universe_returns(self.cfg)
        self.corr = corr_by_date(self.rets, self.W)
        self.corr_cache: dict = {}
        self.folds = {f.name: f for f in d_data.tables()["folds"]}

    def calibrate(self, mi, policy: dict) -> dict:
        """Thresholds for one trigger policy, calibrated on the fold's TRAINING days only."""
        days = [pd.Timestamp(d) for d in mi.split_data("train").dates]
        pos = self.rets.index.get_indexer(days)
        assert (np.diff(pos) == 1).all(), "training sample dates are not contiguous in the return calendar"
        steps = [self.corr[self.rets.index[pos[0] - 1]]] + [self.corr[d] for d in days]
        dtrain = mi.delta_by_day(["train"]) if mi.encoder_type == "mamba" else None
        return calibrate_thresholds(policy["signal"], policy["target_rate"], days, steps, dtrain, self.max_stale,
                                    corr_cache=self.corr_cache.setdefault((mi.fold, mi.graph), {}))


def get_ctx(graph: str, max_stale: float) -> PolicyContext:
    k = (graph, max_stale)
    if k not in _CTX:
        _CTX[k] = PolicyContext(graph, max_stale)
    return _CTX[k]


def latency_ms(arm: str, graph: str, device, n_passes: int = 200) -> float:
    """Median wall time of one single-day inference pass (encode + propagate), synchronised."""
    from dmrgt import train as d_train
    a = d_data.load_snapshot()["CONFIG"]["arms"][arm]
    d_train.set_seed(0)
    net = d_train.build_model(arm).to(device).eval()
    fold = d_data.load_snapshot()["CONFIG"]["folds_selection"][-1]["name"]
    sd = get_split_g(fold, "val", a["lookback"], graph)
    edges = load_daily_edges(graph)
    x = torch.from_numpy(np.asarray(sd.X[:1])).float().to(device)
    ei, et = (t.to(device) for t in edges[pd.Timestamp(sd.dates[0])])
    times = []
    with torch.inference_mode():
        for i in range(n_passes + 10):
            if device.type == "cuda":
                torch.cuda.synchronize()
            t = time.perf_counter()
            net(x, ei, et)
            if device.type == "cuda":
                torch.cuda.synchronize()
            if i >= 10:
                times.append((time.perf_counter() - t) * 1000)
    return float(np.median(times))


def bench_scale(N: int, window: int, T: int, rate_sel: float, enc: str, lookback: int, mamba_cfg: dict,
                cfg_w: dict, device, seed: int = 0, n_sample: int = 20) -> dict:
    """Extended compute benchmark (Phase 4.1 C3): graph construction vs model inference at N stocks."""
    rng = np.random.default_rng(seed)
    nf = max(3, N // 10)
    L1, L2 = rng.normal(size=(nf, N)), rng.normal(size=(nf, N))
    Fm = rng.normal(size=(T + window, nf))
    R = np.where((np.arange(T + window) < (T + window) // 2)[:, None], Fm @ L1, Fm @ L2) * 0.01 + rng.normal(size=(T + window, N)) * 0.01
    sec = rng.integers(0, 11, size=N)
    grp = [np.flatnonzero(sec == k) for k in range(11)]
    src, dst = [], []
    for g in grp:                                               # intra-sector directed edges (both ways)
        s_, d_ = np.meshgrid(g, g, indexing="ij"); m = s_ != d_
        src.append(s_[m]); dst.append(d_[m])
    sec_src, sec_dst = np.concatenate(src).tolist(), np.concatenate(dst).tolist()
    bld = lambda C: edges_from_corr(C, cfg_w, sec_src, sec_dst)
    sync = (lambda: torch.cuda.synchronize()) if device.type == "cuda" else (lambda: None)
    days = np.linspace(window, T + window - 1, n_sample).astype(int)
    t_full, t_edges, t_xfer = [], [], []
    for d in days:
        t = time.perf_counter(); C = np.corrcoef(R[d - window + 1: d + 1].T); t_full.append(time.perf_counter() - t)
        t = time.perf_counter(); ei, et = bld(C); t_edges.append(time.perf_counter() - t)
        sync(); t = time.perf_counter(); ei.to(device); et.to(device); sync(); t_xfer.append(time.perf_counter() - t)
    rc = RollingCorr(window, N)
    for r in R[:window]:
        rc.push(r)
    t_inc = []
    for r in R[window: window + min(T, 120)]:                   # per-update cost is flat in t: time 120 updates
        t = time.perf_counter(); rc.push(r); rc.corr(); t_inc.append(time.perf_counter() - t)
    # model inference for this N (winner architecture, untrained weights, random inputs), timed on the device
    net = d_models.make_model(enc, mamba_cfg).to(device).eval()
    try:
        x = torch.randn(1, N, lookback, 8, device=device)
        e1, e2 = ei.to(device), et.to(device)
        with torch.inference_mode():
            for _ in range(3):
                net(x, e1, e2)
            t_model = []
            for _ in range(10):
                sync(); t = time.perf_counter(); net(x, e1, e2); sync(); t_model.append(time.perf_counter() - t)
        t_model = float(np.median(t_model))
    except torch.cuda.OutOfMemoryError:
        t_model = float("nan"); torch.cuda.empty_cache()
    del net
    m = lambda a: float(np.median(a))
    tf, te, tx, ti = m(t_full), m(t_edges), m(t_xfer), m(t_inc)
    r = rate_sel
    g_full = tf + te + tx                                       # full recompute every day
    g_inc = ti + te + tx                                        # incremental, rebuild every day
    g_trig = ti + r * (te + tx)                                 # incremental + selected trigger rate
    return {"n_stocks": N, "window": window, "days": T, "ms_full_recompute": tf * 1e3, "ms_incremental_update": ti * 1e3,
            "ms_edge_build": te * 1e3, "ms_edge_transfer": tx * 1e3, "ms_model_inference": t_model * 1e3,
            "trigger_rebuild_rate": r,
            "graph_ms_per_day_full_daily": g_full * 1e3, "graph_ms_per_day_incremental_daily": g_inc * 1e3,
            "graph_ms_per_day_incremental_trigger": g_trig * 1e3,
            "graph_share_full_daily": g_full / (g_full + t_model), "graph_share_incremental_daily": g_inc / (g_inc + t_model),
            "graph_share_incremental_trigger": g_trig / (g_trig + t_model),
            "total_s_full_daily": (g_full + t_model) * T, "total_s_incremental_trigger": (g_trig + t_model) * T}


In [ ]:
%%writefile /kaggle/working/dmrgt/models.py
"""DMRGT Phase 5 — models (Phase 4 spec §6.3 + Phase 4.1 F1).

Ported verbatim from Phase 3 model.py: PositionalEncoding, TransformerEncoder
(arm A), RelationMessagePassing (hand-rolled R-GCN), stack_per_sample_edges,
and the concat prediction head. New: MambaEncoder (pure PyTorch), and the
split forward (encode / propagate) on DMRGT so the Phase 4B graph decision
can happen between the two steps.
"""

from __future__ import annotations

import math
from collections.abc import Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (Vaswani et al. 2017)."""

    def __init__(self, d_model: int, max_len: int = 512) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        divterm = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * divterm)
        pe[:, 1::2] = torch.cos(position * divterm)
        # [1, max_len, d_model] so it broadcasts against [B, T, D]
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]
        return x + self.pe[:, : x.size(1), :]


class TransformerEncoder(nn.Module):
    """Temporal encoder: 2 layers, 4 heads, causal mask over the
    lookback axis, 8 → 64. Byte-identical to Phase 1/2 — every trained
    Phase 3 model reuses this exact backbone."""

    def __init__(
        self,
        n_features: int = 8,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        dim_ff: int = 256,
        dropout: float = 0.1,
        max_len: int = 64,
    ) -> None:
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers

        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc = PositionalEncoding(d_model=d_model, max_len=max_len)
        self.input_dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def encode(self, X: torch.Tensor) -> torch.Tensor:
        """X: [B, n_stocks, lookback, n_features] → [B, n_stocks, d_model].

        Each stock's lookback window is encoded independently along the
        time axis with a causal mask (position t sees positions ≤ t);
        the last hidden state is the stock's embedding.
        """
        if X.dim() != 4:
            raise ValueError(
                f"encode() expects 4D input [B, n_stocks, L, F], "
                f"got {tuple(X.shape)}"
            )
        B, n_stocks, lookback, n_features = X.shape
        assert n_features == self.n_features, (
            f"Model built for n_features={self.n_features} but got "
            f"n_features={n_features}"
        )
        h = X.reshape(B * n_stocks, lookback, n_features)
        h = self.input_proj(h)
        h = self.pos_enc(h)
        h = self.input_dropout(h)

        causal_mask = torch.triu(
            torch.ones(lookback, lookback, dtype=torch.bool, device=X.device),
            diagonal=1,
        )
        h = self.encoder(h, mask=causal_mask)  # [B*n_stocks, lookback, d_model]
        h_last = h[:, -1, :]                   # [B*n_stocks, d_model]
        return h_last.reshape(B, n_stocks, self.d_model)


class RelationMessagePassing(nn.Module):
    """Hand-rolled R-GCN layer: 3 relation-specific [in_dim, out_dim]
    weight matrices, aggregation via index_add_. No PyG dependency —
    with 40 nodes and 640 edges this is fast on CPU and GPU and carries
    zero dependency risk when Kaggle bumps its torch version.
    """

    def __init__(
        self,
        in_dim: int = 64,
        out_dim: int = 64,
        num_relations: int = 3,
    ) -> None:
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_relations = num_relations

        # One weight matrix per relation type (asserted at construction)
        self.W = nn.ParameterList([
            nn.Parameter(torch.empty(in_dim, out_dim))
            for _ in range(num_relations)
        ])
        self.bias = nn.Parameter(torch.zeros(out_dim))
        self.reset_parameters()
        assert len(self.W) == 3, (
            f"R-GCN must have exactly 3 relation weight matrices, got "
            f"{len(self.W)}"
        )

    def reset_parameters(self) -> None:
        for w in self.W:
            nn.init.xavier_uniform_(w)

    def forward(
        self,
        x: torch.Tensor,            # [N, in_dim]
        edge_index: torch.Tensor,    # [2, E]
        edge_type: torch.Tensor,     # [E]
    ) -> torch.Tensor:
        """Z = Σ_r ( aggregate_{j→i, r} x_j @ W_r ) + bias → [N, out_dim].

        Every node has the same degree per relation by graph
        construction (k_pos, k_neg out-edges per stock; 4 sector peers),
        so mean-vs-sum aggregation only differs by a constant factor
        per relation — absorbed by the weight matrices.
        """
        N = x.size(0)
        device = x.device
        out = torch.zeros(N, self.out_dim, device=device)

        for r in range(self.num_relations):
            mask = edge_type == r
            if not mask.any():
                continue
            src = edge_index[0, mask]
            dst = edge_index[1, mask]
            msgs = x[src] @ self.W[r]     # [E_r, out_dim]
            out = out.index_add(0, dst, msgs)

        return out + self.bias


def stack_per_sample_edges(
    edge_indices: Sequence[torch.Tensor],
    edge_types: Sequence[torch.Tensor],
    n_nodes: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Concatenate per-sample edge lists, shifting sample b's node ids by
    b*n_nodes.

    Flattening a batch of B dates gives [B*n_nodes, d_model] with sample
    b's nodes occupying rows [b*n_nodes, (b+1)*n_nodes). Shifting its node
    ids by that offset lets ONE index_add cover the whole batch instead of
    one R-GCN call per sample.

    The graph and the arithmetic are identical either way — every
    destination node accumulates the same source messages in the same
    order. What changes is the number of kernel launches: with 40 nodes
    and ~640 edges the per-sample call is launch-bound, not compute-bound,
    so B sequential calls cost roughly B times one batched call.
    """
    if len(edge_indices) != len(edge_types):
        raise ValueError(
            f"{len(edge_indices)} edge_index tensors but "
            f"{len(edge_types)} edge_type tensors"
        )
    if not edge_indices:
        raise ValueError("no per-sample edge lists given")
    ei = torch.cat([e + b * n_nodes for b, e in enumerate(edge_indices)],
                   dim=1)
    et = torch.cat(list(edge_types), dim=0)
    return ei, et


# ─── Mamba encoder (new in Phase 4; pure PyTorch, no compiled kernels) ─────

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def selective_scan_seq(u, delta, A, B, C, D):
    """Reference implementation (one Python step per time step) — kept for the tests and as the fallback.
    u, delta: (Bt, L, Di); A: (Di, S); B, C: (Bt, L, S); D: (Di,)"""
    Bt, L, Di = u.shape
    h = torch.zeros(Bt, Di, A.shape[1], device=u.device, dtype=u.dtype)
    ys = []
    for t in range(L):
        dA = torch.exp(delta[:, t].unsqueeze(-1) * A)
        dBu = delta[:, t].unsqueeze(-1) * B[:, t].unsqueeze(1) * u[:, t].unsqueeze(-1)
        h = dA * h + dBu
        ys.append((h * C[:, t].unsqueeze(1)).sum(-1))
    return torch.stack(ys, dim=1) + u * D


SCAN_CHUNK, SCAN_MAX_LOG = 8, 60.0          # pre-registered in CONFIG["selective_scan"]; set at import by the notebook


def selective_scan_chunked(u, delta, A, B, C, D, chunk: int | None = None, max_log: float | None = None):
    """Same recurrence h_t = a_t h_{t-1} + b_t (a_t = exp(delta_t A) in (0, 1]) as selective_scan_seq, evaluated
    chunk by chunk. Inside a chunk: h_t = exp(c_t) (h_0 + sum_{k<=t} exp(-c_k) b_k) with c_t = cumsum(log a).
    A chunk falls back to the step-by-step path whenever exp(-c) could overflow (max -c > max_log)."""
    chunk = SCAN_CHUNK if chunk is None else chunk
    max_log = SCAN_MAX_LOG if max_log is None else max_log
    Bt, L, Di = u.shape
    h = u.new_zeros(Bt, Di, A.shape[1])
    ys = []
    for s in range(0, L, chunk):
        e = min(s + chunk, L)
        la = delta[:, s:e].unsqueeze(-1) * A                                    # (Bt, l, Di, S), <= 0
        b = delta[:, s:e].unsqueeze(-1) * B[:, s:e].unsqueeze(2) * u[:, s:e].unsqueeze(-1)
        c = torch.cumsum(la, dim=1)
        if (-c).max().item() > max_log:                                          # overflow guard -> sequential chunk
            SCAN_STATS["fallback_chunks"] += 1
            hs_list = []
            for t in range(e - s):
                h = torch.exp(la[:, t]) * h + b[:, t]
                hs_list.append(h)
            hs = torch.stack(hs_list, 1)
        else:
            SCAN_STATS["fast_chunks"] += 1
            hs = torch.exp(c) * (h.unsqueeze(1) + torch.cumsum(torch.exp(-c) * b, dim=1))
            h = hs[:, -1]
        ys.append((hs * C[:, s:e].unsqueeze(2)).sum(-1))
    return torch.cat(ys, 1) + u * D


SCAN_STATS = {"fast_chunks": 0, "fallback_chunks": 0}


class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2,
                 dt_min=1e-3, dt_max=1e-1, dt_init_floor=1e-4, dt_scale=1.0):
        super().__init__()
        self.d_inner, self.d_state = expand * d_model, d_state
        self.dt_rank = math.ceil(d_model / 16)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                groups=self.d_inner, padding=d_conv - 1)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        # --- F1: standard Mamba dt initialisation ---
        dt_init_std = (self.dt_rank ** -0.5) * dt_scale
        nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        dt = torch.exp(torch.rand(self.d_inner) * (math.log(dt_max) - math.log(dt_min))
                       + math.log(dt_min)).clamp(min=dt_init_floor)
        inv_softplus_dt = dt + torch.log(-torch.expm1(-dt))      # softplus(bias) == dt
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_softplus_dt)
        self.dt_proj.bias._no_reinit = True
        self.dt_proj.bias._no_weight_decay = True
        # --------------------------------------------

        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.A_log._no_weight_decay = True
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.D._no_weight_decay = True
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.last_delta = None                    # read-only, for Phase 4B signal + explainability

    def forward(self, x):
        L = x.shape[1]
        u, z = self.in_proj(x).chunk(2, dim=-1)
        u = F.silu(self.conv1d(u.transpose(1, 2))[..., :L].transpose(1, 2))   # causal conv
        dt, B, C = torch.split(self.x_proj(u), [self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(dt))
        self.last_delta = delta.detach()
        y = selective_scan_chunked(u, delta, -torch.exp(self.A_log), B, C, self.D)
        return self.out_proj(y * F.silu(z))


class MambaEncoder(nn.Module):
    def __init__(self, n_features=8, d_model=64, n_layers=2, d_state=16, d_conv=4,
                 expand=2, dropout=0.1, d_out=None, **dt_kw):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.layers = nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand, **dt_kw) for _ in range(n_layers)])
        self.norms = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.final_norm = RMSNorm(d_model)
        self.out = nn.Linear(d_model, d_out) if (d_out and d_out != d_model) else nn.Identity()
        self.return_sequence = False              # True only inside the causality test
        self.grad_checkpoint = False              # O5: recompute each block's forward in backward (same math, less memory)

    def forward(self, x):
        h = self.input_proj(x)
        for norm, block in zip(self.norms, self.layers):
            if self.grad_checkpoint and self.training and torch.is_grad_enabled():
                from torch.utils.checkpoint import checkpoint
                h = h + self.dropout(checkpoint(block, norm(h), use_reentrant=False))
            else:
                h = h + self.dropout(block(norm(h)))
        h = self.final_norm(h)
        return h if self.return_sequence else self.out(h[:, -1])

    def delta_signal(self):
        """Mean memory-reset strength at the final step (last layer), one number per call."""
        return float(self.layers[-1].last_delta[:, -1].mean())

    def delta_channels_per_group(self, n_groups: int):
        """Last layer, final step, averaged over stocks → [n_groups, d_inner] (diagnostics)."""
        d = self.layers[-1].last_delta[:, -1]
        return d.reshape(n_groups, -1, d.shape[-1]).mean(1)

    def a_min(self) -> float:
        """Smallest state decay rate of the last layer: min exp(A_log)."""
        return float(torch.exp(self.layers[-1].A_log).min())

    def delta_per_sample(self, n_groups: int):
        """Per-group version of delta_signal for a batch of n_groups days:
        last layer, final step, averaged over stocks and channels → [n_groups]."""
        d = self.layers[-1].last_delta[:, -1]                # [n_groups * n_stocks, d_inner]
        return d.reshape(n_groups, -1).mean(-1)


class MambaStockEncoder(nn.Module):
    """Applies MambaEncoder independently to every stock's lookback window.
    X: [B, n_stocks, L, F] → [B, n_stocks, d_model] (same contract as
    TransformerEncoder.encode)."""

    def __init__(self, n_features=8, d_model=64, n_layers=2, d_state=16, d_conv=4,
                 expand=2, dropout=0.1, **dt_kw):
        super().__init__()
        self.n_features, self.d_model = n_features, d_model
        self.mamba = MambaEncoder(n_features, d_model, n_layers, d_state, d_conv,
                                  expand, dropout, d_out=d_model, **dt_kw)

    def encode(self, X: torch.Tensor) -> torch.Tensor:
        if X.dim() != 4:
            raise ValueError(f"encode() expects 4D input [B, n_stocks, L, F], got {tuple(X.shape)}")
        B, n_stocks, lookback, n_features = X.shape
        assert n_features == self.n_features
        h = self.mamba(X.reshape(B * n_stocks, lookback, n_features))   # [B*n_stocks, d_model]
        return h.reshape(B, n_stocks, self.d_model)

    def delta_signal(self):
        return self.mamba.delta_signal()

    def delta_per_day(self, n_days: int):
        return self.mamba.delta_per_sample(n_days)

    def delta_channels_per_day(self, n_days: int):
        return self.mamba.delta_channels_per_group(n_days)


# ─── DMRGT with a split forward ────────────────────────────────────────────

class DMRGT(nn.Module):
    """Same R-GCN + concat head as Phase 3; the temporal encoder is chosen
    by `encoder` ('transformer' → arm A, 'mamba' → arms B/C/D).

    Construction order (encoder → rgcn → head) is the Phase 3 order, so the
    transformer arm draws the same initial weights as Phase 3 for a seed.

      encode(x)               x [B, N, L, F]  → h [B, N, d]
      propagate(h, ei, et)    R-GCN + concat head → [B, N]
      forward                 propagate(encode(x), ...)
    """

    def __init__(self, encoder: str = "transformer", n_features: int = 8,
                 d_model: int = 64, n_heads: int = 4, n_layers: int = 2,
                 dim_ff: int = 256, dropout: float = 0.1, max_len: int = 64,
                 num_relations: int = 3, mamba_cfg: dict | None = None) -> None:
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model
        self.encoder_type = encoder
        if encoder == "transformer":
            self.encoder = TransformerEncoder(
                n_features=n_features, d_model=d_model, n_heads=n_heads,
                n_layers=n_layers, dim_ff=dim_ff, dropout=dropout, max_len=max_len)
        elif encoder == "mamba":
            mc = dict(mamba_cfg or {})
            self.encoder = MambaStockEncoder(
                n_features=n_features, d_model=mc.get("d_model", d_model),
                n_layers=mc.get("n_layers", 2), d_state=mc.get("d_state", 16),
                d_conv=mc.get("d_conv", 4), expand=mc.get("expand", 2),
                dropout=mc.get("dropout", dropout),
                dt_min=mc.get("dt_min", 1e-3), dt_max=mc.get("dt_max", 1e-1),
                dt_init_floor=mc.get("dt_init_floor", 1e-4))
            assert mc.get("d_model", d_model) == d_model
        else:
            raise ValueError(f"unknown encoder {encoder!r}")
        self.rgcn = RelationMessagePassing(in_dim=d_model, out_dim=d_model,
                                           num_relations=num_relations)
        self.head = nn.Sequential(
            nn.Linear(2 * d_model, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def _as_batched(self, X: torch.Tensor) -> tuple[torch.Tensor, bool]:
        if X.dim() == 3:
            x, squeeze = X.unsqueeze(0), True
        elif X.dim() == 4:
            x, squeeze = X, False
        else:
            raise ValueError(f"Expected X [n_stocks, L, F] or [B, n_stocks, L, F], got {tuple(X.shape)}")
        assert X.shape[-1] == self.n_features
        return x, squeeze

    def encode(self, X: torch.Tensor) -> torch.Tensor:
        """X [N, L, F] or [B, N, L, F] → h [N, d] or [B, N, d]."""
        x, squeeze = self._as_batched(X)
        h = self.encoder.encode(x)
        return h.squeeze(0) if squeeze else h

    def propagate(self, h: torch.Tensor, edge_index: torch.Tensor,
                  edge_type: torch.Tensor) -> torch.Tensor:
        """h [N, d] or [B, N, d]; edges over the flattened batch nodes
        (stack_per_sample_edges). R-GCN + concat head → [N] or [B, N]."""
        squeeze = h.dim() == 2
        if squeeze:
            h = h.unsqueeze(0)
        B, n_stocks = h.size(0), h.size(1)
        Z = self.rgcn(h.reshape(B * n_stocks, self.d_model), edge_index,
                      edge_type).reshape(B, n_stocks, self.d_model)
        fused = torch.cat([h, Z], dim=-1)
        out = self.head(fused).squeeze(-1)
        return out.squeeze(0) if squeeze else out

    def propagate_with_edge_lists(self, h: torch.Tensor,
                                  edge_indices: Sequence[torch.Tensor],
                                  edge_types: Sequence[torch.Tensor]) -> torch.Tensor:
        B, n_stocks = h.size(0), h.size(1)
        if len(edge_indices) != B:
            raise ValueError(f"{len(edge_indices)} edge lists for a batch of {B}")
        ei, et = stack_per_sample_edges(edge_indices, edge_types, n_stocks)
        return self.propagate(h, ei, et)

    def forward(self, X: torch.Tensor, edge_index: torch.Tensor,
                edge_type: torch.Tensor) -> torch.Tensor:
        """One edge list applied to every sample in the batch."""
        x, squeeze = self._as_batched(X)
        B, n_stocks = x.size(0), x.size(1)
        if B == 1:
            ei, et = edge_index, edge_type
        else:
            ei, et = stack_per_sample_edges([edge_index] * B, [edge_type] * B, n_stocks)
        out = self.propagate(self.encoder.encode(x), ei, et)
        return out.squeeze(0) if squeeze else out

    def forward_with_edge_lists(self, X: torch.Tensor,
                                edge_indices: Sequence[torch.Tensor],
                                edge_types: Sequence[torch.Tensor]) -> torch.Tensor:
        """One edge list PER sample (a batch of dates). [B, n_stocks]."""
        x, squeeze = self._as_batched(X)
        out = self.propagate_with_edge_lists(self.encoder.encode(x), edge_indices, edge_types)
        return out.squeeze(0) if squeeze else out


def assert_dt_init(model: nn.Module) -> float:
    """F1: right after construction, mean softplus(dt_proj.bias) must lie in [0.001, 0.1]."""
    vals = [F.softplus(m.dt_proj.bias.detach()).mean().item()
            for m in model.modules() if isinstance(m, MambaBlock)]
    for v in vals:
        assert 0.001 <= v <= 0.1, f"Mamba dt init out of range: mean softplus(dt_proj.bias) = {v:.5f}"
    return float(np.mean(vals)) if vals else float("nan")


def make_model(encoder: str, mamba_cfg: dict | None = None, n_features: int = 8,
               grad_checkpoint: bool = False) -> DMRGT:
    """The one constructor used by training, the probes and Phase 4B inference."""
    m = DMRGT(encoder=encoder, n_features=n_features, mamba_cfg=mamba_cfg)
    if encoder == "mamba":
        assert_dt_init(m)
        m.encoder.mamba.grad_checkpoint = bool(grad_checkpoint)
    return m


def param_groups(model: nn.Module, weight_decay: float):
    """F1: two AdamW groups. No weight decay on A_log, D, dt_proj.bias, biases and norm weights
    (anything tagged _no_weight_decay or with ndim < 2). Same rule for every arm."""
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if getattr(p, "_no_weight_decay", False) or p.ndim < 2:
            no_decay.append(p)
        else:
            decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]



# ─── self-tests shared by the kernel (CPU) and the GPU worker ───────────────

def selftest_scan(device, seed: int = 0) -> dict:
    """Chunked scan == sequential scan: forward and gradients, at initialisation-scale Δ and at 20x Δ (the 20x case
    must take the sequential-fallback path at least once)."""
    out = {}
    for label, mult in (("init_scale_delta", 1.0), ("delta_x20", 20.0)):
        g = torch.Generator().manual_seed(seed)
        Bt, L, Di, S = 4, 24, 8, 16
        u = torch.randn(Bt, L, Di, generator=g)
        delta = torch.exp(torch.rand(Bt, L, Di, generator=g) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3)) * mult
        A = -torch.arange(1, S + 1, dtype=torch.float32).repeat(Di, 1)
        B, C = torch.randn(Bt, L, S, generator=g), torch.randn(Bt, L, S, generator=g)
        D = torch.ones(Di)
        w = torch.randn(Bt, L, Di, generator=g)
        res = {}
        for name, fn in (("seq", selective_scan_seq), ("chunked", selective_scan_chunked)):
            ins = [t.clone().to(device).requires_grad_(True) for t in (u, delta, A, B, C, D)]
            fb0 = SCAN_STATS["fallback_chunks"]
            y = fn(*ins)
            (y * w.to(device)).sum().backward()
            res[name] = (y.detach().cpu(), [t.grad.detach().cpu() for t in ins], SCAN_STATS["fallback_chunks"] - fb0)
        rel = lambda a, b: float((a - b).abs().max() / (b.abs().max() + 1e-12))
        fwd = rel(res["chunked"][0], res["seq"][0])
        grads = [rel(a, b) for a, b in zip(res["chunked"][1], res["seq"][1])]
        out[label] = {"forward_rel_diff": fwd, "grad_rel_diff_max": max(grads), "fallback_chunks": res["chunked"][2]}
        assert fwd < 1e-4, f"{label}: chunked scan forward differs from sequential by {fwd:.2e}"
        assert max(grads) < 1e-4, f"{label}: chunked scan gradients differ from sequential by {max(grads):.2e}"
    assert out["delta_x20"]["fallback_chunks"] >= 1, "the x20 case did not exercise the sequential fallback"
    assert out["init_scale_delta"]["fallback_chunks"] == 0, "initialisation-scale Δ should not need the fallback"
    return out


def selftest_checkpoint(device, seed: int = 0) -> dict:
    """Gradient checkpointing changes memory, not math: same seed and batch -> same loss and gradients."""
    import copy
    torch.manual_seed(seed)
    enc = MambaEncoder(n_features=8, d_model=64, n_layers=2, d_out=64).to(device)
    enc.train()
    x = torch.randn(6, 20, 8, device=device)
    y = torch.randn(6, 64, device=device)
    res = {}
    for flag in (False, True):
        m = copy.deepcopy(enc)
        m.grad_checkpoint = flag
        torch.manual_seed(1234)
        loss = ((m(x) - y) ** 2).mean()
        loss.backward()
        res[flag] = (float(loss), [p.grad.detach().cpu().clone() for p in m.parameters() if p.grad is not None])
    dl = abs(res[False][0] - res[True][0])
    dg = max(float((a - b).abs().max()) for a, b in zip(res[False][1], res[True][1]))
    assert dl <= 1e-6 and dg <= 1e-6 + 1e-4 * max(float(a.abs().max()) for a in res[False][1]), \
        f"checkpointing changed the result: loss diff {dl:.2e}, grad diff {dg:.2e}"
    return {"loss_abs_diff": dl, "grad_abs_diff_max": dg}


In [ ]:
%%writefile /kaggle/working/dmrgt/portfolio.py
"""DMRGT Phase 5 — cost-aware portfolio construction (spec §4.6).

Changes how predictions become positions, not the models. Return and cost accounting are the harness's
(evaluate.backtest_from_weights), which uses the same alignment, turnover definition and cost model as backtest().
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from dmrgt import evaluate as ev


def ensemble_scores(pred: pd.DataFrame) -> pd.DataFrame:
    """pred columns: date, ticker, seed, pred. Returns a date x ticker matrix of ensembled scores."""
    g = pred.groupby(["date", "seed"])["pred"]
    z = (pred["pred"] - g.transform("mean")) / g.transform("std", ddof=0)
    return pred.assign(z=z).groupby(["date", "ticker"])["z"].mean().unstack().sort_index()


def ema_smooth(S: pd.DataFrame, half_life: int) -> pd.DataFrame:
    if half_life == 0:
        return S
    alpha = 1.0 - 0.5 ** (1.0 / half_life)
    return S.ewm(alpha=alpha, adjust=False).mean()   # uses only past and current days


def buffered_weights(S: pd.DataFrame, q_in: float, q_out: float) -> pd.DataFrame:
    """Enter a leg only from the top/bottom q_in; stay in it while inside the top/bottom q_out.
    Equal-weight legs, dollar-neutral (same leg weighting as the ported harness)."""
    assert q_in <= q_out < 0.5, "q_out must be < 0.5 so legs can never overlap"
    N = S.shape[1]
    n_leg, n_out = int(round(q_in * N)), int(round(q_out * N))
    W = np.zeros(S.shape)
    longs, shorts = set(), set()
    for t, row in enumerate(S.to_numpy()):
        desc = np.argsort(-row, kind="stable")
        rank_desc = np.empty(N, dtype=int); rank_desc[desc] = np.arange(N)
        asc = desc[::-1]
        rank_asc = N - 1 - rank_desc

        longs = {i for i in longs if rank_desc[i] < n_out}
        for i in desc[:n_leg]:
            if len(longs) >= n_leg: break
            longs.add(i)
        shorts = {i for i in shorts if rank_asc[i] < n_out}
        for i in asc[:n_leg]:
            if len(shorts) >= n_leg: break
            shorts.add(i)

        assert not (longs & shorts)
        W[t, list(longs)] = 1.0 / n_leg
        W[t, list(shorts)] = -1.0 / n_leg
    return pd.DataFrame(W, index=S.index, columns=S.columns)


def strategy_series(pred_long: pd.DataFrame, raw_wide: pd.DataFrame, universe: list, bt_cfg: dict,
                    half_life: int, q_in: float, q_out: float) -> tuple[pd.Series, pd.Series]:
    """Ensemble → EMA → buffer → backtest for ONE fold's contiguous prediction series
    (all seeds, validation then test in date order). Returns (daily after-cost returns,
    daily one-way turnover) over every date of the series; callers slice to the window of interest."""
    p = pred_long.rename(columns={"y_pred": "pred"})[["date", "ticker", "seed", "pred"]]
    S = ensemble_scores(p).reindex(columns=universe)
    S = ema_smooth(S, half_life)
    W = buffered_weights(S, q_in, q_out)
    ret, st = ev.backtest_from_weights(W, raw_wide.reindex(columns=universe), bt_cfg["commission_bps"],
                                    bt_cfg["slippage_bps"])
    return ret, st["w_change"] / 2


In [ ]:
%%writefile /kaggle/working/dmrgt/evaluate.py
"""DMRGT Phase 5 — evaluation harness (Phase 4 spec §6.4 + Phase 4.1 F3 + Phase 5 §6).

Ported verbatim from Phase 3 eval.py (incl. the source= keyword at both
load_predictions call sites). Phase 4 edits, all mechanical:
  * `import paths` → d_data (cache / working-dir resolution);
  * evaluate_fold (the closed-form-baseline test path) asserts the test
    lock files exist before reading any test window.
The leg-direction correction and all metric formulas are unchanged.
"""

from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Callable, Iterable

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from dmrgt import data as d_data


# ─── per-day metrics (formulas unchanged from Phase 1/2) ─────────────────

def _huber_loss(residuals: np.ndarray, delta: float) -> np.ndarray:
    """Per-element Huber loss: quadratic for |r|<=delta, linear otherwise."""
    r = np.asarray(residuals, dtype=np.float64)
    abs_r = np.abs(r)
    quadratic = 0.5 * r * r
    linear = delta * (abs_r - 0.5 * delta)
    return np.where(abs_r <= delta, quadratic, linear)


def per_day_rank_ic(pred: np.ndarray, true: np.ndarray) -> float:
    """Spearman rank IC for a single cross-section. NaN if either side
    has zero variance; the aggregator drops NaN days."""
    if pred.size < 2:
        return float("nan")
    if np.std(pred) == 0 or np.std(true) == 0:
        return float("nan")
    rho, _ = spearmanr(pred, true)
    return float(rho) if np.isfinite(rho) else float("nan")


def per_day_directional_acc(pred: np.ndarray, true: np.ndarray) -> float:
    """Fraction of stocks with sign(pred) == sign(true), over valid pairs."""
    mask = np.isfinite(pred) & np.isfinite(true)
    if not mask.any():
        return float("nan")
    return float(np.mean(np.sign(pred[mask]) == np.sign(true[mask])))


def per_day_huber(pred: np.ndarray, true: np.ndarray, delta: float) -> float:
    mask = np.isfinite(pred) & np.isfinite(true)
    if not mask.any():
        return float("nan")
    return float(np.mean(_huber_loss(pred[mask] - true[mask], delta)))


def per_day_mse(pred: np.ndarray, true: np.ndarray) -> float:
    mask = np.isfinite(pred) & np.isfinite(true)
    if not mask.any():
        return float("nan")
    return float(np.mean((pred[mask] - true[mask]) ** 2))


# ─── dataclasses ──────────────────────────────────────────────────────────

@dataclass
class FoldMetrics:
    """All metrics for one model × fold × seed × split."""
    model: str
    fold: str
    seed: int | None
    split: str
    n_days: int
    # Prediction metrics — aggregated across the split's days
    mean_rank_ic: float
    std_rank_ic: float
    directional_acc: float
    huber_loss: float
    mse: float
    # Backtest metrics
    sharpe_after_costs: float
    max_drawdown: float
    turnover: float
    cum_return: float


@dataclass
class FoldResult:
    """Full output of evaluating one model on one fold.

    ``per_day`` is the un-aggregated per-day metric table;
    ``backtest_returns`` is the daily portfolio return series after
    costs (used by stats.py's bootstrap).
    """
    model: str
    fold: str
    seed: int | None
    split: str
    per_day: pd.DataFrame
    fold_metrics: FoldMetrics
    backtest_returns: pd.Series


def summarise(results: Iterable[FoldResult]) -> pd.DataFrame:
    """Flatten FoldResults into a tidy DataFrame."""
    return pd.DataFrame([asdict(r.fold_metrics) for r in results])


# ─── realised raw returns ─────────────────────────────────────────────────

def raw_returns_from_targets(targets: pd.DataFrame,
                             universe: list[str]) -> pd.DataFrame:
    """Recover realised RAW daily returns from the excess-return target.

    The backtest earns the raw return, not the demeaned target; raw[t] =
    excess[t] + cross-sectional mean[t]. Wide frame, date × universe.
    """
    tgt_wide = (targets.pivot(index="date", columns="ticker",
                              values="excess_return")
                .sort_index()
                .reindex(columns=universe))
    return tgt_wide.add(tgt_wide.mean(axis=1), axis=0)


# ─── post-processing (§9, applied identically to both arms — Rule 19) ─────

DEFAULT_POSTPROCESSING: dict = {
    "no_trade_band": 0.0,
    "decile_fraction": None,   # None → fall back to cfg backtest.decile_fraction
    "smoothing_days": 1,
}


def apply_postprocessing(preds: pd.DataFrame,
                         postprocessing: dict | None) -> pd.DataFrame:
    """Apply the Tier-1 post-processing rules to a long predictions frame.

    Rules, in order:
      1. smoothing_days — causal (trailing) rolling mean of y_pred over
         `smoothing_days` dates, per ticker, within each (model, fold,
         seed) group. Never looks ahead. 1 → no-op.
      2. no_trade_band — stocks whose |smoothed prediction| < band are
         flagged untradable on that date (``tradable`` column, False).
         The backtest excludes them from leg selection; rank IC keeps
         scoring them (the band is a trading rule, not a model change).

    decile_fraction travels in the same dict but is a backtest parameter
    (leg size), not a per-row transformation — backtest() reads it.

    Returns a new DataFrame; the input is not mutated.
    """
    if postprocessing is None:
        pp = dict(DEFAULT_POSTPROCESSING)
    else:
        pp = {**DEFAULT_POSTPROCESSING, **{
            k: v for k, v in postprocessing.items()
            if k in DEFAULT_POSTPROCESSING and v is not None
        }}

    df = preds.copy()
    if "tradable" not in df.columns:
        df["tradable"] = True

    # ── 1. causal smoothing ──────────────────────────────────────────
    smooth = int(pp.get("smoothing_days", 1) or 1)
    if smooth > 1:
        # Rolling per ticker within each (model, fold, seed) run — rows
        # of different tickers must never be mixed into one window.
        keys = [c for c in ("model", "fold", "seed", "ticker")
                if c in df.columns]
        df = df.sort_values(keys + ["date"])
        df["y_pred"] = (df.groupby(keys, sort=False)["y_pred"]
                        .transform(lambda s: s.rolling(smooth,
                                                       min_periods=1).mean()))
        df = df.reset_index(drop=True)

    # ── 2. no-trade band ─────────────────────────────────────────────
    band = float(pp.get("no_trade_band", 0.0) or 0.0)
    df["tradable"] = df["y_pred"].abs() >= band

    return df


# ─── the cost-aware backtest (§5.2) ───────────────────────────────────────

def backtest_daily_deciles(
    predictions_by_day: pd.DataFrame,   # date × ticker, values = y_pred
    true_returns_by_day: pd.DataFrame,  # date × ticker, values = realised raw return
    tickers: list[str],
    commission_bps: float,
    slippage_bps: float,
    decile_fraction: float,
    tradable_by_day: pd.DataFrame | None = None,  # date × ticker bool; None = all tradable
) -> tuple[pd.Series, dict]:
    """Long the top decile, short the bottom decile, equal-weighted within
    each leg. Legs are selected among the day's tradable stocks only.

    Positions decided from day-t predictions earn day-t+1 returns
    (weights are shifted one day before multiplying realised returns —
    no same-day lookahead). Commission + slippage are charged on
    weight changes: fee = (commission_bps + slippage_bps) / 1e4, applied
    to half the absolute daily weight change (one-way turnover).

    Returns (daily portfolio return series after costs, stats dict).
    """
    pred = predictions_by_day[tickers]
    rets = true_returns_by_day[tickers]
    trad = tradable_by_day[tickers] if tradable_by_day is not None else None

    n = len(tickers)
    fee = (commission_bps + slippage_bps) / 10_000.0

    weights = pd.DataFrame(0.0, index=pred.index, columns=tickers)
    for i, d in enumerate(pred.index):
        row = pred.loc[d].to_numpy(dtype=float)
        ok = (trad.loc[d].to_numpy(dtype=bool)
              if trad is not None else np.ones(n, dtype=bool))
        idx = np.flatnonzero(ok & np.isfinite(row))
        if idx.size == 0:
            continue  # nothing tradable → flat that day
        scores = row[idx]
        k = max(1, int(round(idx.size * decile_fraction)))
        k = min(k, idx.size)  # degenerate wide legs cancel to ~0 net
        order = np.argsort(-scores, kind="stable")  # descending: best first
        longs = idx[order[:k]]
        shorts = idx[order[-k:]]
        w = np.zeros(n)
        w[longs] += 1.0 / len(longs)
        w[shorts] -= 1.0 / len(shorts)
        weights.iloc[i] = w

    # Positions taken at close t earn returns realised at t+1.
    raw_port = (weights.shift(1).fillna(0) * rets).sum(axis=1)
    # Cost: one-way fee on each unit of weight change.
    w_change = weights.diff().abs().sum(axis=1).fillna(0)
    cost = fee * w_change / 2
    daily_ret = (raw_port - cost).astype(float)
    daily_ret.index = pd.DatetimeIndex(daily_ret.index)

    # Sharpe — annualised, risk-free 0, sample std.
    sd = float(daily_ret.std(ddof=1))
    sharpe = float(daily_ret.mean() / sd * np.sqrt(252)) if sd > 0 else 0.0
    # Max drawdown from the equity curve.
    eq = (1 + daily_ret.fillna(0)).cumprod()
    peak = eq.cummax()
    dd = (eq - peak) / peak
    max_dd = float(-dd.min()) if dd.notna().any() else float("nan")
    # Turnover — one-way, fraction of NAV changing per day.
    turnover = float(w_change.mean() / 2)
    cum = float(eq.iloc[-1] - 1) if len(eq) > 0 else float("nan")

    return daily_ret, {
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "turnover": turnover,
        "cum_return": cum,
    }


def _effective_decile_fraction(cfg: dict, postprocessing: dict | None) -> float:
    """Post-processing decile_fraction overrides the config default."""
    if postprocessing and postprocessing.get("decile_fraction") is not None:
        return float(postprocessing["decile_fraction"])
    return float(cfg["backtest"]["decile_fraction"])


def _score_wide_group(
    wide_pred: pd.DataFrame,
    wide_true: pd.DataFrame,
    raw_wide: pd.DataFrame,
    tradable_wide: pd.DataFrame | None,
    cfg: dict,
    decile_fraction: float,
    universe: list[str],
) -> tuple[pd.DataFrame, pd.Series, dict]:
    """Per-day metrics + backtest for one (model, fold, seed) wide group."""
    dates = wide_pred.index
    per_day_rows = []
    for d in dates:
        y_pred = wide_pred.loc[d, universe].to_numpy()
        y_true = wide_true.loc[d, universe].to_numpy()
        per_day_rows.append({
            "date": d,
            "rank_ic": per_day_rank_ic(y_pred, y_true),
            "dir_acc": per_day_directional_acc(y_pred, y_true),
            "huber": per_day_huber(y_pred, y_true,
                                   delta=cfg["training"]["huber_delta"]),
            "mse": per_day_mse(y_pred, y_true),
        })
    per_day = pd.DataFrame(per_day_rows).set_index("date")

    bt_raw = raw_wide.reindex(dates).reindex(columns=universe)
    daily_ret, bt_stats = backtest_daily_deciles(
        predictions_by_day=wide_pred,
        true_returns_by_day=bt_raw,
        tickers=universe,
        commission_bps=cfg["backtest"]["commission_bps"],
        slippage_bps=cfg["backtest"]["slippage_bps"],
        decile_fraction=decile_fraction,
        tradable_by_day=tradable_wide,
    )
    return per_day, daily_ret, bt_stats


# ─── baseline path: model_fn over a fold's split (§5.3 gate) ──────────────

def evaluate_fold(
    model_name: str,
    model_fn: Callable[[np.ndarray], np.ndarray],
    fold,
    features: pd.DataFrame,
    targets: pd.DataFrame,
    cfg: dict,
    seed: int | None = None,
    lookback: int | None = None,
) -> FoldResult:
    """Run one closed-form model through the per-day harness on one fold.

    Assembles the same windows the trained model sees: for prediction
    date d, features from [d-lookback+1 .. d] INCLUSIVE of d (Rule 3 —
    features come from day t and earlier). Note: Phase 2's baseline
    harness used a window ending at d-1; the trained models' assembly
    always ended at d. Phase 3 unifies on the spec-correct inclusive
    window, so persistence numbers shift slightly vs Phase 2's recorded
    floor; the random gate is unaffected (noise ignores input).
    """
    d_data.assert_test_unlocked()   # Phase 4: this path reads test windows
    universe = list(cfg["universe"]["tickers"])
    n_stocks = len(universe)
    if lookback is None:
        lookback = int(cfg["features"]["lookback_days"])
    feature_cols = list(cfg["features"]["columns"])

    # Reindex to guarantee the feature level follows configs.yaml order —
    # pivot's MultiIndex columns are sorted lexicographically, which would
    # silently reorder X's feature axis if configs ever listed features
    # out of alphabetical order.
    feat_wide = (features.pivot(index="date", columns="ticker",
                                values=feature_cols)
                 .sort_index())
    feat_wide = feat_wide.reindex(columns=pd.MultiIndex.from_product(
        [feature_cols, list(feat_wide.columns.get_level_values("ticker")
                            .unique())],
        names=["feature", "ticker"]))
    tgt_wide = (targets.pivot(index="date", columns="ticker",
                              values="excess_return")
                .sort_index())
    raw_wide = raw_returns_from_targets(targets, universe)

    test_dates = list(fold.test_dates)
    pred_rows, true_rows, daily_metrics = [], [], []
    for d in test_dates:
        if d not in feat_wide.index or d not in tgt_wide.index:
            continue  # e.g. last date dropped from targets
        d_loc = feat_wide.index.get_loc(d)
        if d_loc < lookback - 1:
            continue  # not enough history for an inclusive window
        window_dates = feat_wide.index[d_loc - lookback + 1: d_loc + 1]
        assert len(window_dates) == lookback
        assert window_dates[-1] == d
        X = np.empty((n_stocks, lookback, len(feature_cols)), dtype=np.float64)
        for j, t in enumerate(universe):
            X[j] = feat_wide.loc[window_dates, (slice(None), t)].values
        if not np.isfinite(X).all():
            continue
        y_pred = model_fn(X)
        y_true = tgt_wide.loc[d, universe].values
        pred_rows.append(pd.Series(y_pred, index=universe, name=d))
        true_rows.append(pd.Series(y_true, index=universe, name=d))
        daily_metrics.append({
            "date": d,
            "rank_ic": per_day_rank_ic(y_pred, y_true),
            "dir_acc": per_day_directional_acc(y_pred, y_true),
            "huber": per_day_huber(y_pred, y_true,
                                   delta=cfg["training"]["huber_delta"]),
            "mse": per_day_mse(y_pred, y_true),
        })

    if not daily_metrics:
        raise RuntimeError(
            f"No valid test days for fold {fold.name!r} — check that "
            f"features/targets cover the test window."
        )
    daily_df = pd.DataFrame(daily_metrics).set_index("date")
    pred_df = pd.DataFrame(pred_rows)
    true_df = pd.DataFrame(true_rows)

    _, daily_ret, bt_stats = _score_wide_group(
        pred_df, true_df, raw_wide, None, cfg,
        decile_fraction=float(cfg["backtest"]["decile_fraction"]),
        universe=universe,
    )

    metrics = FoldMetrics(
        model=model_name,
        fold=fold.name,
        seed=seed,
        split="test",
        n_days=len(daily_df),
        mean_rank_ic=float(np.nanmean(daily_df["rank_ic"])),
        std_rank_ic=float(np.nanstd(daily_df["rank_ic"], ddof=1)),
        directional_acc=float(np.nanmean(daily_df["dir_acc"])),
        huber_loss=float(np.nanmean(daily_df["huber"])),
        mse=float(np.nanmean(daily_df["mse"])),
        sharpe_after_costs=bt_stats["sharpe"],
        max_drawdown=bt_stats["max_drawdown"],
        turnover=bt_stats["turnover"],
        cum_return=bt_stats["cum_return"],
    )
    return FoldResult(
        model=model_name,
        fold=fold.name,
        seed=seed,
        split="test",
        per_day=daily_df,
        fold_metrics=metrics,
        backtest_returns=daily_ret,
    )


# ─── predictions-parquet path (trained models) ────────────────────────────

def load_predictions(model_id: str | None = None,
                     split: str | None = None,
                     source: str | Path | None = None) -> pd.DataFrame:
    """Load a Phase 3 predictions parquet.

    Resolution order: explicit `source` (file or directory) →
    WORK/predictions_{model_id}.parquet. Optionally filters to one
    `split` ("val" / "test").
    """
    src = Path(source) if source is not None else d_data.WORK
    if src.is_dir():
        if model_id is None:
            matches = sorted(src.glob("predictions_*.parquet"))
            if not matches:
                raise FileNotFoundError(f"No predictions_*.parquet in {src}")
            if len(matches) > 1 and model_id is None:
                raise ValueError(
                    f"Multiple prediction files in {src}; pass model_id "
                    f"to disambiguate: {[m.name for m in matches]}"
                )
            src = matches[0]
        else:
            src = src / f"predictions_{model_id}.parquet"
    if not src.exists():
        raise FileNotFoundError(
            f"Predictions not found: {src}. Download the Kaggle output "
            f"into artifacts/ first."
        )
    df = pd.read_parquet(src)
    required = {"model", "fold", "seed", "split", "date", "ticker",
                "y_pred", "y_true"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"predictions parquet {src.name} missing columns: {missing}. "
            f"Have: {list(df.columns)}"
        )
    if split is not None:
        df = df[df["split"] == split]
        if df.empty:
            raise ValueError(
                f"{src.name} has no rows with split == {split!r}. "
                f"Splits present: {sorted(pd.read_parquet(src)['split'].unique())}"
            )
    return df


def evaluate_predictions(
    source: str | Path | pd.DataFrame,
    targets: pd.DataFrame,
    cfg: dict,
    split: str | None = None,
    postprocessing: dict | None = None,
    universe: list[str] | None = None,
) -> list[FoldResult]:
    """Re-evaluate a predictions parquet (Phase 3 schema with a `split`
    column) through the same code path as evaluate_fold. One FoldResult
    per (model, fold, seed) — the pairing stats.py's Wilcoxon test uses.
    """
    if universe is None:
        universe = list(cfg["universe"]["tickers"])
    pred_df = (source if isinstance(source, pd.DataFrame)
               else load_predictions(source=source, split=split))
    pred_df = apply_postprocessing(pred_df, postprocessing)

    if split is not None:
        pred_df = pred_df[pred_df["split"] == split]
    if pred_df.empty:
        raise ValueError(f"No prediction rows to evaluate (split={split!r})")
    if pred_df["model"].nunique() > 1:
        raise ValueError(
            "evaluate_predictions refuses to pool multiple models into one "
            "result set — evaluate one arm at a time so the two arms of "
            "the comparison can never be silently mixed."
        )

    raw_wide = raw_returns_from_targets(targets, universe)
    frac = _effective_decile_fraction(cfg, postprocessing)

    results: list[FoldResult] = []
    for (model_name, fold_name, seed), grp in pred_df.groupby(
            ["model", "fold", "seed"]):
        # A group that mixed splits would silently blend the val and test
        # windows into one return series — refuse loudly instead.
        assert grp["split"].nunique() == 1, (
            f"group model={model_name} fold={fold_name} seed={seed} spans "
            f"splits {sorted(grp['split'].unique())} — filter with "
            f"split= before evaluating; never pool the tuning and scoring "
            f"windows."
        )
        wide_pred = (grp.pivot(index="date", columns="ticker",
                               values="y_pred").sort_index()
                     .reindex(columns=universe))
        wide_true = (grp.pivot(index="date", columns="ticker",
                               values="y_true").sort_index()
                     .reindex(columns=universe))
        tradable_wide = None
        if "tradable" in grp.columns:
            tradable_wide = (grp.pivot(index="date", columns="ticker",
                                       values="tradable").sort_index()
                             .reindex(columns=universe))

        per_day, daily_ret, bt_stats = _score_wide_group(
            wide_pred, wide_true, raw_wide, tradable_wide, cfg,
            decile_fraction=frac, universe=universe,
        )
        split_val = str(grp["split"].iloc[0])
        metrics = FoldMetrics(
            model=str(model_name),
            fold=str(fold_name),
            seed=int(seed),
            split=split_val,
            n_days=len(per_day),
            mean_rank_ic=float(np.nanmean(per_day["rank_ic"])),
            std_rank_ic=float(np.nanstd(per_day["rank_ic"], ddof=1)),
            directional_acc=float(np.nanmean(per_day["dir_acc"])),
            huber_loss=float(np.nanmean(per_day["huber"])),
            mse=float(np.nanmean(per_day["mse"])),
            sharpe_after_costs=bt_stats["sharpe"],
            max_drawdown=bt_stats["max_drawdown"],
            turnover=bt_stats["turnover"],
            cum_return=bt_stats["cum_return"],
        )
        results.append(FoldResult(
            model=str(model_name),
            fold=str(fold_name),
            seed=int(seed),
            split=split_val,
            per_day=per_day,
            fold_metrics=metrics,
            backtest_returns=daily_ret,
        ))
    return results


# ─── aggregate backtest — the grid's / compare.py interface ───────────────

def backtest(
    predictions: pd.DataFrame | str | Path,
    cfg: dict,
    split: str | None = None,
    postprocessing: dict | None = None,
    targets: pd.DataFrame | None = None,
) -> dict:
    """Score a predictions frame through the cost-aware backtest and
    aggregate across every (model, fold, seed) group it contains.

    Aggregation is the MEAN across groups (4 folds × 5 seeds = 20 runs
    per arm): mean of per-run mean rank ICs, mean of per-run Sharpes,
    mean of per-run max drawdowns and turnovers. Returns one flat dict;
    `sharpe` is the key the grid's selection_metric reads.

    Refuses to pool more than one model — a pooled aggregate across the
    two arms would be meaningless.
    """
    df = (predictions if isinstance(predictions, pd.DataFrame)
          else load_predictions(source=predictions, split=split))
    if targets is None:
        targets = pd.read_parquet(d_data.resolve_cache() / "targets.parquet")
    results = evaluate_predictions(df, targets, cfg, split=split,
                                   postprocessing=postprocessing)
    if not results:
        raise ValueError("No (model, fold, seed) groups to backtest")

    per_run = []
    for r in results:
        m = asdict(r.fold_metrics)
        m["sharpe"] = m["sharpe_after_costs"]
        per_run.append(m)
    keys = ["mean_rank_ic", "std_rank_ic", "directional_acc", "huber_loss",
            "mse", "sharpe_after_costs", "max_drawdown", "turnover",
            "cum_return"]
    agg = {k: float(np.mean([run[k] for run in per_run])) for k in keys}
    agg["sharpe"] = agg["sharpe_after_costs"]
    agg["n_runs"] = len(per_run)
    agg["n_days"] = int(sum(run["n_days"] for run in per_run))
    agg["per_run"] = per_run
    agg["split"] = split
    return agg

# ─── closed-form baselines (baselines.py; random_model changed by Phase 4.1 F3) ──

def random_model(X: np.ndarray, seed: int, day_index: int) -> np.ndarray:
    """Standard-normal noise, a different draw for every (seed, day)."""
    rng = np.random.default_rng([seed, day_index])
    return rng.standard_normal(X.shape[0])


def persistence_model(X: np.ndarray, feature_idx_logret: int = 0) -> np.ndarray:
    """Predict tomorrow's excess return = today's log-return, demeaned.

    Uses X[:, -1, feature_idx_logret] only — the most recent day's
    log-return feature (the last row of the lookback window, i.e. the
    prediction date itself). The mean subtraction matches the target's
    cross-sectional demeaning.
    """
    today_return = X[:, -1, feature_idx_logret]
    return today_return - np.nanmean(today_return)

# ─── Phase 4 addition: prediction-frame helper (schema of load_predictions) ──

def make_pred_frame(model_name: str, fold: str, seed: int, split: str, dates,
                    tickers, preds: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    n, N = preds.shape
    assert y.shape == preds.shape and len(tickers) == N and len(dates) == n
    return pd.DataFrame({
        "model": model_name, "fold": fold, "seed": int(seed), "split": split,
        "date": np.repeat(pd.to_datetime(np.asarray(dates)).values, N),
        "ticker": np.tile(np.array(list(tickers)), n),
        "y_pred": np.asarray(preds, dtype=np.float64).reshape(-1),
        "y_true": np.asarray(y, dtype=np.float64).reshape(-1),
    })


def val_half_labels(fold, dates) -> np.ndarray:
    """'es' / 'sel' label for each date of a fold's validation period (Phase 4.1 F2.1)."""
    from dmrgt import data as d_data
    f = d_data.fold_by_name(fold) if isinstance(fold, str) else fold
    es_dates, _ = d_data.split_val(f.val_dates)
    return np.where(pd.DatetimeIndex(pd.to_datetime(np.asarray(dates))).isin(es_dates), "es", "sel")


# ─── cost-aware portfolio construction lives in portfolio.py; the harness gains backtest_from_weights ───

def backtest_from_weights(W: pd.DataFrame, returns: pd.DataFrame, commission_bps: float,
                          slippage_bps: float) -> tuple[pd.Series, dict]:
    """Same return alignment (positions set on day t earn the next row's realised return via the
    one-row shift), turnover definition and cost model as backtest_daily_deciles — only the
    weights come from outside. Returns (daily after-cost return series, stats dict)."""
    rets = returns[list(W.columns)].reindex(W.index)
    fee = (commission_bps + slippage_bps) / 10_000.0
    raw_port = (W.shift(1).fillna(0) * rets).sum(axis=1)
    w_change = W.diff().abs().sum(axis=1).fillna(0)
    cost = fee * w_change / 2
    daily_ret = (raw_port - cost).astype(float)
    daily_ret.index = pd.DatetimeIndex(daily_ret.index)
    sd = float(daily_ret.std(ddof=1))
    sharpe = float(daily_ret.mean() / sd * np.sqrt(252)) if sd > 0 else 0.0
    eq = (1 + daily_ret.fillna(0)).cumprod()
    peak = eq.cummax()
    dd = (eq - peak) / peak
    return daily_ret, {
        "sharpe": sharpe,
        "max_drawdown": float(-dd.min()) if dd.notna().any() else float("nan"),
        "turnover": float(w_change.mean() / 2),
        "cum_return": float(eq.iloc[-1] - 1) if len(eq) > 0 else float("nan"),
        "gross_mean_daily": float(raw_port.mean()), "cost_mean_daily": float(cost.mean()),
        "w_change": w_change,
    }


def pooled_stats(ret: pd.Series, turnover: pd.Series) -> dict:
    """Annualised Sharpe etc. of a pooled (date-ordered) daily series."""
    ret = ret.sort_index()
    sd = float(ret.std(ddof=1))
    eq = (1 + ret.fillna(0)).cumprod()
    dd = (eq - eq.cummax()) / eq.cummax()
    return {"sharpe": float(ret.mean() / sd * np.sqrt(252)) if sd > 0 else 0.0,
            "turnover": float(turnover.reindex(ret.index).mean()), "max_drawdown": float(-dd.min()),
            "cum_return": float(eq.iloc[-1] - 1), "n_days": int(len(ret))}


In [ ]:
%%writefile /kaggle/working/dmrgt/stats.py
"""DMRGT Phase 4 — significance testing (spec §6.4).

Ported verbatim from Phase 3 stats.py: Wilcoxon signed-rank on paired
(fold, seed) rank IC, and the NumPy Politis–Romano stationary bootstrap on
the Sharpe difference (seed-averaged per fold, folds pooled, paired
resampling). Phase 4 edits: imports (paths/data → d_data, eval → d_eval),
the local CLI was dropped, and the Holm helper + generic wrappers at the
bottom were added. The "dynamic"/"frozen" key names inside the ported
functions are kept as-is: in Phase 4, "dynamic" = arm/policy A under test
and "frozen" = the reference it is compared to.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon as scipy_wilcoxon

from dmrgt import data as d_data
from dmrgt import evaluate as ev


# ─── inputs assembled FROM prediction frames ───────────────────────────────

def _as_frame(pred) -> pd.DataFrame:
    """Accept a predictions DataFrame or a path to its parquet."""
    if isinstance(pred, pd.DataFrame):
        df = pred
    else:
        df = pd.read_parquet(pred)
    return df


def _test_rows(pred) -> pd.DataFrame:
    """The test-split rows of a predictions file/frame.

    Callers run strictly after winner selection is locked (§1), so this
    is the first and only place test rows are read.
    """
    df = _as_frame(pred)
    assert df["model"].nunique() == 1, (
        "a significance-test input must hold exactly one arm's rows"
    )
    out = df[df["split"] == "test"]
    assert len(out) > 0, "no test rows in the predictions input"
    return out


def per_fold_seed_ic(pred_df: pd.DataFrame, cfg: dict) -> dict:
    """{(fold, seed): mean rank IC} for one arm's TEST rows — the 20
    paired observations the Wilcoxon test consumes.

    Scores through the same evaluation pipeline as everything else
    (identity post-processing, the declared final configuration), so the
    ICs here are exactly the per-run mean_rank_ic values compare.py
    reports.
    """
    targets = pd.read_parquet(d_data.resolve_cache() / "targets.parquet")
    runs = ev.evaluate_predictions(_test_rows(pred_df), targets, cfg,
                                   split=None)
    keyed = {(r.fold, r.seed): float(r.fold_metrics.mean_rank_ic)
             for r in runs}
    assert len(keyed) == len(runs), "duplicate (fold, seed) keys"
    return keyed


def daily_returns(pred_df: pd.DataFrame, cfg: dict) -> pd.Series:
    """Pooled daily after-cost return series for one arm's TEST rows.

    Seed-averaged per (model, fold), then folds concatenated
    chronologically (by each series' first date — disjoint calendar
    years make the stitch points unambiguous).
    """
    targets = pd.read_parquet(d_data.resolve_cache() / "targets.parquet")
    runs = ev.evaluate_predictions(_test_rows(pred_df), targets, cfg,
                                   split=None)
    by_fold: dict[str, list[pd.Series]] = {}
    for r in runs:
        by_fold.setdefault(r.fold, []).append(r.backtest_returns)
    per_fold = []
    for fold, series in by_fold.items():
        df = pd.concat(series, axis=1)
        assert df.index.is_monotonic_increasing, f"{fold}: unordered index"
        per_fold.append(df.mean(axis=1))
    per_fold.sort(key=lambda s: s.index[0])
    return pd.concat(per_fold)


# ─── the two tests (§6) ────────────────────────────────────────────────────

def _sharpe(daily_ret: np.ndarray) -> float:
    """Annualised Sharpe of a daily return series (risk-free 0)."""
    r = np.asarray(daily_ret, dtype=float)
    if r.size < 2:
        return float("nan")
    sd = float(np.std(r, ddof=1))
    return float(np.mean(r) / sd * np.sqrt(252)) if sd > 0 else 0.0


def _sharpe_rows(returns: np.ndarray) -> np.ndarray:
    """Annualised Sharpe of each row of a [reps, n] return matrix."""
    sd = returns.std(axis=1, ddof=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        s = returns.mean(axis=1) / sd * np.sqrt(252)
    return np.where(sd > 0, s, 0.0)


def _stationary_bootstrap_indices(n: int, block: int, reps: int,
                                  seed: int) -> np.ndarray:
    """Politis–Romano stationary bootstrap index draws, shape [reps, n].

    Each replicate starts at a uniform index; every subsequent index
    follows the previous one (t+1 mod n — circular, as in the original
    scheme) with probability 1 − 1/block, else jumps to a fresh uniform
    index. Expected block length is therefore ``block``.
    """
    if not (1 <= block <= n):
        raise ValueError(f"block length {block} out of range for n={n}")
    rng = np.random.default_rng(seed)
    out = np.empty((reps, n), dtype=np.int64)
    cur = rng.integers(0, n, size=reps)
    out[:, 0] = cur
    p_follow = 1.0 - 1.0 / float(block)
    for t in range(1, n):
        follow = rng.random(reps) < p_follow
        cur = np.where(follow, (cur + 1) % n, rng.integers(0, n, size=reps))
        out[:, t] = cur
    return out


def _stationary_bootstrap_diffs(dyn: np.ndarray, fro: np.ndarray,
                                block: int, reps: int, seed: int) -> np.ndarray:
    """Paired stationary bootstrap on the Sharpe difference, [reps].

    One index draw per replicate, applied to BOTH arms — the replicate
    isolates the arms, not the resample. numpy instead of arch's
    StationaryBootstrap because Cell 6 runs in-session with Internet
    OFF and arch is not in Kaggle's base image; same Politis–Romano
    scheme (expected block length ``block``, circular indexing).
    """
    dyn = np.asarray(dyn, dtype=float)
    fro = np.asarray(fro, dtype=float)
    idx = _stationary_bootstrap_indices(len(dyn), block, reps, seed)
    return _sharpe_rows(dyn[idx]) - _sharpe_rows(fro[idx])


def wilcoxon_test(dynamic_ic, frozen_ic, cfg: dict) -> dict:
    """Wilcoxon signed-rank on paired (fold, seed) mean rank ICs.

    ``dynamic_ic`` / ``frozen_ic`` are the per_fold_seed_ic() dicts (or
    equal-length sequences, paired by sort order). The result carries
    ``wins`` — how many paired observations the dynamic arm won — which
    the §6 conclusion string reports.
    """
    if isinstance(dynamic_ic, dict):
        assert isinstance(frozen_ic, dict)
        keys = sorted(set(dynamic_ic) & set(frozen_ic))
        missing = set(dynamic_ic) ^ set(frozen_ic)
        assert not missing, (
            f"unpaired (fold, seed) runs between arms: {sorted(missing)} — "
            f"the Wilcoxon test needs every run in BOTH arms"
        )
        dyn = [dynamic_ic[k] for k in keys]
        fro = [frozen_ic[k] for k in keys]
    else:
        dyn, fro = list(dynamic_ic), list(frozen_ic)
    assert len(dyn) == len(fro), (
        f"unpaired samples: {len(dyn)} dynamic vs {len(fro)} frozen"
    )

    alpha = float(cfg["significance"]["wilcoxon_alpha"])
    stat, p = scipy_wilcoxon(dyn, fro)
    return {
        "test": "Wilcoxon signed-rank, paired on (fold, seed)",
        "n_pairs": len(dyn),
        "wins": int(sum(1 for a, b in zip(dyn, fro) if a > b)),
        "statistic": float(stat),
        "p_value": float(p),
        "alpha": alpha,
        "significant": bool(p < alpha),
        "dynamic_mean_rank_ic": float(np.mean(dyn)),
        "frozen_mean_rank_ic": float(np.mean(fro)),
        "mean_paired_difference": float(np.mean(
            [a - b for a, b in zip(dyn, fro)])),
    }


def bootstrap_sharpe_test(dynamic_returns: pd.Series,
                          frozen_returns: pd.Series,
                          cfg: dict, seed: int = 7) -> dict:
    """Stationary bootstrap on the Sharpe difference between arms.

    Paired resampling: each replicate draws one index sequence and
    applies it to BOTH arms; the replicate statistic is
    Sharpe(dyn_resample) − Sharpe(fro_resample).

    ``bootstrap_p_value`` is the ONE-SIDED p under the pre-declared H1
    (dynamic > frozen) — the number the §6 conclusion string reports.
    """
    sig = cfg["significance"]
    n_iterations = int(sig["bootstrap_iterations"])
    block = int(sig["bootstrap_block_length"])
    alpha = float(sig["wilcoxon_alpha"])

    dyn = np.asarray(dynamic_returns, dtype=float)
    fro = np.asarray(frozen_returns, dtype=float)
    assert len(dyn) == len(fro) > 0, "pooled return series misaligned"

    observed = _sharpe(dyn) - _sharpe(fro)

    # Paired resampling: each replicate draws ONE index sequence and
    # applies it to BOTH arms, so the replicate statistic isolates the
    # arms, not the resample.
    diffs = _stationary_bootstrap_diffs(dyn, fro, block, n_iterations, seed)
    assert diffs.shape == (n_iterations,)

    p_one = float(np.mean(diffs <= 0))          # H1: dynamic > frozen
    p_two = float(min(1.0, 2 * min(p_one, 1 - p_one)))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {
        "test": "Stationary bootstrap (Politis–Romano) on Sharpe difference",
        "n_iterations": n_iterations,
        "block_length": block,
        "construction": (
            "daily after-cost returns, seed-averaged per (model, fold), "
            "pooled across folds chronologically, paired resampling"
        ),
        "observed_sharpe_dynamic": _sharpe(dyn),
        "observed_sharpe_frozen": _sharpe(fro),
        "observed_sharpe_difference": float(observed),
        "ci_95": [float(lo), float(hi)],
        "p_one_sided_dynamic_greater": p_one,
        "p_two_sided": p_two,
        "bootstrap_p_value": p_one,
        "alpha": alpha,
        "significant_one_sided": bool(p_one < alpha),
        "bootstrap_mean_difference": float(np.mean(diffs)),
    }


# ─── the composite the notebook's Cell 6 and compare.py both call ──────────

def run_tests_from_predictions(dynamic_pred, frozen_pred, cfg: dict,
                               winning_graph_config: dict | None = None,
                               ) -> dict:
    """Both tests from the two arms' prediction files/frames, returning
    the dict phase3_significance_results.json holds (§6's exact shape:
    wilcoxon, bootstrap_sharpe, winning_graph_config, conclusion).

    The dynamic input is the WINNING grid variant's prediction file —
    its test rows are the final dmrgt_all3 result (§1: never retrained).
    """
    dyn = _test_rows(dynamic_pred)
    fro = _test_rows(frozen_pred)

    wilcoxon_result = wilcoxon_test(
        dynamic_ic=per_fold_seed_ic(dyn, cfg),
        frozen_ic=per_fold_seed_ic(fro, cfg),
        cfg=cfg,
    )
    bootstrap_result = bootstrap_sharpe_test(
        dynamic_returns=daily_returns(dyn, cfg),
        frozen_returns=daily_returns(fro, cfg),
        cfg=cfg,
    )

    conclusion = (
        f"Dynamic graph rank IC exceeds frozen in {wilcoxon_result['wins']}/"
        f"{wilcoxon_result['n_pairs']} folds (p={wilcoxon_result['p_value']:.3f}); "
        f"Sharpe difference p={bootstrap_result['bootstrap_p_value']:.3f}"
    )
    return {
        "wilcoxon": wilcoxon_result,
        "bootstrap_sharpe": bootstrap_result,
        "winning_graph_config": winning_graph_config,
        "conclusion": conclusion,
    }


# ─── Phase 4 additions ─────────────────────────────────────────────────────

def holm_correction(p_values: dict, alpha: float = 0.05) -> dict:
    """Holm–Bonferroni step-down. Returns {name: {p_raw, p_adj, reject}}."""
    items = sorted(p_values.items(), key=lambda kv: kv[1])
    m = len(items)
    out, running = {}, 0.0
    for rank, (name, p) in enumerate(items):
        adj = min(1.0, (m - rank) * p)
        running = max(running, adj)          # enforce monotonicity
        out[name] = {"p_raw": float(p), "p_adj": float(running), "reject": bool(running < alpha)}
    return out


def compare_arms(test_pred, ref_pred, cfg: dict) -> dict:
    """Wilcoxon + bootstrap for one comparison of two prediction frames
    (test rows). `test_pred` plays the 'dynamic' role, `ref_pred` the
    'frozen' role of the ported functions."""
    a, b = _test_rows(test_pred), _test_rows(ref_pred)
    wil = wilcoxon_test(per_fold_seed_ic(a, cfg), per_fold_seed_ic(b, cfg), cfg)
    boot = bootstrap_sharpe_test(daily_returns(a, cfg), daily_returns(b, cfg), cfg)
    return {"wilcoxon": wil, "bootstrap_sharpe": boot}


In [ ]:
%%writefile /kaggle/working/dmrgt/train.py
"""DMRGT Phase 5 — training (Phase 4 spec §6.5 + Phase 4.1 F1/F2 + Phase 5 arms).

train_one(arm, fold, seed, batch_size, graph, tag): seeded CPU construction + byte
fingerprint (Phase 3 Rule 15 assertion ported), then one of two schedules:

  'adaptive' (arms A, B, C): two-group AdamW (no weight decay on A_log, D, dt_proj.bias,
      biases, norm weights), linear warmup, ReduceLROnPlateau, gradient clipping, early
      stopping — all driven ONLY by a 3-epoch rolling mean of val_es mean rank IC;
      the best checkpoint can only come from epochs >= warmup (Phase 4.1 F2.2).
  'phase3'   (arm P): the Phase 3 schedule — Adam, fixed LR 1e-3, Huber delta 1.0, no
      clipping, early stopping on the validation Huber loss (patience 10) — with the
      loss measured on val_es only (Phase 5 spec §4).

val_sel is logged for the record and is never read by any training logic. Only train and
validation splits are ever requested — no test data is touched here.
"""

from __future__ import annotations

import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from dmrgt import data as d_data
from dmrgt import evaluate as ev
from dmrgt import graph as d_graph
from dmrgt import models as d_models


# ─── seeding + init fingerprint (ported from Phase 3 train.py) ─────────────

def set_seed(seed: int) -> None:
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def param_fingerprint(model: torch.nn.Module) -> str:
    """Byte-level fingerprint of a model's trainable parameters (device-stable)."""
    h = hashlib.sha256()
    with torch.no_grad():
        for p in model.parameters():
            if p.requires_grad:
                h.update(p.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def arm_cfg(arm: str) -> dict:
    return d_data.load_snapshot()["CONFIG"]["arms"][arm]


def build_model(arm: str, snapshot: dict | None = None, grad_checkpoint: bool = False):
    snapshot = snapshot or d_data.load_snapshot()
    a = snapshot["CONFIG"]["arms"][arm]
    return d_models.make_model(a["encoder"], snapshot["CONFIG"]["mamba"], grad_checkpoint=grad_checkpoint)


def ckpt_paths(out_dir: Path, tag: str, fold: str, seed: int) -> dict:
    t = f"{tag}_{fold}_{seed}"
    return {"ckpt": out_dir / f"ckpt_{t}.pt", "valpred": out_dir / f"valpred_{t}.parquet",
            "log": out_dir / f"log_{t}.csv", "done": out_dir / f"done_{t}.json"}


# ─── F2.2: training control sees ONLY val_es metrics ───────────────────────

class ControlMetrics:
    """The only object the stopping rule and LR scheduler receive. Construction asserts that
    every metric handed in is a val_es metric, so val_sel cannot reach training control."""

    def __init__(self, **metrics):
        assert all(k.startswith("val_es") for k in metrics), \
            f"training control may only use val_es metrics, got {sorted(metrics)}"
        self.m = dict(metrics)


class AdaptiveController:
    """Rolling-3 val_es rank IC; best checkpoint only from epochs >= warmup; patience counts
    post-warmup epochs without improvement; LR plateau steps on the same rolling metric."""

    def __init__(self, warmup: int, patience: int):
        self.warmup, self.patience = warmup, patience
        self.hist, self.best, self.best_epoch, self.bad = [], -float("inf"), -1, 0

    def update(self, epoch: int, cm: ControlMetrics) -> dict:
        ic = cm.m["val_es_rank_ic"]
        self.hist.append(ic)
        last = [x for x in self.hist[-3:] if np.isfinite(x)]
        roll = float(np.mean(last)) if last else float("nan")
        improved = stop = False
        if epoch >= self.warmup:                       # warmup epochs are never eligible
            score = roll if np.isfinite(roll) else -float("inf")
            if score > self.best:
                self.best, self.best_epoch, self.bad, improved = score, epoch, 0, True
            else:
                self.bad += 1
            stop = self.bad >= self.patience
        return {"roll3": roll, "improved": improved, "stop": stop}


class Phase3Controller:
    """Phase 3 rule: best = lowest validation Huber loss (strict <), patience on epochs
    without improvement, counted from epoch 0 — evaluated on val_es only."""

    def __init__(self, patience: int):
        self.patience, self.best, self.best_epoch, self.bad = patience, float("inf"), -1, 0

    def update(self, epoch: int, cm: ControlMetrics) -> dict:
        loss = cm.m["val_es_loss"]
        improved = loss < self.best
        if improved:
            self.best, self.best_epoch, self.bad = loss, epoch, 0
        else:
            self.bad += 1
        return {"roll3": float("nan"), "improved": improved, "stop": self.bad >= self.patience}


def phase3_val_loss(pred: np.ndarray, y: np.ndarray, batch_size: int) -> float:
    """Phase 3 train_one_seed: mean over validation batches of the batch-mean Huber loss (delta 1.0)."""
    tot, n = 0.0, 0
    for i in range(0, len(pred), batch_size):
        tot += float(F.huber_loss(torch.from_numpy(pred[i: i + batch_size]).float(),
                                  torch.from_numpy(y[i: i + batch_size]).float(), delta=1.0))
        n += 1
    return tot / max(1, n)


def gross_ls_sharpe(pred: np.ndarray, y: np.ndarray, q: float = 0.20) -> float:
    """Annualised Sharpe of a gross equal-weight top-q minus bottom-q daily strategy. Logged per epoch for the
    record only (val_sel); the cost-aware numbers come from the evaluation harness."""
    n_leg = max(1, int(round(q * pred.shape[1])))
    r = []
    for p, t in zip(pred, y):
        o = np.argsort(-p, kind="stable")
        r.append(t[o[:n_leg]].mean() - t[o[-n_leg:]].mean())
    r = np.asarray(r)
    sd = r.std(ddof=1) if len(r) > 1 else 0.0
    return float(r.mean() / sd * np.sqrt(252)) if sd > 0 else 0.0


def mean_rank_ic(pred: np.ndarray, y: np.ndarray) -> float:
    ics = [ev.per_day_rank_ic(pred[i], y[i]) for i in range(len(pred))]
    return float(np.nanmean(ics)) if np.isfinite(ics).any() else float("nan")


def train_one(arm: str, fold: str, seed: int, batch_size: int, graph: str = "baseline",
              tag: str | None = None, max_epochs: int | None = None, out_dir: Path | None = None,
              shuffle_targets: bool = False, write: bool = True, grad_checkpoint: bool = False) -> dict:
    snap = d_data.load_snapshot()
    C = snap["CONFIG"]
    a = C["arms"][arm]
    ta = C["training_adaptive"]                 # epochs, warmup, plateau, clipping (Phase 3 used 50 epochs too)
    tc = C["training_phase3"] if a["schedule"] == "phase3" else ta   # lr / Huber delta / patience of the arm's schedule
    tag = tag or arm
    out_dir = Path(out_dir) if out_dir is not None else d_data.WORK
    out_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

    t_start = time.time()
    lookback = int(a["lookback"])
    phase3 = a["schedule"] == "phase3"
    cfg = d_data.load_cfg(graph)
    universe = list(cfg["universe"]["tickers"])
    sd_tr = d_graph.get_split_g(fold, "train", lookback, graph)
    sd_va = d_graph.get_split_g(fold, "val", lookback, graph)
    sd_es = d_graph.get_split_g(fold, "val_es", lookback, graph)
    sd_sel = d_graph.get_split_g(fold, "val_sel", lookback, graph)
    bld = d_data.get_builder(fold, lookback, graph_key=graph)
    scaler_mean, scaler_scale = bld.scaler_mean.copy(), bld.scaler_scale.copy()
    edges = d_graph.load_daily_edges(graph)
    y_train = sd_tr.y
    if shuffle_targets:                       # leakage gate: same days, labels scrambled across dates
        y_train = y_train[np.random.default_rng(12345).permutation(len(y_train))]
    n_es = len(sd_es.dates)
    assert len(sd_va.dates) == n_es + len(sd_sel.dates)
    assert (sd_va.dates[:n_es] == sd_es.dates).all() and (sd_va.dates[n_es:] == sd_sel.dates).all()

    # ── Rule 15: random-init assertion, on CPU ────────────────────────
    set_seed(seed)
    net = build_model(arm, snap, grad_checkpoint)   # F1: Mamba dt-init assertion runs inside make_model
    fp1 = param_fingerprint(net)
    set_seed(seed)
    net2 = build_model(arm, snap, grad_checkpoint)
    assert fp1 == param_fingerprint(net2), (
        f"Random-init assertion failed: same seed produced different params (seed={seed}).")
    set_seed(seed + 9999)
    net3 = build_model(arm, snap, grad_checkpoint)
    assert fp1 != param_fingerprint(net3), (
        f"Random-init assertion failed: different seeds produced identical params (seed={seed}).")
    del net2, net3
    dt_mean = d_models.assert_dt_init(net) if a["encoder"] == "mamba" else float("nan")
    net = net.to(device)
    set_seed(seed)                            # restart the RNG at the seeded point
    n_params = int(sum(p.numel() for p in net.parameters()))

    X_tr = torch.from_numpy(sd_tr.X).float().to(device)
    y_tr = torch.from_numpy(y_train).float().to(device)
    X_va = torch.from_numpy(sd_va.X).float().to(device)
    tr_ei = [edges[pd.Timestamp(d)][0].to(device) for d in sd_tr.dates]
    tr_et = [edges[pd.Timestamp(d)][1].to(device) for d in sd_tr.dates]
    va_ei = [edges[pd.Timestamp(d)][0].to(device) for d in sd_va.dates]
    va_et = [edges[pd.Timestamp(d)][1].to(device) for d in sd_va.dates]

    huber_delta = float(tc["huber_delta"])
    epochs = int(max_epochs if max_epochs is not None else ta["max_epochs"])
    patience = int(tc["early_stop_patience"])
    warm = int(ta["warmup_epochs"])
    if phase3:
        opt = torch.optim.Adam(net.parameters(), lr=tc["lr"])         # Phase 3: Adam, fixed LR, no wd
        sched, ctl, clip = None, Phase3Controller(patience), None
    else:
        opt = torch.optim.AdamW(d_models.param_groups(net, ta["weight_decay"]), lr=tc["lr"])
        pl = ta["plateau"]
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode=pl["mode"], factor=pl["factor"], patience=pl["patience"], min_lr=pl["min_lr"])
        ctl, clip = AdaptiveController(warm, patience), ta["grad_clip_norm"]
    lr0, lr1 = ta["warmup_start_lr"], ta["lr"]

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    log, best_state = [], None
    n_train = len(X_tr)

    for epoch in range(epochs):
        t_ep = time.time()
        if not phase3 and epoch < warm:       # linear warmup 1e-4 → 1e-3
            lr = lr0 + (lr1 - lr0) * epoch / max(1, warm - 1)
            for g in opt.param_groups:
                g["lr"] = lr
        lr_now = float(opt.param_groups[0]["lr"])

        net.train()
        perm = torch.randperm(n_train, device=device)
        ep_loss, n_b = 0.0, 0
        for i in range(0, n_train, batch_size):
            idx = perm[i: i + batch_size]
            rows = idx.tolist()
            opt.zero_grad()
            yp = net.forward_with_edge_lists(X_tr[idx], [tr_ei[j] for j in rows], [tr_et[j] for j in rows])
            loss = F.huber_loss(yp, y_tr[idx], delta=huber_delta)
            loss.backward()
            if clip is not None:
                torch.nn.utils.clip_grad_norm_(net.parameters(), clip)
            opt.step()
            ep_loss += float(loss.item())
            n_b += 1
        train_loss = ep_loss / max(1, n_b)

        net.eval()
        val_pred = predict(net, X_va, va_ei, va_et, batch_size)
        es_ic = mean_rank_ic(val_pred[:n_es], sd_es.y)
        es_loss = phase3_val_loss(val_pred[:n_es], sd_es.y, batch_size)
        sel_ic = mean_rank_ic(val_pred[n_es:], sd_sel.y)         # logged for the record ONLY
        sel_sh = gross_ls_sharpe(val_pred[n_es:], sd_sel.y)      # logged for the record ONLY
        cm = ControlMetrics(val_es_rank_ic=es_ic, val_es_loss=es_loss)   # val_sel is not passed on
        res = ctl.update(epoch, cm)
        log.append({"epoch": epoch, "lr": lr_now, "train_loss": train_loss, "val_es_loss": es_loss,
                    "val_es_rank_ic": es_ic, "val_es_rank_ic_roll3": res["roll3"],
                    "val_sel_rank_ic": sel_ic, "val_sel_sharpe": sel_sh, "epoch_seconds": time.time() - t_ep})
        if res["improved"]:
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        if sched is not None and epoch >= warm:                  # plateau scheduler: not during warmup
            r3 = res["roll3"]
            sched.step(r3 if np.isfinite(r3) else -1.0)
        if res["stop"]:
            break

    fell_back = best_state is None
    if best_state is not None:
        net.load_state_dict(best_state)
    net.eval()
    val_pred = predict(net, X_va, va_ei, va_et, batch_size)
    assert val_pred.shape == (len(sd_va.dates), len(universe))
    peak = (torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0
    best_row = log[ctl.best_epoch] if ctl.best_epoch >= 0 else log[-1]
    result = {
        "arm": arm, "tag": tag, "fold": fold, "seed": seed, "graph": graph, "lookback": lookback,
        "schedule": a["schedule"], "batch_size": batch_size, "grad_checkpoint": bool(grad_checkpoint),
        "fingerprint": fp1, "n_params": n_params, "dt_init_mean": dt_mean,
        "epochs_run": len(log), "best_epoch": ctl.best_epoch, "no_eligible_epoch_fallback_to_last": bool(fell_back),
        "best_val_es_rank_ic": float(best_row["val_es_rank_ic"]), "best_val_sel_rank_ic": float(best_row["val_sel_rank_ic"]),
        "n_train_days": int(n_train), "n_val_days": int(len(sd_va.dates)), "n_val_es_days": int(n_es),
        "mean_epoch_seconds": float(np.mean([r["epoch_seconds"] for r in log])),
        "last_epoch_seconds": float(log[-1]["epoch_seconds"]),
        "peak_gpu_gb": float(peak), "total_seconds": time.time() - t_start,
        "shuffled_targets": bool(shuffle_targets),
    }
    if write:
        p = ckpt_paths(out_dir, tag, fold, seed)
        torch.save({"arm": arm, "tag": tag, "fold": fold, "seed": seed, "lookback": lookback, "graph": graph,
                    "encoder": a["encoder"], "model_state_dict": best_state if best_state is not None else net.state_dict(),
                    "scaler_mean": scaler_mean, "scaler_scale": scaler_scale}, p["ckpt"])
        vf = ev.make_pred_frame(tag, fold, seed, "val", sd_va.dates, universe, val_pred, sd_va.y)
        vf["val_half"] = np.where(np.arange(len(vf)) < n_es * len(universe), "es", "sel")
        assert (vf["val_half"].to_numpy() == ev.val_half_labels(fold, vf["date"])).all()
        vf.to_parquet(p["valpred"], index=False)
        pd.DataFrame(log).to_csv(p["log"], index=False)
        p["done"].write_text(json.dumps(result, indent=2))      # marker written last
    result["val_pred"] = val_pred
    return result


def predict(net, X, ei, et, batch_size: int) -> np.ndarray:
    out = []
    with torch.inference_mode():
        for i in range(0, len(X), batch_size):
            n_b = min(batch_size, len(X) - i)
            out.append(net.forward_with_edge_lists(X[i: i + n_b], ei[i: i + n_b], et[i: i + n_b])
                       .cpu().numpy())
    return np.concatenate(out, 0)


def probe_memory(arm: str, fold: str, batch_size: int, graph: str = "baseline", grad_checkpoint: bool = False) -> float:
    """One forward+backward at `batch_size`; returns peak GB (0 on CPU)."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    a = arm_cfg(arm)
    sd = d_graph.get_split_g(fold, "train", int(a["lookback"]), graph)
    edges = d_graph.load_daily_edges(graph)
    set_seed(0)
    net = build_model(arm, grad_checkpoint=grad_checkpoint).to(device)
    n = min(batch_size, len(sd.X))
    X = torch.from_numpy(sd.X[:n]).float().to(device)
    y = torch.from_numpy(sd.y[:n]).float().to(device)
    ei = [edges[pd.Timestamp(d)][0].to(device) for d in sd.dates[:n]]
    et = [edges[pd.Timestamp(d)][1].to(device) for d in sd.dates[:n]]
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    net.train()
    loss = F.huber_loss(net.forward_with_edge_lists(X, ei, et), y)
    loss.backward()
    peak = (torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0
    del net, X, y, loss
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return float(peak)


In [ ]:
%%writefile /kaggle/working/dmrgt/worker.py
"""DMRGT Phase 5 — one worker process per GPU (run 2: ALL GPU work happens here, never in the notebook kernel).

usage: python -m dmrgt.worker --jobs /kaggle/working/jobs_S2_gpu0.json --batch-size 32 [--grad-checkpoint]
Processes, not threads: no shared RNG state, and a process's GPU memory is returned to the system when it exits.
Job kinds: train (default), probe_memory, probe_timing, gate_random, gate_shuffled, unit_gpu, infer_arm, infer_policy,
delta_channels, latency, bench_scale. Non-train jobs write a pickle to WORK/results/<kind>__<key>.pkl (the done marker).
"""
import os

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # before torch initialises CUDA

import argparse
import json
import pickle
import subprocess
import sys
import time
import traceback
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))   # so `from dmrgt import ...` works

import numpy as np
import pandas as pd
import torch

from dmrgt import data as d_data
from dmrgt import evaluate as d_eval
from dmrgt import graph as d_graph
from dmrgt import models as d_models
from dmrgt import train as d_train


def job_kind(job):
    return job.get("kind", "train")


def job_tag(job):
    return job.get("tag") or job["arm"]


def result_path(job):
    return d_data.WORK / "results" / f"{job_kind(job)}__{job['key']}.pkl"


def done_marker(job):
    if job_kind(job) == "train":
        return d_train.ckpt_paths(d_data.WORK, job_tag(job), job["fold"], job["seed"])["done"]
    return result_path(job)


def load_result(job):
    return pickle.loads(result_path(job).read_bytes())


def error_file(job, attempt):
    return d_data.WORK / f"error_{job_kind(job)}_{job.get('key') or job_tag(job)}_{job.get('fold', '')}_{job.get('seed', '')}_attempt{attempt}.json"


def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2))


def _device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _snapshot():
    return d_data.load_snapshot()


def _ic_stats(pred, y):
    ics = np.array([d_eval.per_day_rank_ic(pred[i], y[i]) for i in range(len(pred))])
    ics = ics[np.isfinite(ics)]
    se = float(ics.std(ddof=1) / np.sqrt(len(ics)))
    return float(ics.mean()), se, len(ics), ics


# ─── job implementations ────────────────────────────────────────────────────

def job_probe_memory(job, args):
    dev = _device()
    try:
        peak = d_train.probe_memory(job["arm"], job["fold"], int(job["batch_size"]), job.get("graph", "baseline"),
                                    bool(job.get("grad_checkpoint", False)))
        status = "ok"
    except torch.cuda.OutOfMemoryError:
        peak, status = None, "OOM"
        torch.cuda.empty_cache()
    total = torch.cuda.get_device_properties(0).total_memory / 2**30 if dev.type == "cuda" else None
    return {"batch_size": int(job["batch_size"]), "grad_checkpoint": bool(job.get("grad_checkpoint", False)),
            "peak_gb": peak, "total_gb": total, "status": status}


def job_probe_timing(job, args):
    r = d_train.train_one(job["arm"], job["fold"], 0, int(job["batch_size"]), graph=job.get("graph", "baseline"),
                          max_epochs=2, out_dir=d_data.WORK / "probe", write=False,
                          grad_checkpoint=bool(job.get("grad_checkpoint", False)))
    r.pop("val_pred", None)
    return r


def job_gate_random(job, args):
    """Random-weights Mamba (lookback 30) through the full pipeline on every selection fold's validation data."""
    dev = _device()
    edges = d_graph.load_daily_edges("baseline")
    allics, per_fold = [], {}
    for fn in job["folds"]:
        d_train.set_seed(0)
        net = d_train.build_model("B").to(dev).eval()
        sd = d_graph.get_split_g(fn, "val", _snapshot()["CONFIG"]["arms"]["B"]["lookback"], "baseline")
        X = torch.from_numpy(np.asarray(sd.X)).float().to(dev)
        ei = [edges[pd.Timestamp(d)][0].to(dev) for d in sd.dates]
        et = [edges[pd.Timestamp(d)][1].to(dev) for d in sd.dates]
        pred = d_train.predict(net, X, ei, et, 32)
        m, se, n, ics = _ic_stats(pred, sd.y)
        per_fold[fn] = {"mean_rank_ic": m, "se": se, "n_days": n}
        allics.append(ics)
        del net, X
    return {"per_fold": per_fold, "ics": np.concatenate(allics)}


def job_gate_shuffled(job, args):
    fn = job["fold"]
    res = d_train.train_one("B", fn, 0, int(job["batch_size"]), graph="baseline", out_dir=d_data.WORK / "leakage_gate",
                            shuffle_targets=True, write=False, grad_checkpoint=bool(job.get("grad_checkpoint", False)))
    sd = d_graph.get_split_g(fn, "val", _snapshot()["CONFIG"]["arms"]["B"]["lookback"], "baseline")
    m, se, n, _ = _ic_stats(res["val_pred"], sd.y)
    return {"mean_rank_ic": m, "se": se, "n_days": n, "epochs_run": res["epochs_run"]}


def job_unit_gpu(job, args):
    dev = _device()
    assert dev.type == "cuda" or os.environ.get("DMRGT_DEV_ALLOW_CPU"), "unit_gpu needs a CUDA device"
    return {"device": str(dev), "scan": d_models.selftest_scan(dev), "checkpoint": d_models.selftest_checkpoint(dev)}


def job_infer_arm(job, args):
    mi = d_graph.ModelInference(d_train.ckpt_paths(d_data.WORK, job["tag"], job["fold"], job["seed"])["ckpt"],
                                _device(), _snapshot())
    edges = d_graph.load_daily_edges(job["graph"])
    return {"frame": mi.frame(job["name"], "test", mi.predict_with_graphs("test", edges))}


def job_infer_policy(job, args):
    """Phase 4B for one (fold, seed) of the winner: mode 'val' calibrates + runs every policy to the end of validation
    (selection); mode 'test' runs selected / daily / frozen_at_start to the end of the test period (scoring)."""
    snap = _snapshot()
    fn, seed, graph = job["fold"], int(job["seed"]), job["graph"]
    ctx = d_graph.get_ctx(graph, job["max_stale"])
    fold = ctx.folds[fn]
    mi = d_graph.ModelInference(d_train.ckpt_paths(d_data.WORK, job["tag"], fn, seed)["ckpt"], _device(), snap)
    mamba = mi.encoder_type == "mamba"
    lookback = mi.lookback
    if job["mode"] == "val":
        pols = job["policies"]
        sel_set = set(pd.Timestamp(d) for d in d_graph.get_split_g(fn, "val_sel", lookback, graph).dates)
        dval = mi.delta_by_day(["val"]) if mamba else None
        thr = {p["name"]: ctx.calibrate(mi, p) for p in pols if p["kind"] == "trigger"}
        frames, rates = {}, {}
        for p in pols:
            g, rb, _ = d_graph.run_named_policy(p, thr.get(p["name"]), fold, ctx.rets, fold.val_dates[-1], dval,
                                                ctx.cfg, ctx.W, ctx.max_stale)
            frames[p["name"]] = mi.frame(p["name"], "val", mi.predict_with_graphs("val", g))
            rates[p["name"]] = [sum(1 for d in rb if d in sel_set), len(sel_set)]
        return {"thresholds": thr, "frames": frames, "rates": rates}
    # test scoring
    test_set = set(pd.Timestamp(d) for d in d_graph.get_split_g(fn, "test", lookback, graph).dates)
    th = job["thresholds"] if not job.get("calibrate") else ctx.calibrate(mi, job["selected_policy"])
    dby = mi.delta_by_day(["val", "test"]) if mamba else None
    frames, stats = {}, []
    n_days = len([d for d in d_graph.policy_days(ctx.rets, fold, fold.test_dates[-1]) if d in test_set])
    for name, p in job["policies"].items():
        g, rb, st = d_graph.run_named_policy(p, th if p["kind"] == "trigger" else None, fold, ctx.rets,
                                             fold.test_dates[-1], dby, ctx.cfg, ctx.W, ctx.max_stale)
        frames[name] = (mi.frame(name, "val", mi.predict_with_graphs("val", g)),
                        mi.frame(name, "test", mi.predict_with_graphs("test", g)))
        stats.append({"policy": name, "fold": fn, "seed": seed, "test_rebuilds": sum(1 for d in rb if d in test_set),
                      "test_days": n_days, "graph_build_seconds": st["graph_build_seconds"],
                      "policy_seconds": st["policy_seconds"]})
    return {"threshold": th, "frames": frames, "stats": stats}


def job_delta_channels(job, args):
    mi = d_graph.ModelInference(d_train.ckpt_paths(d_data.WORK, job["tag"], job["fold"], job["seed"])["ckpt"],
                                _device(), _snapshot())
    return {"channels": mi.delta_channels("test"), "a_min": mi.model.encoder.mamba.a_min(),
            "dates": [pd.Timestamp(d) for d in mi.split_data("test").dates]}


def job_latency(job, args):
    return {"latency_ms": d_graph.latency_ms(job["arm"], job["graph"], _device())}


def job_bench_scale(job, args):
    cfg_w = d_data.load_cfg(job["graph"])
    return d_graph.bench_scale(int(job["N"]), int(job["window"]), int(job["T"]), float(job["rate_sel"]), job["enc"],
                               int(job["lookback"]), job["mamba_cfg"], cfg_w, _device())


JOBS = {"probe_memory": job_probe_memory, "probe_timing": job_probe_timing, "gate_random": job_gate_random,
        "gate_shuffled": job_gate_shuffled, "unit_gpu": job_unit_gpu, "infer_arm": job_infer_arm,
        "infer_policy": job_infer_policy, "delta_channels": job_delta_channels, "latency": job_latency,
        "bench_scale": job_bench_scale}


def run_job(job, args):
    kind = job_kind(job)
    if kind == "train":
        tr = {k: v for k, v in job.items() if k in ("arm", "fold", "seed", "graph", "tag")}
        return d_train.train_one(**tr, batch_size=args.batch_size, grad_checkpoint=args.grad_checkpoint)
    out = JOBS[kind](job, args)
    p = result_path(job)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_suffix(".tmp")
    tmp.write_bytes(pickle.dumps(out))
    os.replace(tmp, p)                                   # marker appears only when the result is complete
    return out


def _nvidia_smi():
    try:
        return subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30).stdout
    except Exception as e:                               # noqa: BLE001
        return f"nvidia-smi unavailable: {e}"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--jobs", required=True)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--grad-checkpoint", action="store_true")
    args = ap.parse_args()
    jobs = json.loads(Path(args.jobs).read_text())
    for job in jobs:
        if done_marker(job).exists():
            continue                                     # safe if a stage cell is ever re-entered
        for attempt in (1, 2):                           # retry once on failure
            try:
                t0 = time.time()
                r = run_job(job, args)
                msg = (f"epochs={r['epochs_run']} ic={r['best_val_es_rank_ic']:+.4f} " if job_kind(job) == "train" else "")
                print(f"[worker] done {job_kind(job)} {job.get('key') or job} {msg}{time.time() - t0:.0f}s", flush=True)
                break
            except Exception:
                tb = traceback.format_exc()
                info = {"job": job, "traceback": tb}
                if "out of memory" in tb.lower() and torch.cuda.is_available():
                    info["nvidia_smi"] = _nvidia_smi()
                    torch.cuda.empty_cache()
                    time.sleep(30)                       # let the allocator and driver settle before attempt 2
                print(f"[worker] FAILED {job} attempt {attempt}\n{tb}", flush=True)
                write_json(error_file(job, attempt), info)


if __name__ == "__main__":
    main()


In [ ]:
# ─── Cell 10 prelude: import the modules written above ───
import sys
sys.path.insert(0, str(WORK))
os.environ["DMRGT_WORK"] = str(WORK)
from dmrgt import data as d_data, models as d_models, graph as d_graph, evaluate as d_eval
from dmrgt import portfolio as d_portfolio, stats as d_stats, train as d_train, worker as d_worker
import numpy as np, pandas as pd
cfg = d_data.load_cfg("baseline")
UNIVERSE = list(cfg["universe"]["tickers"])
FOLD_NAMES = [f["name"] for f in CONFIG["folds_selection"]]                # folds 1-4
HOLDOUT_NAMES = [f["name"] for f in CONFIG["folds_holdout"]]               # folds 5-6
print(f"modules imported; N = {len(UNIVERSE)}; test split locked: {not d_data.test_unlocked()}")
assert d_models.SCAN_CHUNK == CONFIG["selective_scan"]["chunk"] and d_models.SCAN_MAX_LOG == CONFIG["selective_scan"]["fallback_max_log"], \
    "scan constants in models.py differ from the pre-registered CONFIG"
kernel_guard('Cell 12 — import the modules')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 11c: the shared job launcher — one worker process per GPU (the only place GPU work is started) ───
import glob, subprocess

def lpt_assign(jobs, n_gpus):
    """Longest-job-first onto the GPU with the least assigned time. jobs: [(dur, job)]."""
    loads, assign = [0.0] * n_gpus, [[] for _ in range(n_gpus)]
    for dur, job in sorted(jobs, key=lambda x: -x[0]):
        k = int(np.argmin(loads))
        loads[k] += dur
        assign[k].append(job)
    return assign, loads

def _tag(job):
    return job.get("tag") or job["arm"]

def launch_stage(name, jobs_with_dur, required=True, batch_size=None, grad_checkpoint=None):
    """Run jobs in worker subprocesses (skip-if-done markers, retry once, completeness check). GPUs must be free first."""
    bs = int(batch_size if batch_size is not None else globals().get("BATCH_SIZE") or CONFIG["memory_ladder"][0]["batch_size"])
    gc = bool(grad_checkpoint if grad_checkpoint is not None else globals().get("GRAD_CKPT", False))
    jobs = [j for _, j in jobs_with_dur]
    pending = [(d, j) for d, j in jobs_with_dur if not d_worker.done_marker(j).exists()]
    assign, loads = lpt_assign(pending, n_gpus)
    (WORK / "logs").mkdir(exist_ok=True)
    print(f"[{name}] {len(jobs)} jobs ({len(pending)} pending) on {n_gpus} GPU(s); projected makespan "
          f"{max(loads) / 3600 if pending else 0:.2f} h; batch size {bs}, grad checkpoint {gc}", flush=True)
    if not pending:
        return 0.0
    assert_gpus_free()
    procs = []
    for k in range(n_gpus):
        if not assign[k]:
            continue
        (WORK / f"jobs_{name}_gpu{k}.json").write_text(json.dumps(assign[k], indent=2))
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k), CUBLAS_WORKSPACE_CONFIG=":4096:8",
                   PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", DMRGT_WORK=str(WORK), DMRGT_CACHE=str(CACHE))
        log = open(WORK / "logs" / f"{name}_gpu{k}.log", "w")
        cmd = [sys.executable, "-m", "dmrgt.worker", "--jobs", str(WORK / f"jobs_{name}_gpu{k}.json"), "--batch-size", str(bs)]
        if gc:
            cmd.append("--grad-checkpoint")
        procs.append(subprocess.Popen(cmd, env=env, cwd=str(WORK), stdout=log, stderr=subprocess.STDOUT))
    t_launch = time.time()
    n_done = lambda: sum(d_worker.done_marker(j).exists() for j in jobs)
    last_print = 0.0
    while any(p.poll() is None for p in procs):
        time.sleep(5)
        if time.time() - last_print >= 60:
            last_print = time.time()
            d, el = n_done(), time.time() - t_launch
            base = len(jobs) - len(pending)
            eta = (el / max(1, d - base) * (len(jobs) - d)) if d > base else float("nan")
            print(f"[{name} {el / 60:6.1f} min] {d}/{len(jobs)} jobs done; projected remaining {eta / 60:.0f} min", flush=True)
    missing = [j for j in jobs if not d_worker.done_marker(j).exists()]
    if missing and required:
        for p_ in sorted(glob.glob(str(WORK / "error_*.json"))):
            print(p_, "\n", json.loads(Path(p_).read_text())["traceback"][-3000:])
        raise RuntimeError(f"[{name}] {len(missing)} of {len(jobs)} jobs did not complete: {missing[:3]} ...")
    print(f"[{name}] complete: {len(jobs) - len(missing)}/{len(jobs)} ({(time.time() - t_launch) / 3600:.2f} h)", flush=True)
    return time.time() - t_launch
kernel_guard('Cell 13 — worker launcher (the only place GPU work starts)')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 11b: fold + edge caches, built once on the CPU (vectorised) and reused by every process ───
_t0 = time.time()
_variants = GRID_VARIANTS
_edges_by_variant = {v: d_graph.load_daily_edges(v) for v in _variants}               # builds and persists each variant's edges
print(f"edge caches ready for {len(_variants)} graph variants ({time.time() - _t0:.0f}s)")
_baseline_w = CONFIG["graph_grid"]["variants"]["baseline"]["corr_window"]
_all_folds = d_data.tables()["folds"]
_windows = {}
for _v, _p in _variants.items():
    _windows.setdefault(_p["corr_window"], set(_edges_by_variant[_v].keys()))

def prebuild_for_window(window, lookbacks):
    n = d_data.prebuild_fold_cache(_all_folds, lookbacks, [_windows[window]])
    return n

_built = 0
for _w in sorted(_windows):
    # arm A is the only arm trained on a non-baseline window in S1 (lookback 30); the baseline window also needs lookback 60
    _built += prebuild_for_window(_w, [30, 60] if _w == _baseline_w else [30])
FOLD_CACHE_SECONDS = time.time() - _t0
_n_dirs = len([q for q in d_data.FOLD_CACHE.iterdir() if q.is_dir() and not q.name.startswith(".")])
print(f"fold cache: {_built} new builds, {_n_dirs} cached builders on disk ({FOLD_CACHE_SECONDS:.0f}s)")
kernel_guard('Cell 14 — fold + edge caches (CPU, built once)')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 10: unit tests (hard stop on any failure) ───
import hashlib, tempfile
import torch

_t0 = time.time()
UT = {}
_mc = CONFIG["mamba"]
_MB = dict(d_model=_mc["d_model"], n_layers=_mc["n_layers"], d_state=_mc["d_state"], d_conv=_mc["d_conv"],
           expand=_mc["expand"], dropout=_mc["dropout"])
TARGETS = pd.read_parquet(CACHE / "targets.parquet")

# 1. Mamba causality — future days must not leak into past states
torch.manual_seed(0)
_enc = d_models.MambaEncoder(8, **{k: _MB[k] for k in ("d_model", "n_layers", "d_state", "d_conv", "expand", "dropout")}).eval()
_enc.return_sequence = True
for _L in (30, 60):
    _x = torch.randn(6, _L, 8)
    with torch.no_grad():
        _y1 = _enc(_x)
        for _t in (0, 3, _L // 2, _L - 2):
            _x2 = _x.clone()
            _x2[:, _t + 1:] = torch.randn_like(_x2[:, _t + 1:])
            _y2 = _enc(_x2)
            assert torch.allclose(_y1[:, :_t + 1], _y2[:, :_t + 1], atol=1e-6), \
                f"Mamba CAUSALITY VIOLATED (L={_L}, t={_t}): future days leak into past states"
UT["mamba_causality"] = "pass"

# 2. Mamba shape, determinism, stability
_enc.return_sequence = False
_enc.train()
for _L in (30, 60):
    _out = _enc(torch.randn(40, _L, 8))
    assert tuple(_out.shape) == (40, _mc["d_model"]), _out.shape
def _mk(seed):
    torch.manual_seed(seed)
    return d_models.MambaEncoder(8, **_MB)
def _hash(m):
    h = hashlib.sha256()
    for p in m.parameters():
        h.update(p.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
assert _hash(_mk(7)) == _hash(_mk(7)) and _hash(_mk(7)) != _hash(_mk(8))
_enc2 = _mk(1)
torch.manual_seed(1)
for _ in range(200):
    _o = _enc2(torch.randn(40, 30, 8))
    _o.pow(2).mean().backward()
    assert torch.isfinite(_o).all(), "NaN/Inf in Mamba output"
    assert all(torch.isfinite(p.grad).all() for p in _enc2.parameters() if p.grad is not None), "NaN/Inf grad"
    _enc2.zero_grad()
UT["mamba_shape_determinism_stability"] = "pass"

# 2b. F1: Δ at initialisation is in the intended range and spans an order of magnitude
_enc3 = _mk(3).eval()
_dl = []
with torch.no_grad():
    for _ in range(50):
        _enc3(torch.randn(40, 30, 8))
        _dl.append(_enc3.layers[-1].last_delta.flatten())
_dl = torch.cat(_dl).numpy()
_p5, _p95 = np.percentile(_dl, [5, 95])
assert 0.0005 <= _dl.mean() <= 0.2, f"initial mean Δ {_dl.mean():.4f} outside [0.0005, 0.2]"
assert _p95 / _p5 >= 10, f"initial Δ 5th-95th percentile ratio {_p95 / _p5:.1f} < one order of magnitude"
_m = d_models.make_model("mamba", CONFIG["mamba"])
d_models.assert_dt_init(_m)
_tagged = [n for n, p in _m.named_parameters() if getattr(p, "_no_weight_decay", False)]
assert any("A_log" in n for n in _tagged) and any("dt_proj.bias" in n for n in _tagged) and any(n.endswith(".D") for n in _tagged)
_g = d_models.param_groups(_m, 1e-4)
assert _g[0]["weight_decay"] == 1e-4 and _g[1]["weight_decay"] == 0.0
assert sum(p.numel() for p in _g[0]["params"]) + sum(p.numel() for p in _g[1]["params"]) == sum(p.numel() for p in _m.parameters())
UT["f1_dt_init"] = {"mean_delta": float(_dl.mean()), "p5": float(_p5), "p95": float(_p95), "ratio": float(_p95 / _p5)}

# 3. Incremental correlation == full recomputation on every date
_rets = d_graph.universe_returns(cfg)
_BASE = CONFIG["graph_grid"]["variants"]["baseline"]
_W = _BASE["corr_window"]
_rc = d_graph.RollingCorr(_W, _rets.shape[1], CONFIG["phase4b"]["full_recompute_every"])
_maxdiff, _n = 0.0, 0
for _i, _r in enumerate(_rets.to_numpy(dtype=np.float64)):
    _rc.push(_r)
    if _i >= _W - 1:
        _full = _rets.iloc[_i - _W + 1: _i + 1].corr().values
        _maxdiff = max(_maxdiff, float(np.abs(_rc.corr() - _full).max()))
        _n += 1
assert _maxdiff < 1e-9, f"RollingCorr deviates from full recomputation: max |diff| {_maxdiff:.3e}"
UT["rolling_corr_max_abs_diff"] = _maxdiff
UT["rolling_corr_dates_checked"] = _n

# 4 + 6. Daily policy and the in-notebook baseline variant both reproduce the cache's graphs_baseline.parquet
_cached = d_graph.load_cached_baseline_edges()
_build = d_graph.make_graph_builder(cfg)
_rows = d_graph.return_rows(_rets)
_kpos, _kneg = _BASE["k_pos"], _BASE["k_neg"]
def _edge_set(g):
    return set(zip(g[1].tolist(), g[0][0].tolist(), g[0][1].tolist()))
def _tie_check(_d, ours, theirs):
    _i = _rets.index.get_loc(_d)
    _C = _rets.iloc[_i - _W + 1: _i + 1].corr().values
    for (_t, _s, _dd) in (ours ^ theirs):
        assert _t != 2, f"sector edge mismatch on {_d.date()}"
        _row = np.delete(_C[_s], _s)
        _v = np.sort(_row)[::-1] if _t == 0 else np.sort(_row)
        _k = _kpos if _t == 0 else _kneg
        assert abs(_v[_k - 1] - _v[_k]) < 1e-8, f"non-tie edge mismatch on {_d.date()} (type {_t}, stock {_s})"
_days = [pd.Timestamp(d) for d in _rets.index[_W:]]
_rc4 = d_graph.RollingCorr(_W, _rets.shape[1], CONFIG["phase4b"]["full_recompute_every"])
for _r in _rets.iloc[:_W].to_numpy(dtype=np.float64):
    _rc4.push(_r)
_graphs, _ = d_graph.run_policy(_days, lambda d: _rows[d], _build, lambda d: 0.0,
                                 d_graph.rule_daily, float("inf"), _rc4)
_first = pd.Timestamp(_rets.index[_W - 1])
_checked = [(_first, _build(d_graph.warm_rolling_corr(_rets, _first, _W).corr()))] + [(d, _graphs[d]) for d in _days]
_bad = [d for d, g in _checked if _edge_set(g) != _edge_set(_cached[d])]
assert len(_bad) / len(_checked) <= 0.005, f"{len(_bad)}/{len(_checked)} dates differ from graphs_baseline.parquet"
for _d in _bad:
    _tie_check(_d, _edge_set(dict(_checked)[_d]), _edge_set(_cached[_d]))
UT["daily_policy_vs_cache"] = {"dates_checked": len(_checked), "mismatching_dates": len(_bad),
                               "match_fraction": 1 - len(_bad) / len(_checked)}
_var = d_graph.load_daily_edges("baseline")
assert set(_var) == set(_cached), "baseline variant covers different dates than the cache"
_bad6 = [d for d in _cached if _edge_set(_var[d]) != _edge_set(_cached[d])]
assert len(_bad6) / len(_cached) <= 0.005, f"baseline variant differs from cache on {len(_bad6)}/{len(_cached)} dates"
for _d in _bad6:
    _tie_check(_d, _edge_set(_var[_d]), _edge_set(_cached[_d]))
UT["baseline_variant_vs_cache"] = {"dates": len(_cached), "mismatching_dates": len(_bad6),
                                   "match_fraction": 1 - len(_bad6) / len(_cached)}

# 5. Data alignment + test gate (all six folds, both lookbacks)
_tb = d_data.tables()
_prices = pd.read_parquet(CACHE / "prices.parquet")
_close = _prices.pivot(index="date", columns="ticker", values="close").sort_index()[UNIVERSE]
_nxt = _close.shift(-1) / _close - 1
_ex = _nxt.sub(_nxt.mean(axis=1), axis=0)
_folds_by_name = {f.name: f for f in _tb["folds"]}
for _fold in FOLD_NAMES + HOLDOUT_NAMES:
    for _lb in (30, 60):
        _sd = d_graph.get_split_g(_fold, "train", _lb, "baseline")
        _bld = d_data.get_builder(_fold, _lb, graph_key="baseline")
        assert _bld.scaler_fit_dates[1] <= _folds_by_name[_fold].train_dates[-1], "scaler fit saw dates beyond the training window"
        for _j in np.linspace(0, len(_sd.dates) - 1, 5).astype(int):
            _d = pd.Timestamp(_sd.dates[_j])
            _target_date = _close.index[_close.index.get_loc(_d) + 1]
            assert _d < _target_date, "feature window not strictly before the target date"
            assert np.allclose(_sd.y[_j], _ex.loc[_d].values, atol=1e-12), "target is not the next-day excess return"
            assert _sd.X.shape[1:] == (len(UNIVERSE), _lb, 8)
try:
    d_data.get_split("fold_1_2019_calm", "test")
    raise AssertionError("test split was readable while locked")
except PermissionError:
    pass
UT["data_alignment_and_test_gate"] = "pass"

# 7. Lock enforcement: test rows need ALL FOUR lock files
_real_locks = d_data.LOCKS
with tempfile.TemporaryDirectory() as _td:
    _tl = [Path(_td) / n for n in ("a.json", "b.json", "c.json", "d.json")]
    d_data.LOCKS = _tl
    try:
        for _k in range(4):
            try:
                d_data.assert_test_unlocked()
                raise AssertionError(f"test readable with only {_k} of 4 locks")
            except PermissionError:
                pass
            _tl[_k].write_text("{}")
        d_data.assert_test_unlocked()                    # all four present → allowed
    finally:
        d_data.LOCKS = _real_locks
assert len(_real_locks) == 4 and not d_data.test_unlocked()
UT["four_lock_enforcement"] = "pass"

# F2: val_es / val_sel are disjoint, chronological, and cover the validation period; control cannot see val_sel
for _f in _tb["folds"]:
    _es, _sel = d_data.split_val(_f.val_dates)
    assert len(_es) + len(_sel) == len(_f.val_dates) and _es.max() < _sel.min() and len(set(_es) & set(_sel)) == 0
    _sdv = d_graph.get_split_g(_f.name, "val", 30, "baseline")
    _sde = d_graph.get_split_g(_f.name, "val_es", 30, "baseline")
    _sds = d_graph.get_split_g(_f.name, "val_sel", 30, "baseline")
    assert len(_sde.dates) + len(_sds.dates) == len(_sdv.dates) and _sde.dates.max() < _sds.dates.min()
    assert (d_eval.val_half_labels(_f, _sdv.dates) == np.r_[["es"] * len(_sde.dates), ["sel"] * len(_sds.dates)]).all()
try:
    d_train.ControlMetrics(val_es_rank_ic=0.0, val_sel_rank_ic=0.1)
    raise AssertionError("training control accepted a val_sel metric")
except AssertionError as _e:
    assert "val_es" in str(_e)
_ac = d_train.AdaptiveController(warmup=3, patience=2)
for _e_, _ic in enumerate([0.9, 0.9, 0.9, 0.01, 0.0, 0.0, 0.0]):     # huge IC during warmup must not become "best"
    _r = _ac.update(_e_, d_train.ControlMetrics(val_es_rank_ic=_ic, val_es_loss=0.0))
assert _ac.best_epoch >= 3, "a warmup epoch became the best checkpoint"
UT["val_es_val_sel_separation"] = "pass"

# F3: random baseline draws fresh noise for every (seed, day)
_X0 = np.zeros((len(UNIVERSE), 30, 8))
_a, _b, _c = (d_eval.random_model(_X0, 0, 1), d_eval.random_model(_X0, 0, 2), d_eval.random_model(_X0, 0, 1))
assert not np.allclose(_a, _b) and np.array_equal(_a, _c) and not np.allclose(_a, d_eval.random_model(_X0, 1, 1))
UT["f3_random_baseline"] = "pass"

# Phase 5 §10.1: the weights path reproduces backtest() (single seed, half-life 0, q_out == q_in)
_fold = _tb["folds"][0]
_sdv = d_graph.get_split_g(_fold.name, "val", 30, "baseline")
_rng = np.random.default_rng(5)
_pred = _rng.standard_normal(_sdv.y.shape)
_pf = d_eval.make_pred_frame("x", _fold.name, 0, "val", _sdv.dates, UNIVERSE, _pred, _sdv.y)
_bt = d_eval.backtest(_pf, cfg, split="val", targets=TARGETS)
_raw = d_eval.raw_returns_from_targets(TARGETS, UNIVERSE)
_Sw = pd.DataFrame(_pred, index=pd.DatetimeIndex(_sdv.dates), columns=UNIVERSE)
_Ww = d_portfolio.buffered_weights(_Sw, CONFIG["postproc"]["q_in"], CONFIG["postproc"]["q_in"])
_ret, _st = d_eval.backtest_from_weights(_Ww, _raw, CONFIG["backtest"]["commission_bps"], CONFIG["backtest"]["slippage_bps"])
for _k, _kk in (("sharpe", "sharpe"), ("turnover", "turnover"), ("max_drawdown", "max_drawdown"), ("cum_return", "cum_return")):
    assert abs(_st[_kk] - _bt[_k]) < 1e-10, f"weights path differs from backtest() on {_k}: {_st[_kk]} vs {_bt[_k]}"
_r2, _t2 = d_portfolio.strategy_series(_pf, _raw, UNIVERSE, CONFIG["backtest"], 0, 0.2, 0.2)   # via the ensemble (z-score) path
assert abs(d_eval.pooled_stats(_r2, _t2)["sharpe"] - _bt["sharpe"]) < 1e-10
UT["harness_regression_new_path"] = "pass"

# Phase 5 §10.2: 3-day, 6-stock hand-computed example
_tk = list("ABCDEF")
_sc = pd.DataFrame([[6, 5, 4, 3, 2, 1], [1, 2, 3, 4, 5, 6], [3, 6, 1, 2, 5, 4]], columns=_tk,
                   index=pd.to_datetime(["2020-01-01", "2020-01-02", "2020-01-03"]), dtype=float)
_rt = pd.DataFrame([[0, 0, 0, 0, 0, 0], [0.02, 0.01, 0, -0.01, 0, -0.02], [-0.01, 0.03, 0.02, 0, 0, 0.01]],
                   columns=_tk, index=_sc.index, dtype=float)
# hand-derived: weights d0 long A short F; d1 long F short A; d2 long B short C (k = round(0.2*6) = 1)
# earned: d0 0; d1 = W0.r1 = 0.02-(-0.02) = 0.04; d2 = W1.r2 = 0.01-(-0.01) = 0.02
# turnover |dW| (sum): d0 0 (first row), d1 4, d2 4; fee 15 bps -> cost = 0.0015*4/2 = 0.003 on d1 and d2
# after-cost: 0, 0.037, 0.017 -> mean 0.018, sample sd sqrt(0.000343); one-way turnover mean = (0+4+4)/3/2
_exp = {"sharpe": 0.018 / np.sqrt(0.000343) * np.sqrt(252), "turnover": 8 / 3 / 2, "cum_return": 1.037 * 1.017 - 1,
        "max_drawdown": 0.0}
_r_h, _s_h = d_eval.backtest_daily_deciles(_sc, _rt, _tk, 10, 5, 0.2)
_r_w, _s_w = d_eval.backtest_from_weights(d_portfolio.buffered_weights(_sc, 0.2, 0.2), _rt, 10, 5)
for _name, _s in (("ported harness", _s_h), ("weights path", _s_w)):
    for _k, _v in _exp.items():
        assert abs(_s[_k if _k != "max_drawdown" else "max_drawdown"] - _v) < 1e-9, f"hand check failed ({_name}, {_k}): {_s[_k]} vs {_v}"
assert np.allclose(_r_h.values, [0, 0.037, 0.017]) and np.allclose(_r_w.values, [0, 0.037, 0.017])
UT["harness_hand_check"] = "pass"

# Phase 5 §10.3-5: EMA causality, buffer logic, ensemble
_S = pd.DataFrame(np.random.default_rng(1).standard_normal((40, 20)), index=pd.date_range("2020-01-01", periods=40, freq="B"))
for _hl in (2, 5):
    _a = d_portfolio.ema_smooth(_S, _hl)
    _S2 = _S.copy(); _S2.iloc[21:] = np.random.default_rng(2).standard_normal(_S2.iloc[21:].shape)
    _b = d_portfolio.ema_smooth(_S2, _hl)
    assert np.array_equal(_a.iloc[:21].to_numpy(), _b.iloc[:21].to_numpy()), "EMA is not causal"
assert d_portfolio.ema_smooth(_S, 0) is _S
_n = 20; _nl = int(round(0.2 * _n))
_Sx = pd.DataFrame(np.random.default_rng(3).standard_normal((300, _n)))
_W0 = d_portfolio.buffered_weights(_Sx, 0.2, 0.2)
_plain = np.zeros(_Sx.shape)
for _t_, _row in enumerate(_Sx.to_numpy()):
    _o = np.argsort(-_row, kind="stable"); _plain[_t_, _o[:_nl]] = 1 / _nl; _plain[_t_, _o[-_nl:]] = -1 / _nl
assert np.array_equal(_W0.to_numpy(), _plain), "q_out == q_in must equal plain top/bottom selection"
_Wb = d_portfolio.buffered_weights(_Sx, 0.2, 0.4)
assert (_Wb.abs().sum(axis=1) > 0).all() and ((_Wb > 0).sum(axis=1) == _nl).all() and ((_Wb < 0).sum(axis=1) == _nl).all()
assert not ((_Wb > 0) & (_Wb < 0)).any().any()
assert _Wb.diff().abs().sum(axis=1).mean() <= _W0.diff().abs().sum(axis=1).mean(), "buffer increased turnover"
_pz = pd.DataFrame({"date": np.repeat(pd.date_range("2020-01-01", periods=5), 8), "ticker": list("ABCDEFGH") * 5,
                    "seed": 0, "pred": np.random.default_rng(4).standard_normal(40)})
_one = d_portfolio.ensemble_scores(_pz)
_three = d_portfolio.ensemble_scores(pd.concat([_pz.assign(seed=s) for s in (0, 1, 2)]))
assert np.allclose(_one.to_numpy(), _three.to_numpy()), "ensemble of identical seeds != single-seed z-score"
UT["portfolio_ema_buffer_ensemble"] = "pass"

# ── run-2 additions (spec §3) ──
# R1. fold cache byte-identical to a fresh build with the ported per-day loop builder (fold 1, both lookbacks)
_gd = set(d_graph.load_daily_edges("baseline").keys())
_f1 = d_data.fold_by_name(FOLD_NAMES[0])
UT["fold_cache_identity"] = {}
for _lb in (30, 60):
    _c = d_data.FoldBuilder(_f1, _lb, _gd)
    assert _c.cache_hit, f"fold cache missing for {FOLD_NAMES[0]} lookback {_lb}"
    _r = d_data.FoldBuilder(_f1, _lb, _gd, use_cache=False, builder="loop")
    for _sp in ("train", "val"):
        _a, _b = _c._splits[_sp], _r._splits[_sp]
        assert np.asarray(_a.X).dtype == _b.X.dtype and np.array_equal(np.asarray(_a.X), _b.X), f"cached X differs ({_sp}, L{_lb})"
        assert np.array_equal(_a.y, _b.y) and list(_a.dates) == list(_b.dates), f"cached y/dates differ ({_sp}, L{_lb})"
    assert np.array_equal(_c.scaler_mean, _r.scaler_mean) and np.array_equal(_c.scaler_scale, _r.scaler_scale)
    UT["fold_cache_identity"][f"L{_lb}"] = {"train_days": int(len(_c._splits["train"].dates)), "val_days": int(len(_c._splits["val"].dates))}
    del _c, _r
# R2. chunked scan == sequential scan (forward + gradients; the x20 case exercises the fallback)
UT["scan_equivalence_cpu"] = d_models.selftest_scan(torch.device("cpu"))
# R3. gradient checkpointing leaves loss and gradients unchanged
UT["grad_checkpoint_identity_cpu"] = d_models.selftest_checkpoint(torch.device("cpu"))
# R4 / R5. the kernel has not touched CUDA; GPUs are free before the first worker launch
assert not torch.cuda.is_initialized(), "the notebook kernel initialised CUDA"
UT["kernel_cuda_uninitialised"] = "pass"
assert_gpus_free()
UT["gpus_free_before_first_launch"] = "pass"

(WORK / "unit_tests.json").write_text(json.dumps(UT, indent=2))
print(json.dumps(UT, indent=2))
print(f"unit tests passed in {time.time() - _t0:.0f}s")
kernel_guard('Cell 15 — unit tests')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 13: leakage gate + GPU self-tests (full validation period of folds 1-4; test is locked) ───
# The three GPU jobs run in worker subprocesses; this cell only reads their results.
_t0 = time.time()
_max_se = CONFIG["leakage_gate"]["max_abs_ic_in_standard_errors"]
GATE = {"max_abs_ic_in_standard_errors": _max_se}
_bs0 = CONFIG["memory_ladder"][0]["batch_size"]
_J_RAND = {"kind": "gate_random", "key": "gate_random", "folds": FOLD_NAMES}
_J_SHUF = {"kind": "gate_shuffled", "key": "gate_shuffled", "fold": FOLD_NAMES[0], "batch_size": _bs0, "grad_checkpoint": True}
_J_UGPU = {"kind": "unit_gpu", "key": "unit_gpu"}
launch_stage("GATE", [(120, _J_RAND), (300, _J_SHUF), (60, _J_UGPU)], batch_size=_bs0, grad_checkpoint=True)

# 1. Random-weights model (Mamba, lookback 30) through the full pipeline, all folds' validation data
_r = d_worker.load_result(_J_RAND)
_ics = _r["ics"]
_m, _se = float(_ics.mean()), float(_ics.std(ddof=1) / np.sqrt(len(_ics)))
GATE["random_weights"] = {"mean_rank_ic": _m, "se": _se, "n_days": int(len(_ics)),
                          "abs_ic_in_se": abs(_m) / _se, "per_fold": _r["per_fold"],
                          "passed": bool(abs(_m) < _max_se * _se)}

# 2. Shuffled-target run: arm B, fold 1, seed 0, training labels scrambled across dates
_s = d_worker.load_result(_J_SHUF)
GATE["shuffled_targets"] = {"mean_rank_ic": _s["mean_rank_ic"], "se": _s["se"], "n_days": _s["n_days"],
                            "abs_ic_in_se": abs(_s["mean_rank_ic"]) / _s["se"], "epochs_run": _s["epochs_run"],
                            "passed": bool(abs(_s["mean_rank_ic"]) < _max_se * _s["se"])}

# 3. Mamba causality (unit test 1) and the same scan / checkpoint self-tests, run on the real GPU
GATE["mamba_causality_unit_test"] = UT.get("mamba_causality") == "pass"
GATE["gpu_selftests"] = d_worker.load_result(_J_UGPU)                    # the worker asserts; reaching here means they passed
UT["gpu_selftests"] = GATE["gpu_selftests"]
(WORK / "unit_tests.json").write_text(json.dumps(UT, indent=2, default=float))

GATE["passed"] = bool(GATE["random_weights"]["passed"] and GATE["shuffled_targets"]["passed"]
                      and GATE["mamba_causality_unit_test"])
GATE["seconds"] = time.time() - _t0
(WORK / "leakage_gate.json").write_text(json.dumps(GATE, indent=2, default=float))
print(json.dumps(GATE, indent=2, default=float))
if not GATE["passed"]:
    raise RuntimeError("LEAKAGE GATE FAILED — see leakage_gate.json; ending the run before long training.")
kernel_guard('Cell 16 — leakage gate + GPU self-tests (workers)')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 14: memory ladder + timing probe (in workers) → per-stage budget plan ───
_B = CONFIG["budget"]
_big = FOLD_NAMES[-1]                                    # fold 4: the largest selection fold
PLAN = {"n_gpus": n_gpus, "probe_fold": _big, "n_stocks": len(UNIVERSE), "run": 2}
_lim = CONFIG["memory_limits_gb"]

# 1. Memory: heaviest arm (C) on the largest selection fold, one worker process per rung on an empty GPU.
#    Rungs are tried in order: first without gradient checkpointing, then with it, then a smaller batch (O5).
_mem_log, BATCH_SIZE, GRAD_CKPT = [], None, False
for _i, _rung in enumerate(CONFIG["memory_ladder"]):
    _j = {"kind": "probe_memory", "key": f"mem_{_i}", "arm": "C", "fold": _big, "graph": "baseline",
          "batch_size": _rung["batch_size"], "grad_checkpoint": _rung["grad_checkpoint"]}
    launch_stage("MEM", [(60, _j)], batch_size=_rung["batch_size"], grad_checkpoint=_rung["grad_checkpoint"])
    _res = d_worker.load_result(_j)
    _fits = (_res["status"] == "ok" and (_res["peak_gb"] is None or
             (_res["peak_gb"] <= _lim["max_peak"] and (_res["total_gb"] is None or _res["total_gb"] - _res["peak_gb"] >= _lim["min_headroom"]))))
    _mem_log.append({**_rung, **{k: _res[k] for k in ("peak_gb", "total_gb", "status")}, "fits": bool(_fits)})
    print("memory rung:", _mem_log[-1])
    if _fits:
        BATCH_SIZE, GRAD_CKPT = _rung["batch_size"], _rung["grad_checkpoint"]
        break
if BATCH_SIZE is None:
    raise RuntimeError(f"No rung of the memory ladder fits arm C on {_big}: {_mem_log}")
PLAN["memory_probe"], PLAN["batch_size"], PLAN["grad_checkpoint"] = _mem_log, BATCH_SIZE, GRAD_CKPT
print(f"memory probe → batch size {BATCH_SIZE}, gradient checkpointing {GRAD_CKPT} (applies to every arm)")

# 2. Timing: every arm on fold 4 / seed 0, 2 epochs, time epoch 2 (probe runs are discarded)
_tj = {a: {"kind": "probe_timing", "key": f"time_{a}", "arm": a, "fold": _big, "graph": "baseline",
           "batch_size": BATCH_SIZE, "grad_checkpoint": GRAD_CKPT} for a in ("P", "A", "B", "C")}
launch_stage("TIME", [(120, j) for j in _tj.values()], batch_size=BATCH_SIZE, grad_checkpoint=GRAD_CKPT)
SEC_PER_EPOCH = {}
for _arm, _j in _tj.items():
    SEC_PER_EPOCH[_arm] = d_worker.load_result(_j)["last_epoch_seconds"]
    print(f"arm {_arm}: {SEC_PER_EPOCH[_arm]:.1f}s per epoch (epoch 2)")
PLAN["sec_per_epoch"] = SEC_PER_EPOCH

# 3. Projection, stage by stage (stages are sequential: S2 needs the graph winner, S4 needs all locks)
ALL_FOLDS = FOLD_NAMES + HOLDOUT_NAMES
def _n_train(arm, fold):
    return len(d_graph.get_split_g(fold, "train", CONFIG["arms"][arm]["lookback"], "baseline").dates)

PER_RUN = {}
for _arm in ("P", "A", "B", "C"):
    _base = _n_train(_arm, _big)
    for _fn in ALL_FOLDS:
        PER_RUN[(_arm, _fn)] = (SEC_PER_EPOCH[_arm] * (_n_train(_arm, _fn) / _base)
                                * _B["expected_epochs_for_projection"] * _B["safety_factor"])

def grid_tag(variant):
    return f"A@{variant}"

def s1_jobs(grid_seeds):
    out = []
    for v in GRID_VARIANTS:
        for fn in FOLD_NAMES:
            for s in grid_seeds[v]:
                out.append((PER_RUN[("A", fn)], {"arm": "A", "fold": fn, "seed": s, "graph": v, "tag": grid_tag(v)}))
    return out

def arm_jobs(arms, folds, seeds, graph):
    return [(PER_RUN[(a, fn)], {"arm": a, "fold": fn, "seed": s, "graph": graph, "tag": a})
            for a in arms for fn in folds for s in seeds]

def project(grid_seeds, s2_arms, holdout_seeds):
    worst_winner = "C" if "C" in s2_arms else "B"
    stages = {
        "S1": s1_jobs(grid_seeds),
        "S2": arm_jobs(s2_arms, FOLD_NAMES, CONFIG["seeds"], "baseline"),
        "S4": arm_jobs([worst_winner, "A", "P"], HOLDOUT_NAMES, holdout_seeds, "baseline"),
    }
    spans = {k: max(lpt_assign(v, n_gpus)[1]) / 3600 for k, v in stages.items()}
    _el = (time.time() - RUN_T0) / 3600
    tail = PLANNING["phase4b_estimate_minutes"] / 60 + PLANNING["scoring_and_bench_estimate_minutes"] / 60
    return {"stage_makespan_h": spans, "n_jobs": {k: len(v) for k, v in stages.items()},
            "elapsed_h": _el, "serial_tail_h": tail, "projected_total_h": _el + sum(spans.values()) + tail}

# 4. Decision (pre-registered budget_drop_order; applied in sequence, nothing else is ever shrunk)
_limit = _B["plan_limit_hours"]
_seeds = CONFIG["seeds"]
GRID_SEEDS = {v: list(_seeds) for v in GRID_VARIANTS}
S2_ARMS, HOLDOUT_SEEDS = ["P", "B", "C"], list(CONFIG["holdout"]["seeds"])
PLAN["drops_applied"], PLAN["projection_steps"] = [], []
_do = _B["drop_order"]
_steps = [("none", None), (_do[0], "grid"), (_do[1], "dropC"), (_do[2], "holdout")]   # _do[3] = fail fast (the for/else below)
for _name, _act in _steps:
    if _act == "grid":
        GRID_SEEDS = {v: (list(_seeds) if v == "baseline" else [0, 1, 2]) for v in GRID_VARIANTS}
    elif _act == "dropC":
        S2_ARMS = ["P", "B"]
    elif _act == "holdout":
        HOLDOUT_SEEDS = [0, 1, 2]
    if _act:
        PLAN["drops_applied"].append(_name)
    PROJ = project(GRID_SEEDS, S2_ARMS, HOLDOUT_SEEDS)
    PLAN["projection_steps"].append({"after": _name, **PROJ})
    print(f"projection after '{_name}': {PROJ['projected_total_h']:.2f} h  {PROJ['stage_makespan_h']}")
    if PROJ["projected_total_h"] <= _limit:
        break
else:
    (WORK / "budget_plan.json").write_text(json.dumps(PLAN, indent=2, default=float))
    raise RuntimeError(f"Even after every pre-registered drop the projection ({PROJ['projected_total_h']:.2f} h) "
                       f"exceeds the {_limit} h plan limit — failing fast. {PROJ}")
SELECTION_CANDIDATES = [a for a in CONFIG["winner_selection"]["candidates"] if a in S2_ARMS]
PLAN.update(grid_seeds=GRID_SEEDS, s2_arms=S2_ARMS, holdout_seeds=HOLDOUT_SEEDS, selection_candidates=SELECTION_CANDIDATES,
            per_run_seconds={f"{a}|{f}": v for (a, f), v in PER_RUN.items()}, final_projection=PROJ)
(WORK / "budget_plan.json").write_text(json.dumps(PLAN, indent=2, default=float))
print(json.dumps({k: PLAN[k] for k in ("batch_size", "drops_applied", "s2_arms", "holdout_seeds", "final_projection")},
                 indent=2, default=float))
kernel_guard('Cell 17 — memory ladder + timing probe (workers) → budget plan')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 13 (S1): graph grid with arm A's setup → lock_graph.json (validation val_sel only) ───
from dmrgt import evaluate as d_eval

TARGETS = pd.read_parquet(CACHE / "targets.parquet")
_t0 = time.time()
launch_stage("S1", s1_jobs(GRID_SEEDS))

_rows = []
for _v, _params in GRID_VARIANTS.items():
    _frames = [pd.read_parquet(d_train.ckpt_paths(WORK, grid_tag(_v), f, s)["valpred"])
               for f in FOLD_NAMES for s in GRID_SEEDS[_v]]
    _df = pd.concat(_frames, ignore_index=True)
    _df = _df[_df["val_half"] == "sel"]                    # selection reads val_sel ONLY
    assert set(_df["val_half"].unique()) == {"sel"} and set(_df["split"].unique()) == {"val"}
    _res = d_eval.evaluate_predictions(_df, TARGETS, cfg, split="val")
    assert len(_res) == len(FOLD_NAMES) * len(GRID_SEEDS[_v])
    _rows.append({"variant": _v, **_params, "n_runs": len(_res), "seeds": len(GRID_SEEDS[_v]),
                  "val_sel_sharpe_raw": float(np.mean([r.fold_metrics.sharpe_after_costs for r in _res])),
                  "val_sel_mean_rank_ic": float(np.mean([r.fold_metrics.mean_rank_ic for r in _res])),
                  "val_sel_dir_acc": float(np.mean([r.fold_metrics.directional_acc for r in _res])),
                  "val_sel_turnover": float(np.mean([r.fold_metrics.turnover for r in _res])),
                  "val_sel_cum_return": float(np.mean([r.fold_metrics.cum_return for r in _res]))})
GRID = pd.DataFrame(_rows).sort_values("val_sel_sharpe_raw", ascending=False).reset_index(drop=True)
GRID.to_csv(WORK / "graph_grid.csv", index=False)
print(GRID.to_string(index=False))

_g = CONFIG["graph_grid"]
GRAPH_WINNER = GRID.loc[0, "variant"]
_tie = GRID[GRID["val_sel_sharpe_raw"] >= GRID.loc[0, "val_sel_sharpe_raw"] - _g["tie_threshold"]]
if len(_tie) > 1:                                          # within 0.02 of the best → higher val_sel rank IC
    GRAPH_WINNER = _tie.sort_values("val_sel_mean_rank_ic", ascending=False).iloc[0]["variant"]
A_TAG = grid_tag(GRAPH_WINNER)
GRAPH_WINNER_PARAMS = CONFIG["graph_grid"]["variants"][GRAPH_WINNER]
print(f"Graph winner: {GRAPH_WINNER} {GRAPH_WINNER_PARAMS} (tiebreak applied: {len(_tie) > 1})")
update_snapshot(GRAPH_WINNER={"name": GRAPH_WINNER, **GRAPH_WINNER_PARAMS})
(WORK / "lock_graph.json").write_text(json.dumps({
    "winner": GRAPH_WINNER, "params": GRAPH_WINNER_PARAMS, "tiebreak_applied": bool(len(_tie) > 1),
    "selection_metric": _g["selection_metric"], "tie_threshold": _g["tie_threshold"],
    "grid": GRID.to_dict(orient="records"), "trained_with_arm": "A", "grid_seeds": GRID_SEEDS,
    "config_sha256": CONFIG_SHA256, "selected_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=2))
S1_SECONDS = time.time() - _t0
print(f"S1 done in {S1_SECONDS / 3600:.2f} h")
kernel_guard('Cell 18 — S1 graph grid')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 14 (S2): train P, B, C on the winning graph (arm A is reused from S1) ───
_t0 = time.time()
prebuild_for_window(GRAPH_WINNER_PARAMS["corr_window"], [30, 60])    # fold cache for the winning graph window (CPU, vectorised)
_jobs = arm_jobs(S2_ARMS, FOLD_NAMES, CONFIG["seeds"], GRAPH_WINNER)
# top up arm A to all 5 seeds if the winning variant ran with 3 (budget rule); same job shape as the grid runs
_jobs += [(PER_RUN[("A", f)], {"arm": "A", "fold": f, "seed": s, "graph": GRAPH_WINNER, "tag": A_TAG})
          for f in FOLD_NAMES for s in CONFIG["seeds"]
          if not d_train.ckpt_paths(WORK, A_TAG, f, s)["done"].exists()]
launch_stage("S2", _jobs)

def tag_for(arm, fold):
    """Checkpoint tag of an arm on a fold: arm A on folds 1-4 is the winning grid variant's runs."""
    return A_TAG if (arm == "A" and fold in FOLD_NAMES) else arm

def seeds_for(fold):
    return list(CONFIG["seeds"]) if fold in FOLD_NAMES else list(HOLDOUT_SEEDS)

for _a in ["P", "A"] + S2_ARMS[1:]:
    for _f in FOLD_NAMES:
        for _s in CONFIG["seeds"]:
            assert d_train.ckpt_paths(WORK, tag_for(_a, _f), _f, _s)["done"].exists(), (_a, _f, _s)
S2_SECONDS = time.time() - _t0
print(f"S2 done in {S2_SECONDS / 3600:.2f} h; arms P, A({A_TAG}), {S2_ARMS[1:]} complete (20 runs each)")
kernel_guard('Cell 19 — S2 train P, B, C')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 15 (S3): post-processing selection + winner arm — validation val_sel only ───
import hashlib

_t0 = time.time()
_PP = CONFIG["postproc"]
RAW_WIDE = d_eval.raw_returns_from_targets(TARGETS, UNIVERSE)
ARMS4 = ["P", "A"] + S2_ARMS[1:]                                   # P, A, B (, C)

def val_frame(arm, fold):
    """Validation predictions (es + sel halves, all seeds) of an arm on a fold."""
    return pd.concat([pd.read_parquet(d_train.ckpt_paths(WORK, tag_for(arm, fold), fold, s)["valpred"])
                      for s in seeds_for(fold)], ignore_index=True).assign(model=arm)

VAL = {(a, f): val_frame(a, f) for a in ARMS4 for f in FOLD_NAMES}

def strategy_val_sel(arm, hl, q_out):
    """Pooled (folds 1-4, date order) val_sel strategy stats of an arm under one post-processing config."""
    rets, turns = [], []
    for f in FOLD_NAMES:
        df = VAL[(arm, f)]
        ret, turn = d_portfolio.strategy_series(df, RAW_WIDE, UNIVERSE, CONFIG["backtest"], hl, _PP["q_in"], q_out)
        sel = pd.DatetimeIndex(sorted(df.loc[df["val_half"] == "sel", "date"].unique()))
        rets.append(ret.reindex(sel)); turns.append(turn.reindex(sel))
    return d_eval.pooled_stats(pd.concat(rets), pd.concat(turns))

_rows = []
for _a in ARMS4:
    for _hl in _PP["ema_half_life_days"]:
        for _q in _PP["q_out"]:
            _s = strategy_val_sel(_a, _hl, _q)
            _rows.append({"arm": _a, "ema_half_life": _hl, "q_out": _q, "val_sel_strategy_sharpe": _s["sharpe"],
                          "val_sel_turnover": _s["turnover"], "val_sel_max_drawdown": _s["max_drawdown"]})
PP_TABLE = pd.DataFrame(_rows)
PP_TABLE.to_csv(WORK / "postproc_table.csv", index=False)      # 9 configs x 4 arms
_cfg_tab = (PP_TABLE.groupby(["ema_half_life", "q_out"])[["val_sel_strategy_sharpe", "val_sel_turnover"]].mean()
            .reset_index().sort_values("val_sel_strategy_sharpe", ascending=False).reset_index(drop=True))
print(_cfg_tab.to_string(index=False))
_best = _cfg_tab.loc[0, "val_sel_strategy_sharpe"]
_near = _cfg_tab[_cfg_tab["val_sel_strategy_sharpe"] >= _best - _PP["tie_threshold"]]
_pick = _near.sort_values("val_sel_turnover").iloc[0] if len(_near) > 1 else _cfg_tab.iloc[0]
CHOSEN_HL, CHOSEN_QOUT = int(_pick["ema_half_life"]), float(_pick["q_out"])
print(f"Chosen post-processing: EMA half-life {CHOSEN_HL}, q_out {CHOSEN_QOUT} (q_in {_PP['q_in']}); "
      f"tie rule applied: {len(_near) > 1}")

# raw (Phase 3 construction, per seed) and strategy numbers for every arm, val_sel only
_sum = []
for _a in ARMS4:
    _df = pd.concat([VAL[(_a, f)] for f in FOLD_NAMES])
    _res = d_eval.evaluate_predictions(_df[_df["val_half"] == "sel"], TARGETS, cfg, split="val")
    _st = strategy_val_sel(_a, CHOSEN_HL, CHOSEN_QOUT)
    _sum.append({"arm": _a, "encoder": CONFIG["arms"][_a]["encoder"], "lookback": CONFIG["arms"][_a]["lookback"],
                 "schedule": CONFIG["arms"][_a]["schedule"], "n_runs": len(_res),
                 "val_sel_raw_sharpe": float(np.mean([r.fold_metrics.sharpe_after_costs for r in _res])),
                 "val_sel_raw_mean_rank_ic": float(np.mean([r.fold_metrics.mean_rank_ic for r in _res])),
                 "val_sel_raw_turnover": float(np.mean([r.fold_metrics.turnover for r in _res])),
                 "val_sel_strategy_sharpe": _st["sharpe"], "val_sel_strategy_turnover": _st["turnover"]})
VAL5 = pd.DataFrame(_sum)
VAL5.to_csv(WORK / "validation_summary.csv", index=False)
print(VAL5.to_string(index=False))

(WORK / "lock_postproc.json").write_text(json.dumps({
    "chosen": {"ema_half_life_days": CHOSEN_HL, "q_in": _PP["q_in"], "q_out": CHOSEN_QOUT,
               "ensemble": _PP["ensemble"]},
    "tie_rule_applied": bool(len(_near) > 1), "selection": {"rule": _PP["selection"],
                                                               "tie_threshold": _PP["tie_threshold"]},
    "config_means": _cfg_tab.to_dict(orient="records"), "table_9x4": PP_TABLE.to_dict(orient="records"),
    "config_sha256": CONFIG_SHA256, "selected_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=2))

# winner arm: Mamba B vs C under the chosen configuration (val_sel strategy Sharpe)
_sel = CONFIG["winner_selection"]
_cand = VAL5[VAL5["arm"].isin(SELECTION_CANDIDATES)].sort_values("val_sel_strategy_sharpe", ascending=False).reset_index(drop=True)
WINNER, _wtie = _cand.loc[0, "arm"], False
if len(_cand) > 1 and abs(_cand.loc[0, "val_sel_strategy_sharpe"] - _cand.loc[1, "val_sel_strategy_sharpe"]) < _sel["tie_threshold"]:
    _wtie = True
    if _cand.loc[1, "val_sel_raw_mean_rank_ic"] > _cand.loc[0, "val_sel_raw_mean_rank_ic"]:
        WINNER = _cand.loc[1, "arm"]
WENC = CONFIG["arms"][WINNER]["encoder"]
print(f"Phase 5 winner: {WINNER} (tiebreak applied: {_wtie}; candidates {SELECTION_CANDIDATES})")
_ckh = {f"{f}|{s}": hashlib.sha256(d_train.ckpt_paths(WORK, WINNER, f, s)["ckpt"].read_bytes()).hexdigest()
        for f in FOLD_NAMES for s in CONFIG["seeds"]}
(WORK / "lock_winner.json").write_text(json.dumps({
    "winner_arm": WINNER, "arm_config": CONFIG["arms"][WINNER], "tiebreak_applied": _wtie, "selection_rule": _sel,
    "candidates": SELECTION_CANDIDATES, "validation_summary": VAL5.to_dict(orient="records"),
    "checkpoint_sha256": _ckh, "config_sha256": CONFIG_SHA256,
    "selected_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=2))
S3_SECONDS = time.time() - _t0
print(f"wrote lock_postproc.json and lock_winner.json (test still locked until lock_policy.json) — {S3_SECONDS / 60:.1f} min")
kernel_guard('Cell 20 — S3 post-processing + winner selection')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 16 (4B): trigger calibration (training period) + policy selection (val_sel strategy returns) ───
# Inference only: the winner's 20 checkpoints; nothing is trained. Test is still locked until the file is written.

_t0 = time.time()
_P = CONFIG["phase4b"]
cfg_w = d_data.load_cfg()                                  # graph parameters of the S1 winner
_W = int(cfg_w["graph"]["corr_window"])                     # the policy rebuilds the graph the models were trained on
_MAXSTALE = _P["max_staleness_days"]
_folds = {f.name: f for f in d_data.tables()["folds"]}
RETS = d_graph.universe_returns(cfg_w)                      # CPU; the policy simulations themselves run in workers
_sigs = [s for s in _P["signals"] if WENC == "mamba" or s == "corr_drift"]    # Δ needs a Mamba encoder

TRIGGERS = [{"name": f"{s}@{int(round(r * 100))}", "kind": "trigger", "signal": s, "target_rate": r}
            for s in _sigs for r in _P["target_rebuild_rates"]]
DAILY = {"name": "daily", "kind": "daily"}
FROZEN = {"name": "frozen_at_start", "kind": "frozen"}
ALL_POL = TRIGGERS + [DAILY, FROZEN]
SNAPSHOT = json.loads((WORK / "config_snapshot.json").read_text())

# One worker job per (fold, seed): calibrate every trigger on the TRAINING days, run every policy continuously from the
# last training day through both validation halves, return the validation predictions and the realised rebuild counts.
_vjobs = {(fn, sd): {"kind": "infer_policy", "key": f"val_{fn}_{sd}", "mode": "val", "fold": fn, "seed": sd, "tag": WINNER,
                      "graph": GRAPH_WINNER, "max_stale": _MAXSTALE, "policies": ALL_POL}
          for fn in FOLD_NAMES for sd in CONFIG["seeds"]}
launch_stage("4B", [(270, j) for j in _vjobs.values()])
THRESH = {}                                                 # "fold|seed" → {policy: thresholds}
_frames = {p["name"]: [] for p in ALL_POL}
_rate = {p["name"]: [0, 0] for p in ALL_POL}                # rebuilds / days on val_sel only
for (_fn, _seed), _j in _vjobs.items():
    _r = d_worker.load_result(_j)
    THRESH[f"{_fn}|{_seed}"] = _r["thresholds"]
    for _p in ALL_POL:
        _frames[_p["name"]].append(_r["frames"][_p["name"]])
        _rate[_p["name"]][0] += _r["rates"][_p["name"]][0]
        _rate[_p["name"]][1] += _r["rates"][_p["name"]][1]

_rows = []
for _p in ALL_POL:
    _df = pd.concat(_frames[_p["name"]], ignore_index=True)
    rets, turns = [], []
    for _fn in FOLD_NAMES:
        _d = _df[_df["fold"] == _fn]
        ret, turn = d_portfolio.strategy_series(_d, RAW_WIDE, UNIVERSE, CONFIG["backtest"], CHOSEN_HL, CONFIG["postproc"]["q_in"], CHOSEN_QOUT)
        sel = pd.DatetimeIndex(sorted(_d.loc[_d["val_half"] == "sel", "date"].unique()))
        rets.append(ret.reindex(sel)); turns.append(turn.reindex(sel))
    _s = d_eval.pooled_stats(pd.concat(rets), pd.concat(turns))
    _raw = d_eval.evaluate_predictions(_df[_df["val_half"] == "sel"], TARGETS, cfg, split="val")
    _rows.append({"policy": _p["name"], "kind": _p["kind"], "signal": _p.get("signal"), "target_rate": _p.get("target_rate"),
                  "val_sel_rebuild_rate": _rate[_p["name"]][0] / _rate[_p["name"]][1],
                  "val_sel_strategy_sharpe": _s["sharpe"], "val_sel_strategy_turnover": _s["turnover"],
                  "val_sel_raw_mean_rank_ic": float(np.mean([r.fold_metrics.mean_rank_ic for r in _raw])),
                  "n_runs": len(_raw)})
VAL4B = pd.DataFrame(_rows)
VAL4B.to_csv(WORK / "policy_validation.csv", index=False)
print(VAL4B.to_string(index=False))

# integration check: the incremental `daily` policy must reproduce the winner arm's own val_sel strategy Sharpe
_daily_sh = float(VAL4B.loc[VAL4B["policy"] == "daily", "val_sel_strategy_sharpe"].iloc[0])
_arm_sh = float(strategy_val_sel(WINNER, CHOSEN_HL, CHOSEN_QOUT)["sharpe"])
assert abs(_daily_sh - _arm_sh) < 0.05, f"daily-policy val_sel strategy Sharpe {_daily_sh:.4f} != winner's {_arm_sh:.4f}"

# selection rule (pre-registered): lowest realised val_sel rebuild rate among trigger policies whose val_sel
# STRATEGY Sharpe >= daily's − margin; ties → higher val_sel rank IC; none qualifies → highest Sharpe (flagged)
_margin = _P["val_noninferiority_margin_sharpe"]
_trig = VAL4B[VAL4B["kind"] == "trigger"]
_ok = _trig[_trig["val_sel_strategy_sharpe"] >= _daily_sh - _margin]
if len(_ok):
    _best = _ok.sort_values(["val_sel_rebuild_rate", "val_sel_raw_mean_rank_ic"], ascending=[True, False]).iloc[0]
    _noninf = True
else:
    _best = _trig.sort_values("val_sel_strategy_sharpe", ascending=False).iloc[0]
    _noninf = False
SELECTED = next(p for p in TRIGGERS if p["name"] == _best["policy"])
print(f"Phase 4B selected policy: {SELECTED['name']} (val_sel rebuild rate {_best['val_sel_rebuild_rate']:.3f}, "
      f"strategy Sharpe {_best['val_sel_strategy_sharpe']:.3f} vs daily {_daily_sh:.3f}; non-inferiority met: {_noninf})")

(WORK / "lock_policy.json").write_text(json.dumps({
    "policy": SELECTED, "winner_arm": WINNER, "graph": GRAPH_WINNER, "max_staleness_days": _MAXSTALE, "corr_window": _W,
    "thresholds": {k: v[SELECTED["name"]] for k, v in THRESH.items()}, "all_thresholds": THRESH,
    "realised_val_sel_rebuild_rate": float(_best["val_sel_rebuild_rate"]),
    "val_sel_strategy_sharpe_selected": float(_best["val_sel_strategy_sharpe"]), "val_sel_strategy_sharpe_daily": _daily_sh,
    "val_margin": _margin, "validation_noninferiority_met": _noninf,
    "daily_vs_winner_val_sel_strategy_sharpe_diff": _daily_sh - _arm_sh,
    "validation_table": VAL4B.to_dict(orient="records"), "config_sha256": CONFIG_SHA256,
    "selected_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=2))
PHASE4B_SECONDS = time.time() - _t0
assert d_data.test_unlocked(), "all four lock files should exist now"
print(f"wrote lock_policy.json — all four locks exist; test split unlocked ({PHASE4B_SECONDS / 60:.1f} min)")
kernel_guard('Cell 21 — 4B trigger calibration + selection')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 17 (S4): holdout training — winner, A, P on folds 5-6 (after all four locks exist; test still unread) ───
_t0 = time.time()
assert d_data.test_unlocked(), "S4 runs only after all four lock files exist"
_jobs = arm_jobs([WINNER, "A", "P"], HOLDOUT_NAMES, HOLDOUT_SEEDS, GRAPH_WINNER)
launch_stage("S4", _jobs)
for _a in (WINNER, "A", "P"):
    for _f in HOLDOUT_NAMES:
        for _s in HOLDOUT_SEEDS:
            assert d_train.ckpt_paths(WORK, _a, _f, _s)["done"].exists(), (_a, _f, _s)
S4_SECONDS = time.time() - _t0
print(f"S4 done in {S4_SECONDS / 3600:.2f} h")
kernel_guard('Cell 22 — S4 holdout training')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 18: ONE-SHOT test scoring — folds 1-4 and holdout folds 5-6 (first time any test row is read) ───
import hashlib
random_model, persistence_model = d_eval.random_model, d_eval.persistence_model

if (WORK / "TEST_SCORED.json").exists():
    raise RuntimeError("TEST_SCORED.json already exists — no double scoring within a run.")
d_data.assert_test_unlocked()
_t0 = time.time()
_folds = {f.name: f for f in d_data.tables()["folds"]}
_tb = d_data.tables()
_BT, _QIN, _FEE = CONFIG["backtest"], CONFIG["postproc"]["q_in"], (CONFIG["backtest"]["commission_bps"] + CONFIG["backtest"]["slippage_bps"]) / 1e4
_pol_json = json.loads((WORK / "lock_policy.json").read_text())
ALL_EVAL = FOLD_NAMES + HOLDOUT_NAMES
_pred_hashes = {}
PRED = {}                                                      # model → fold → val+test rows, all seeds

def _save_test(name, fold, seed, df_test):
    p = WORK / f"testpred_{name}_{fold}_{seed}.parquet"
    df_test.to_parquet(p, index=False)
    _pred_hashes[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()

def _val_rows(arm, fold, seed):
    return pd.read_parquet(d_train.ckpt_paths(WORK, tag_for(arm, fold), fold, seed)["valpred"]).assign(model=arm)

# ── arms: every checkpoint, the cached daily graph of the winning variant (inference runs in workers) ──
_arm_sets = {f: ["P", "A"] + S2_ARMS[1:] for f in FOLD_NAMES}
_arm_sets.update({f: [WINNER, "A", "P"] for f in HOLDOUT_NAMES})
_ajobs = {(_arm, _fn, _s): {"kind": "infer_arm", "key": f"{_arm}_{_fn}_{_s}", "name": _arm, "tag": tag_for(_arm, _fn),
                              "fold": _fn, "seed": _s, "graph": GRAPH_WINNER}
          for _fn in ALL_EVAL for _arm in _arm_sets[_fn] for _s in seeds_for(_fn)}
launch_stage("SCORE_ARMS", [(20, j) for j in _ajobs.values()])
for _fn in ALL_EVAL:
    for _arm in _arm_sets[_fn]:
        _fr = []
        for _s in seeds_for(_fn):
            _dt = d_worker.load_result(_ajobs[(_arm, _fn, _s)])["frame"]
            _save_test(_arm, _fn, _s, _dt)
            _fr.append(pd.concat([_val_rows(_arm, _fn, _s), _dt], ignore_index=True))
        PRED.setdefault(_arm, {})[_fn] = pd.concat(_fr, ignore_index=True)
    print(f"arms scored on {_fn} ({time.time() - _t0:.0f}s)", flush=True)

# ── Phase 4B: the winner under selected / daily / frozen_at_start, continuous from the last training day ──
_selN, _dailyN, _frozN = f"{WINNER}_selected", f"{WINNER}_daily", f"{WINNER}_frozen_at_start"
_pols = {_selN: _pol_json["policy"], _dailyN: DAILY, _frozN: FROZEN}
POLICY_STATS, HOLD_THRESH = [], {}
_pjobs = {}
for _fn in ALL_EVAL:
    for _s in seeds_for(_fn):
        _j = {"kind": "infer_policy", "key": f"test_{_fn}_{_s}", "mode": "test", "fold": _fn, "seed": _s, "tag": WINNER,
              "graph": GRAPH_WINNER, "max_stale": _MAXSTALE, "policies": _pols}
        if _fn in FOLD_NAMES:
            _j["thresholds"] = _pol_json["thresholds"][f"{_fn}|{_s}"]
        else:                                                  # holdout: calibrate the selected policy on this fold's training days
            _j["calibrate"], _j["selected_policy"] = True, _pol_json["policy"]
        _pjobs[(_fn, _s)] = _j
launch_stage("SCORE_POLICY", [(120, j) for j in _pjobs.values()])
for (_fn, _s), _j in _pjobs.items():
    _r = d_worker.load_result(_j)
    if _fn in HOLDOUT_NAMES:
        HOLD_THRESH[f"{_fn}|{_s}"] = _r["threshold"]
    for _n in _pols:
        _dv, _dt = _r["frames"][_n]
        _save_test(_n, _fn, _s, _dt)
        PRED.setdefault(_n, {}).setdefault(_fn, []).append(pd.concat([_dv, _dt], ignore_index=True))
    POLICY_STATS += _r["stats"]
for _n in _pols:
    for _fn in ALL_EVAL:
        PRED[_n][_fn] = pd.concat(PRED[_n][_fn], ignore_index=True)

# ── floors: random (fresh draw per seed and day, F3) and persistence, same days as the models ──
_f1 = _tb["feat_wide"]["f1_logret"][UNIVERSE]
for _fn in ALL_EVAL:
    _sv = d_graph.get_split_g(_fn, "val", 30, GRAPH_WINNER)
    _st_ = d_graph.get_split_g(_fn, "test", 30, GRAPH_WINNER)
    _dates = list(_sv.dates) + list(_st_.dates)
    _y = np.concatenate([_sv.y, _st_.y])
    _split = np.array(["val"] * len(_sv.dates) + ["test"] * len(_st_.dates))
    _Xr = [np.zeros((len(UNIVERSE), 1, 8)) for _ in _dates]
    for _i, _d in enumerate(_dates):
        _Xr[_i][:, 0, 0] = _f1.loc[pd.Timestamp(_d)].to_numpy()
    _per = {"persistence": [(0, np.stack([persistence_model(x) for x in _Xr]))],
            "random": [(s, np.stack([random_model(x, s, i) for i, x in enumerate(_Xr)])) for s in seeds_for(_fn)]}
    for _name, _lst in _per.items():
        _fr = []
        for _s, _p in _lst:
            _d = d_eval.make_pred_frame(_name, _fn, _s, "x", _dates, UNIVERSE, _p, _y)
            _d["split"] = np.repeat(_split, len(UNIVERSE))
            _d["val_half"] = np.where(_d["split"] == "val", d_eval.val_half_labels(_fn, _d["date"]), "")
            _fr.append(_d)
        PRED.setdefault(_name, {})[_fn] = pd.concat(_fr, ignore_index=True)

# ── helpers: raw (per run, Phase 3 construction) and strategy (ensemble + chosen post-processing) levels ──
def rows(name, folds, split="test"):
    return pd.concat([PRED[name][f] for f in folds], ignore_index=True).query("split == @split")

def raw_summary(name, folds):
    res = d_eval.evaluate_predictions(rows(name, folds), TARGETS, cfg, split="test")
    m = pd.DataFrame([vars(r.fold_metrics) for r in res])
    return {"model": name, "level": "raw", "n_runs": len(m), "n_days": int(m["n_days"].sum()),
            **{k: float(m[k].mean()) for k in ("mean_rank_ic", "directional_acc", "sharpe_after_costs", "max_drawdown", "turnover", "cum_return")}}

def strat(name, folds, hl=None, q_out=None):
    """Pooled (date order) test-day strategy returns and one-way turnover; EMA/buffer state runs on from validation."""
    hl = CHOSEN_HL if hl is None else hl
    q_out = CHOSEN_QOUT if q_out is None else q_out
    rs, ts = [], []
    for f in folds:
        df = PRED[name][f]
        ret, turn = d_portfolio.strategy_series(df, RAW_WIDE, UNIVERSE, _BT, hl, _QIN, q_out)
        td = pd.DatetimeIndex(sorted(df.loc[df["split"] == "test", "date"].unique()))
        rs.append(ret.reindex(td)); ts.append(turn.reindex(td))
    return pd.concat(rs).sort_index(), pd.concat(ts).sort_index()

def strat_summary(name, folds, hl=None, q_out=None):
    r, t = strat(name, folds, hl, q_out)
    s = d_eval.pooled_stats(r, t)
    return {"model": name, "level": "strategy", "n_runs": int(rows(name, folds)["seed"].nunique() * len(folds)),
            "n_days": s["n_days"], "mean_rank_ic": float("nan"), "directional_acc": float("nan"),
            "sharpe_after_costs": s["sharpe"], "max_drawdown": s["max_drawdown"], "turnover": s["turnover"], "cum_return": s["cum_return"]}

def table(models, folds):
    out = []
    for m in models:
        out += [raw_summary(m, folds), strat_summary(m, folds)]
    return pd.DataFrame(out)

ARMS_FOLDS14 = ["P", "A"] + S2_ARMS[1:]
ARMS_HOLDOUT = [WINNER, "A", "P"]
_pol_models = [_selN, _dailyN, _frozN]
_floors = ["random", "persistence"]
TEST_METRICS = table(ARMS_FOLDS14 + _pol_models + _floors, FOLD_NAMES)
HOLD_METRICS = table(ARMS_HOLDOUT + _pol_models + _floors, HOLDOUT_NAMES)
TEST_METRICS.to_csv(WORK / "test_metrics.csv", index=False)
HOLD_METRICS.to_csv(WORK / "holdout_metrics.csv", index=False)
for _tab in (TEST_METRICS, HOLD_METRICS):
    _rt = _tab[(_tab["model"] == "random") & (_tab["level"] == "raw")]["turnover"].iloc[0]
    assert _rt > 0.5, f"random baseline turnover {_rt:.3f} <= 0.5 — the random floor is not drawing fresh noise per day"
print("TEST folds 1-4\n", TEST_METRICS.to_string(index=False))
print("HOLDOUT folds 5-6\n", HOLD_METRICS.to_string(index=False))

# ── descriptive: contribution of each post-processing component (folds 1-4) ──
_comp = []
for _m in ARMS_FOLDS14:
    _raw = raw_summary(_m, FOLD_NAMES)
    _comp.append({"model": _m, "component": "raw (Phase 3 construction, per seed)", "sharpe": _raw["sharpe_after_costs"],
                  "turnover": _raw["turnover"], "cost_drag_annual": _FEE * _raw["turnover"] * 252})
    for _lab, _hl, _q in (("ensemble only", 0, _QIN), ("+ smoothing", CHOSEN_HL, _QIN), ("+ buffer (chosen)", CHOSEN_HL, CHOSEN_QOUT)):
        _r, _t = strat(_m, FOLD_NAMES, _hl, _q)
        _s = d_eval.pooled_stats(_r, _t)
        _comp.append({"model": _m, "component": _lab, "sharpe": _s["sharpe"], "turnover": _s["turnover"],
                      "cost_drag_annual": _FEE * _s["turnover"] * 252})
COMPONENTS = pd.DataFrame(_comp)
COMPONENTS.to_csv(WORK / "postproc_components.csv", index=False)

# ── significance ──
_alpha = CONFIG["significance"]["alpha"]

def verdict(ci_low, ci_high, name_a, name_b):
    if ci_low > 0:
        return f"{name_a} is significantly better than {name_b}"
    if ci_high < 0:
        return f"{name_a} is significantly worse than {name_b}"
    return f"no significant difference between {name_a} and {name_b}"

def wil(a, b, folds):
    ia = d_stats.per_fold_seed_ic(rows(a, folds), cfg)
    ib = d_stats.per_fold_seed_ic(rows(b, folds), cfg)
    try:
        return d_stats.wilcoxon_test(ia, ib, cfg)
    except ValueError as e:                                   # identical per-run ICs → undefined
        return {"note": f"Wilcoxon undefined: {e}", "p_value": 1.0, "n_pairs": len(ia), "wins": 0}

def boot_strategy(a, b, folds):
    ra, _ = strat(a, folds); rb, _ = strat(b, folds)
    assert ra.index.equals(rb.index), "pooled strategy series are not on the same days"
    out = d_stats.bootstrap_sharpe_test(ra, rb, cfg)
    out["verdict"] = verdict(out["ci_95"][0], out["ci_95"][1], a, b)
    return out

def compare(a, b, folds):
    return {"model_level_wilcoxon_raw_rank_ic": wil(a, b, folds), "strategy_level_bootstrap_sharpe": boot_strategy(a, b, folds)}

SIG = {"note": "model level = Wilcoxon on paired (fold, seed) raw mean rank IC; strategy level = stationary bootstrap on the "
               "after-cost Sharpe difference of pooled daily strategy returns (ensemble + chosen post-processing)",
       "winner_arm": WINNER, "graph": GRAPH_WINNER, "postproc": {"ema_half_life": CHOSEN_HL, "q_in": _QIN, "q_out": CHOSEN_QOUT},
       "selected_policy": _pol_json["policy"]["name"], "primary_holdout": {}, "secondary_folds_1_4": {}, "phase4b": {}}
# primary (confirmatory): holdout folds 5-6
SIG["primary_holdout"][f"{WINNER} vs A"] = compare(WINNER, "A", HOLDOUT_NAMES)
SIG["primary_holdout"][f"{WINNER} vs P"] = compare(WINNER, "P", HOLDOUT_NAMES)
SIG["primary_holdout"]["note"] = "10 paired (fold, seed) observations per Wilcoxon test: low power."
# secondary (Holm-corrected together): folds 1-4
_sec, _pv = {}, {}
_other = [a for a in S2_ARMS[1:] if a != WINNER]
for _a, _b in [(WINNER, "A")] + [(o, "A") for o in _other] + [("A", "P")]:
    _sec[f"{_a} vs {_b}"] = compare(_a, _b, FOLD_NAMES)
    _pv[f"{_a} vs {_b} (model, Wilcoxon)"] = _sec[f"{_a} vs {_b}"]["model_level_wilcoxon_raw_rank_ic"]["p_value"]
    _pv[f"{_a} vs {_b} (strategy, bootstrap two-sided)"] = _sec[f"{_a} vs {_b}"]["strategy_level_bootstrap_sharpe"]["p_two_sided"]
for _m in ARMS_FOLDS14:                           # post-processing effect: strategy vs raw, per arm
    _raw_r = d_stats.daily_returns(rows(_m, FOLD_NAMES), cfg)
    _st_r, _ = strat(_m, FOLD_NAMES)
    _common = _raw_r.index.intersection(_st_r.index)
    assert len(_common) == len(_raw_r) == len(_st_r), "raw and strategy series cover different days"
    _b = d_stats.bootstrap_sharpe_test(_st_r.reindex(_raw_r.index), _raw_r, cfg)
    _b["verdict"] = verdict(_b["ci_95"][0], _b["ci_95"][1], f"{_m} strategy", f"{_m} raw")
    _sec[f"post-processing effect {_m}"] = _b
    _pv[f"post-processing effect {_m} (bootstrap two-sided)"] = _b["p_two_sided"]
SIG["secondary_folds_1_4"]["comparisons"] = _sec
SIG["secondary_folds_1_4"]["holm"] = d_stats.holm_correction(_pv, _alpha)
# Phase 4B (own criteria, not in the Holm family): non-inferiority vs daily, superiority vs frozen_at_start; folds 1-4 and holdout
_m4 = CONFIG["phase4b"]["test_noninferiority_margin_sharpe"]
for _lab, _fs in (("folds_1_4", FOLD_NAMES), ("holdout_5_6", HOLDOUT_NAMES)):
    _vd = compare(_selN, _dailyN, _fs)
    _lo = _vd["strategy_level_bootstrap_sharpe"]["ci_95"][0]
    _vd["noninferiority"] = {"margin": _m4, "ci95_lower_bound": _lo, "non_inferior": bool(_lo > -_m4)}
    SIG["phase4b"][f"{_lab}: selected vs daily"] = _vd
    SIG["phase4b"][f"{_lab}: selected vs frozen_at_start"] = compare(_selN, _frozN, _fs)
_ps = pd.DataFrame(POLICY_STATS)
_ps["is_holdout"] = _ps["fold"].isin(HOLDOUT_NAMES)
SIG["phase4b"]["test_rebuild_rate"] = {f"{n}|{'holdout' if h else 'folds_1_4'}": float(g["test_rebuilds"].sum() / g["test_days"].sum())
                                       for (n, h), g in _ps.groupby(["policy", "is_holdout"])}
SIG["phase4b"]["graph_build_seconds_mean_per_run"] = {n: float(g["graph_build_seconds"].mean()) for n, g in _ps.groupby("policy")}
SIG["phase4b"]["holdout_calibrated_thresholds"] = HOLD_THRESH
(WORK / "significance_results.json").write_text(json.dumps(SIG, indent=2, default=float))

(WORK / "TEST_SCORED.json").write_text(json.dumps({
    "winner_arm": WINNER, "graph": GRAPH_WINNER, "postproc": {"ema_half_life": CHOSEN_HL, "q_in": _QIN, "q_out": CHOSEN_QOUT},
    "policy": _pol_json["policy"], "config_sha256": CONFIG_SHA256, "universe_sha256": UNIVERSE_SHA256,
    "prediction_file_sha256": _pred_hashes, "scored_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=2))
TEST_SECONDS = time.time() - _t0
print(f"test scoring done in {TEST_SECONDS / 60:.1f} min")
for _k, _v in SIG["primary_holdout"].items():
    if isinstance(_v, dict):
        print(f"HOLDOUT {_k}: {_v['strategy_level_bootstrap_sharpe']['verdict']}; "
              f"Wilcoxon p = {_v['model_level_wilcoxon_raw_rank_ic']['p_value']:.4f}")
kernel_guard('Cell 23 — one-shot scoring (first test read)')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 19: efficiency, Δ diagnostics (Phase 4.1 F1), extended compute benchmark (Phase 4.1 C3) ───
from scipy.stats import spearmanr

_t0 = time.time()

# ── model efficiency: every arm, same GPU and batch size ──
def _done(arm):
    fs = FOLD_NAMES
    return [json.loads(d_train.ckpt_paths(WORK, tag_for(arm, f), f, s)["done"].read_text()) for f in fs for s in CONFIG["seeds"]]

_LAT = {a: {"kind": "latency", "key": f"lat_{a}", "arm": a, "graph": GRAPH_WINNER} for a in ARMS_FOLDS14}
launch_stage("LATENCY", [(60, j) for j in _LAT.values()])
def _latency_ms(arm):
    return d_worker.load_result(_LAT[arm])["latency_ms"]

_rows = []
for _arm in ARMS_FOLDS14:
    _d = _done(_arm)
    _rows.append({"arm": _arm, "encoder": CONFIG["arms"][_arm]["encoder"], "lookback": CONFIG["arms"][_arm]["lookback"],
                  "schedule": CONFIG["arms"][_arm]["schedule"], "parameters": _d[0]["n_params"],
                  "mean_seconds_per_epoch": float(np.mean([x["mean_epoch_seconds"] for x in _d])),
                  "mean_epochs_to_early_stop": float(np.mean([x["epochs_run"] for x in _d])),
                  "mean_total_train_seconds_per_run": float(np.mean([x["total_seconds"] for x in _d])),
                  "inference_latency_ms_per_day_median": _latency_ms(_arm),
                  "peak_gpu_gb_max": float(np.max([x["peak_gpu_gb"] for x in _d])), "batch_size": BATCH_SIZE})
EFF = pd.DataFrame(_rows)
EFF.to_csv(WORK / "efficiency.csv", index=False)
print(EFF.to_string(index=False))

# ── Δ diagnostics (reported, not a gate): the winner (if Mamba) and arm B regardless ──
_vix = pd.read_parquet(CACHE / "prices.parquet").query("ticker == '^VIX'").set_index("date")["close"].sort_index()

def _delta_diag(arm, folds, label):
    per_day, per_chan, a_mins = {}, {}, []
    _dj = {(f, sd): {"kind": "delta_channels", "key": f"delta_{arm}_{f}_{sd}", "tag": tag_for(arm, f), "fold": f, "seed": sd,
                      "graph": GRAPH_WINNER} for f in folds for sd in seeds_for(f)}
    launch_stage("DELTA", [(60, j) for j in _dj.values()])
    for f in folds:
        rs = [d_worker.load_result(_dj[(f, sd)]) for sd in seeds_for(f)]
        ch = np.mean([r["channels"] for r in rs], axis=0)                              # [days, d_inner], mean over seeds
        a_mins += [r["a_min"] for r in rs]
        for d, row in zip(rs[0]["dates"], ch):
            per_day[d], per_chan[d] = float(row.mean()), row
    s = pd.Series(per_day).sort_index()
    v = _vix.reindex(s.index)
    out = {"label": label, "n_days": int(len(s)), "mean_delta": float(s.mean()), "std_delta": float(s.std()),
           "cv_daily_mean_delta": float(s.std() / s.mean()),
           "spearman_with_vix": float(spearmanr(s.values, v.values).correlation)}
    covid = s[(s.index >= "2020-03-01") & (s.index <= "2020-04-30")]
    y19 = s[(s.index >= "2019-01-01") & (s.index <= "2019-12-31")]
    out["ratio_mean_delta_covid_2020_03_04_vs_2019"] = float(covid.mean() / y19.mean()) if len(covid) and len(y19) else None
    a_min = float(np.mean(a_mins))
    med = pd.Series({d: float(np.median(r)) for d, r in per_chan.items()}).sort_index()   # median channel's Δ per day
    mem = np.log(2.0) / (med * a_min)                                                   # days until exp(-Δ·A_min)^k < 0.5
    calm, vol = v <= v.quantile(0.25), v >= v.quantile(0.75)
    out["effective_memory_days_median_channel"] = {"a_min": a_min, "calm_days_vix_bottom_quartile": float(mem[calm].mean()),
                                                   "volatile_days_vix_top_quartile": float(mem[vol].mean()),
                                                   "all_days": float(mem.mean())}
    out["flat_delta"] = bool(abs(out["spearman_with_vix"]) < 0.1 and out["cv_daily_mean_delta"] < 0.05)
    return out, s

DELTA = {}
_series = {}
_targets_d = ([("winner", WINNER)] if WENC == "mamba" else []) + ([("arm_B", "B")] if "B" in ARMS_FOLDS14 and WINNER != "B" else [])
if WENC == "mamba" and WINNER == "B":
    _targets_d = [("winner_and_arm_B", "B")]
for _lab, _arm in _targets_d:
    DELTA[f"{_lab}_folds_1_4"], _series[_lab] = _delta_diag(_arm, FOLD_NAMES, f"{_arm} test folds 1-4")
    if _arm == WINNER:
        DELTA[f"{_lab}_holdout_5_6"], _ = _delta_diag(_arm, HOLDOUT_NAMES, f"{_arm} holdout folds 5-6")
DELTA["note"] = ("Reported as it comes out. 'flat_delta' = |Spearman with VIX| < 0.1 and CV < 0.05: Δ does not vary with regime. "
                 "Calm / volatile = bottom / top VIX quartile of the diagnostic dates (this definition is the notebook's, not pre-registered by the spec). "
                 "Arm B on the holdout exists only when B is the winner (S4 trains winner, A, P).")
(WORK / "delta_diagnostics.json").write_text(json.dumps(DELTA, indent=2, default=float))
if _series:
    pd.DataFrame({k: v for k, v in _series.items()}).rename_axis("date").to_csv(WORK / "delta_by_date.csv")
print(json.dumps(DELTA, indent=2, default=float)[:2500])

# ── real-data graph-construction cost (CPU, float64), winner graph window ──
_build = d_graph.make_graph_builder(cfg_w)
_fold4 = _folds[FOLD_NAMES[-1]]
_idx = RETS.index
_pos = [_idx.get_loc(d) for d in _fold4.test_dates if d in _idx][:300]
_rc = d_graph.warm_rolling_corr(RETS, _idx[_pos[0] - 1], _W)
_rr = RETS.to_numpy(dtype=np.float64)
_t_full, _t_inc, _t_edges = [], [], []
for _i in _pos:
    t = time.perf_counter(); RETS.iloc[_i - _W + 1: _i + 1].corr().values; _t_full.append(time.perf_counter() - t)
    t = time.perf_counter(); _rc.push(_rr[_i]); _Ci = _rc.corr(); _t_inc.append(time.perf_counter() - t)
    t = time.perf_counter(); _build(_Ci); _t_edges.append(time.perf_counter() - t)
_rate_sel = SIG["phase4b"]["test_rebuild_rate"][f"{_selN}|folds_1_4"]
_nt = len(_fold4.test_dates)
REAL = {"n_stocks": len(UNIVERSE), "window": _W, "n_updates_timed": len(_pos),
        "full_recompute_ms_median": float(np.median(_t_full) * 1000), "incremental_update_ms_median": float(np.median(_t_inc) * 1000),
        "edge_build_ms_median": float(np.median(_t_edges) * 1000), "selected_policy_test_rebuild_rate": _rate_sel,
        "test_period_days_fold4": _nt,
        "test_period_total_s_full_recompute_daily": _nt * (np.median(_t_full) + np.median(_t_edges)),
        "test_period_total_s_incremental_daily": _nt * (np.median(_t_inc) + np.median(_t_edges)),
        "test_period_total_s_incremental_selected": _nt * (np.median(_t_inc) + np.median(_t_edges) * _rate_sel),
        "policy_run_seconds_mean_per_run": SIG["phase4b"]["graph_build_seconds_mean_per_run"]}
(WORK / "real_graph_cost.json").write_text(json.dumps(REAL, indent=2, default=float))

# ── extended compute benchmark (Phase 4.1 C3): where, if anywhere, does the mechanism matter? ──
_Pb = CONFIG["phase4b"]
_lb = CONFIG["arms"][WINNER]["lookback"]
_bj = {(_N, _w): {"kind": "bench_scale", "key": f"bench_{_N}_{_w}", "N": _N, "window": _w, "T": PLANNING["scale_benchmark_days"],
                   "rate_sel": _rate_sel, "enc": WENC, "lookback": _lb, "mamba_cfg": CONFIG["mamba"], "graph": GRAPH_WINNER}
       for _N in _Pb["scale_benchmark"]["n_stocks"] for _w in _Pb["scale_benchmark"]["windows"]}
launch_stage("BENCH", [(max(60, j["N"] / 10), j) for j in _bj.values()], required=True)
_bench_rows = [d_worker.load_result(j) for j in _bj.values()]
BENCH = pd.DataFrame(_bench_rows)
BENCH.to_csv(WORK / "compute_benchmark.csv", index=False)
print(BENCH[["n_stocks", "window", "ms_full_recompute", "ms_incremental_update", "ms_edge_build", "ms_model_inference",
             "graph_share_full_daily", "graph_share_incremental_trigger"]].to_string(index=False))
EXTRAS_SECONDS = time.time() - _t0
print(f"extras done in {EXTRAS_SECONDS / 60:.1f} min")
kernel_guard('Cell 24 — efficiency, Δ diagnostics, compute benchmark')        # O1: the kernel never initialises CUDA


In [ ]:
# ─── Cell 20: output verification + auto-generated PHASE5_RESULTS_DRAFT.md (hard stop on failure) ───
import glob, hashlib

_required = ["config_snapshot.json", "run_manifest.json", "budget_plan.json", "unit_tests.json",
             "leakage_gate.json", "graph_grid.csv", "lock_graph.json", "lock_postproc.json",
             "validation_summary.csv", "lock_winner.json", "policy_validation.csv", "lock_policy.json",
             "test_metrics.csv", "holdout_metrics.csv", "significance_results.json",
             "TEST_SCORED.json", "efficiency.csv", "delta_diagnostics.json",
             "compute_benchmark.csv"]
_missing = [f for f in _required if not (WORK / f).exists()]
assert not _missing, f"missing required outputs: {_missing}"

_n_runs = len(FOLD_NAMES) * len(CONFIG["seeds"])
for _a in ["P", "B"] + (["C"] if "C" in S2_ARMS else []):
    for _pat in ("ckpt_{}_*.pt", "log_{}_*.csv", "valpred_{}_*.parquet", "done_{}_*.json"):
        _n = len(glob.glob(str(WORK / _pat.format(f"{_a}_fold_[1-4]*"))))
        assert _n == _n_runs, f"arm {_a}: expected {_n_runs} files for {_pat}, found {_n}"
for _pat in ("ckpt_{}_*.pt", "valpred_{}_*.parquet"):
    assert len(glob.glob(str(WORK / _pat.format(f"{A_TAG}_fold_*")))) == _n_runs, f"arm A ({A_TAG}) incomplete"
for _v in GRID_VARIANTS:
    assert len(glob.glob(str(WORK / f"ckpt_{grid_tag(_v)}_fold_*.pt"))) == len(FOLD_NAMES) * len(GRID_SEEDS[_v]), f"grid variant {_v} incomplete"
for _a in (WINNER, "A", "P"):
    assert len(glob.glob(str(WORK / f"ckpt_{_a}_fold_[56]*.pt"))) == len(HOLDOUT_NAMES) * len(HOLDOUT_SEEDS), f"holdout arm {_a} incomplete"
for _p in glob.glob(str(WORK / "valpred_*.parquet"))[:20]:
    assert "val_half" in pd.read_parquet(_p, columns=["val_half"]).columns, f"{_p} lacks val_half"
assert len(glob.glob(str(WORK / "testpred_*.parquet"))) > 0
assert len(glob.glob(str(WORK / "dmrgt" / "*.py"))) >= 7, "dmrgt/*.py (the code that produced the results) missing"
_scored = json.loads((WORK / "TEST_SCORED.json").read_text())
for _name, _h in _scored["prediction_file_sha256"].items():
    assert hashlib.sha256((WORK / _name).read_bytes()).hexdigest() == _h, f"{_name} changed after scoring"
print("all required outputs present and consistent")

# ── draft results, every number read from the files ──
def _md(df, fmt="{:.4f}"):
    cols = list(df.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(fmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in r.values) + " |")
    return "\n".join(out)

_J = lambda n: json.loads((WORK / n).read_text())
_gw, _pp, _win, _pol, _sig = (_J("lock_graph.json"), _J("lock_postproc.json"), _J("lock_winner.json"),
                              _J("lock_policy.json"), _J("significance_results.json"))
_gate, _plan, _real, _dd = _J("leakage_gate.json"), _J("budget_plan.json"), _J("real_graph_cost.json"), _J("delta_diagnostics.json")
_grid, _val5, _val4b = (pd.read_csv(WORK / "graph_grid.csv"), pd.read_csv(WORK / "validation_summary.csv"),
                        pd.read_csv(WORK / "policy_validation.csv"))
_tm, _hm, _eff, _bench = (pd.read_csv(WORK / "test_metrics.csv"), pd.read_csv(WORK / "holdout_metrics.csv"),
                          pd.read_csv(WORK / "efficiency.csv"), pd.read_csv(WORK / "compute_benchmark.csv"))
_comp = pd.read_csv(WORK / "postproc_components.csv")
_U = pd.read_csv(CACHE / "universe.csv")
_bm = json.loads((CACHE / "build_manifest.json").read_text()) if (CACHE / "build_manifest.json").exists() else {}
_alpha = CONFIG["significance"]["alpha"]
_rates = _sig["phase4b"]["test_rebuild_rate"]

L = ["# DMRGT — Phase 5 Results: Nasdaq-100, Mamba fixes, graph retuning, cost-aware portfolio construction", "",
     "*Auto-generated from the run's JSON/CSV files. Every number is read from a file.*", "",
     "**Scope note.** Phase 4.1 was not run separately on the 40-stock universe, so there is no clean 40-stock answer on the fixed Mamba. "
     "All comparisons here are made *within* the Nasdaq-100 run against in-universe baselines P and A. Phase 4 run 1 remains the result of record for 40 stocks.", "",
     "## 1. Universe", "",
     f"- N = {len(UNIVERSE)} stocks (universe.csv SHA-256 `{UNIVERSE_SHA256}`); excluded: " +
     ", ".join(f"{r.ticker} ({r.exclusion_reason})" for r in _U[_U['included'].astype(str) != 'True'].itertuples()) + ".",
     "- **Survivorship bias:** " + (_bm.get("limitations", ["see build_manifest.json"])[0]), "",
     "## 2. Holdout 2023–2024 (headline; never seen by any earlier phase, scored once)", "", _md(_hm), ""]
for _k, _v in _sig["primary_holdout"].items():
    if isinstance(_v, dict):
        _w, _b = _v["model_level_wilcoxon_raw_rank_ic"], _v["strategy_level_bootstrap_sharpe"]
        L += [f"- **{_k}** — strategy level: {_b['verdict']} (Sharpe diff {_b['observed_sharpe_difference']:+.3f}, "
              f"95% CI [{_b['ci_95'][0]:+.3f}, {_b['ci_95'][1]:+.3f}]); model level (raw rank IC): Wilcoxon p = {_w['p_value']:.4f}, "
              f"wins {_w.get('wins', 'n/a')}/{_w.get('n_pairs', 'n/a')} (low power: {_w.get('n_pairs', 'n/a')} pairs)."]
L += ["", "## 3. Test folds 1–4 (2019–2022)", "", _md(_tm), "", "### 3.1 Secondary comparisons (Holm-corrected together)", ""]
for _k, _v in _sig["secondary_folds_1_4"]["comparisons"].items():
    if "strategy_level_bootstrap_sharpe" in _v:
        L.append(f"- {_k}: model Wilcoxon p = {_v['model_level_wilcoxon_raw_rank_ic']['p_value']:.4f}; strategy — {_v['strategy_level_bootstrap_sharpe']['verdict']}.")
    else:
        L.append(f"- {_k}: {_v['verdict']} (Sharpe diff {_v['observed_sharpe_difference']:+.3f}).")
L += ["- Holm-adjusted: " + "; ".join(f"{n}: p_adj = {v['p_adj']:.4f} (reject: {v['reject']})" for n, v in _sig["secondary_folds_1_4"]["holm"].items()), "",
      "## 4. Graph retuning (S1, validation val_sel, arm A's setup)", "", _md(_grid), "",
      f"**Winner: `{_gw['winner']}`** {_gw['params']} (tiebreak applied: {_gw['tiebreak_applied']}).", "",
      "## 5. Cost-aware portfolio construction (S3)", "",
      f"Chosen on validation (val_sel): EMA half-life {_pp['chosen']['ema_half_life_days']} days, q_in {_pp['chosen']['q_in']}, q_out {_pp['chosen']['q_out']}; "
      f"seed ensemble = {_pp['chosen']['ensemble']}.", "", "### 5.1 Validation (val_sel) summary — raw vs strategy", "", _md(_val5), "",
      "### 5.2 Contribution of each component (test folds 1–4)", "", _md(_comp), "",
      f"**Phase 4A winner (B vs C): {_win['winner_arm']}** (tiebreak applied: {_win['tiebreak_applied']}).", "",
      "## 6. Phase 4B — regime-triggered graph refresh", "",
      f"- Selected policy `{_pol['policy']['name']}`: val_sel rebuild rate {_pol['realised_val_sel_rebuild_rate']:.3f}, strategy Sharpe "
      f"{_pol['val_sel_strategy_sharpe_selected']:.3f} vs daily {_pol['val_sel_strategy_sharpe_daily']:.3f} (margin {_pol['val_margin']}; met: {_pol['validation_noninferiority_met']}).",
      "- Test rebuild rates: " + ", ".join(f"{k} = {v:.3f}" for k, v in _rates.items()) + "."]
for _lab in ("folds_1_4", "holdout_5_6"):
    _nd = _sig["phase4b"][f"{_lab}: selected vs daily"]
    _fz = _sig["phase4b"][f"{_lab}: selected vs frozen_at_start"]
    L += [f"- {_lab}: non-inferiority vs daily — CI lower bound {_nd['noninferiority']['ci95_lower_bound']:+.3f} vs −{_nd['noninferiority']['margin']} → "
          f"non-inferior: {_nd['noninferiority']['non_inferior']}; vs frozen_at_start — {_fz['strategy_level_bootstrap_sharpe']['verdict']}."]
L += ["", "### 6.1 Validation table (val_sel, strategy returns)", "", _md(_val4b), "",
      "## 7. Efficiency and technical effect", "", "### 7.1 Model efficiency", "", _md(_eff), "",
      "### 7.2 Δ diagnostics (reported, not a gate)", ""]
for _k, _v in _dd.items():
    if isinstance(_v, dict):
        _m = _v["effective_memory_days_median_channel"]
        L.append(f"- {_v['label']}: mean Δ {_v['mean_delta']:.4f}, CV {_v['cv_daily_mean_delta']:.3f}, Spearman with VIX {_v['spearman_with_vix']:+.3f}, "
                 f"2020 Mar–Apr / 2019 ratio {_v['ratio_mean_delta_covid_2020_03_04_vs_2019']}, effective memory (median channel) "
                 f"calm {_m['calm_days_vix_bottom_quartile']:.1f} d vs volatile {_m['volatile_days_vix_top_quartile']:.1f} d"
                 + ("; **Δ does not vary with regime**" if _v["flat_delta"] else "") + ".")
_max_share = float(_bench["graph_share_incremental_daily"].max())
L += ["", "### 7.3 Graph-construction cost", "",
      f"- Real data (N = {_real['n_stocks']}, window {_real['window']}): full recompute {_real['full_recompute_ms_median']:.3f} ms vs incremental "
      f"{_real['incremental_update_ms_median']:.3f} ms per update; edge build {_real['edge_build_ms_median']:.3f} ms.", "", _md(_bench[
          ["n_stocks", "window", "ms_full_recompute", "ms_incremental_update", "ms_edge_build", "ms_model_inference",
           "graph_share_full_daily", "graph_share_incremental_daily", "graph_share_incremental_trigger"]]), "",
      ("**Graph construction stays below 5% of per-day pipeline time at every benchmarked size; no technical-effect claim is made from this data.**"
       if _max_share < 0.05 else
       f"Graph construction reaches {_max_share:.1%} of per-day pipeline time at the largest benchmarked size under daily incremental rebuilds; see the share columns for where the trigger rate helps."), "",
      "## 8. Reproducibility record", "",
      f"- Pre-registered config SHA-256: `{CONFIG_SHA256}`; changes from Phase 4 listed in `config_snapshot.json`.",
      f"- Leakage gate passed: {_gate['passed']} (random-weights {_gate['random_weights']['abs_ic_in_se']:.2f} SE, shuffled-target {_gate['shuffled_targets']['abs_ic_in_se']:.2f} SE).",
      f"- Budget: batch size {_plan['batch_size']}; drops applied: {_plan['drops_applied'] or 'none'}; final projection {_plan['final_projection']['projected_total_h']:.2f} h.",
      "- Locks written in order: `lock_graph.json`, `lock_postproc.json`, `lock_winner.json`, `lock_policy.json`; test scored once (`TEST_SCORED.json`).",
      "- Code that produced the results: `dmrgt/*.py` in this run's output.", ""]
(WORK / "PHASE5_RESULTS_DRAFT.md").write_text("\n".join(L))
print("wrote PHASE5_RESULTS_DRAFT.md")
print(f"total run time {(time.time() - RUN_T0) / 3600:.2f} h")
kernel_guard('Cell 25 — output verification + PHASE5_RESULTS_DRAFT.md')        # O1: the kernel never initialises CUDA
